# Software-Engineered Chest X-Ray Analysis and Explanation System

**Notebook:** Streamlit User Interface and API Integration  
**Subject:** AIMLCZG546 - Software Engineering for Machine Learning  
**Assessment:** Assignment II  
**Group:** 89

## Group Details & Contribution

| BITS ID | Name | Engineering contribution | Weight |
|---|---|---|---:|
| 2024ac05672@wilp.bits-pilani.ac.in | Ananda Vadivel D | Requirements formulation, GR4ML view preparation, quality requirement justification | 25% |
| 2024ac05968@wilp.bits-pilani.ac.in | Adurti Sai Venkatesh | Dataset preparation, exploratory data analysis, visualization support, data quality validation | 25% |
| 2024ac05653@wilp.bits-pilani.ac.in | D Mallikarjuna Reddy | ML pipeline implementation, model comparison, final evaluation, report integration | 25% |
| 2024ac05055@wilp.bits-pilani.ac.in | Krupashankar Subramani | System architecture, architectural pattern mapping, model registry artifacts, inference and application validation | 25% |

## Notebook Index

1. [Streamlit User Interface and API Integration](#streamlit-user-interface-and-api-integration)
2. [1. Interface Integration Configuration](#1-interface-integration-configuration)
3. [2. Persisted API and Source Contract Inspection](#2-persisted-api-and-source-contract-inspection)
4. [3. UI Request and Response Schema Inspection](#3-ui-request-and-response-schema-inspection)
5. [4. UI-Safe FastAPI Client](#4-ui-safe-fastapi-client)
6. [5. HTTP Client Contract Testing](#5-http-client-contract-testing)
7. [6. Live Backend Readiness and System Information](#6-live-backend-readiness-and-system-information)
8. [7. Streamlit Page Configuration and Safety Boundary](#7-streamlit-page-configuration-and-safety-boundary)
9. [8. Persisted Image-Upload Contract Inspection](#8-persisted-image-upload-contract-inspection)
10. [9. Image Upload and Preview](#9-image-upload-and-preview)
11. [10. Complete-Analysis Submission Workflow](#10-complete-analysis-submission-workflow)
12. [11. Finding Summary and Threshold Interpretation](#11-finding-summary-and-threshold-interpretation)
13. [12. Grad-CAM Visual-Evidence Rendering](#12-grad-cam-visual-evidence-rendering)
14. [13. Grounded-Language Output Rendering](#13-grounded-language-output-rendering)
15. [14. Grounded Follow-Up Question Workflow](#14-grounded-follow-up-question-workflow)
16. [15. Stored-Prediction Retrieval](#15-stored-prediction-retrieval)
17. [16. Operational and LLMOps Metrics Display](#16-operational-and-llmops-metrics-display)
18. [17. UI Component and State-Transition Testing](#17-ui-component-and-state-transition-testing)
19. [18. Independent Streamlit Startup Validation](#18-independent-streamlit-startup-validation)
20. [19. Live Workflow Fixture and Interaction Inspection](#19-live-workflow-fixture-and-interaction-inspection)
21. [20. Live Interface Workflow Validation](#20-live-interface-workflow-validation)
22. [21. Screenshot-Capture Capability Inspection](#21-screenshot-capture-capability-inspection)
23. [22. Screenshot-Ready Interface Validation](#22-screenshot-ready-interface-validation)
24. [23. UI Artifact Registry](#23-ui-artifact-registry)
25. [24. MLflow UI Integration Registration](#24-mlflow-ui-integration-registration)
26. [25. Final Notebook 8 Readiness Gate](#25-final-notebook-8-readiness-gate)

---

The notebook records the executable implementation, retained runtime outputs, and verifiable engineering evidence for this component.


# Streamlit User Interface and API Integration

This notebook develops and validates the Streamlit user interface for the Software-Engineered Chest X-Ray Analysis and Explanation System. The interface communicates with the persisted FastAPI backend through HTTP and does not load or invoke the computer-vision or language models directly.

The interface presents model findings, visual evidence, grounded language outputs, operational metadata, and controlled errors while preserving the educational decision-support boundary.


## 1. Interface Integration Configuration

This section restores only the paths and configuration required for interface development. It validates the persisted FastAPI readiness evidence and OpenAPI contract, prepares the UI output locations, and confirms that sufficient storage remains without starting either application server.


In [1]:
from __future__ import annotations

import json
import os
import shutil
from importlib.metadata import version
from pathlib import Path
from typing import Any

import yaml


# --------------------------------------------------------------------------------------------------
# Canonical configuration
# --------------------------------------------------------------------------------------------------

PATH_REGISTRY_PATH = Path(
    "/home/jovyan/chest-xray-ai-assistant/configs/paths.yaml"
)

CANONICAL_SOLUTION_ROOT = Path(
    "/home/jovyan/chest-xray-ai-assistant"
)

CANONICAL_DATA_ROOT = Path(
    "/home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data"
)

EXPECTED_READINESS_VERSION = (
    "fastapi-integration-readiness-v1"
)

EXPECTED_READINESS_STATUS = (
    "ready_for_interface_integration"
)

EXPECTED_API_PATH_COUNT = 12
EXPECTED_API_OPERATION_COUNT = 12
MINIMUM_FREE_STORAGE_GIB = 4.0


# --------------------------------------------------------------------------------------------------
# Defensive path-registry helpers
# --------------------------------------------------------------------------------------------------

def flatten_mapping(
    value: Any,
    prefix: tuple[str, ...] = (),
) -> dict[str, Any]:
    """Flatten nested mappings without assuming their exact structure."""
    flattened: dict[str, Any] = {}

    if isinstance(value, dict):
        for key, nested_value in value.items():
            flattened.update(
                flatten_mapping(
                    nested_value,
                    prefix + (str(key).lower(),),
                )
            )
    else:
        flattened[".".join(prefix)] = value

    return flattened


def resolve_registered_path(
    flattened_registry: dict[str, Any],
    candidate_keys: tuple[str, ...],
    fallback: Path,
) -> Path:
    """Resolve a registered path with a canonical fallback."""
    normalized_candidates = {
        key.lower().replace("-", "_")
        for key in candidate_keys
    }

    for key, value in flattened_registry.items():
        normalized_key = key.replace("-", "_")
        final_key = normalized_key.rsplit(".", maxsplit=1)[-1]

        if (
            final_key in normalized_candidates
            and isinstance(value, str)
            and value.strip()
        ):
            return Path(value).expanduser().resolve()

    return fallback.expanduser().resolve()


# --------------------------------------------------------------------------------------------------
# Load the persisted path registry
# --------------------------------------------------------------------------------------------------

if not PATH_REGISTRY_PATH.is_file():
    raise FileNotFoundError(
        f"Path registry was not found: {PATH_REGISTRY_PATH}"
    )

with PATH_REGISTRY_PATH.open("r", encoding="utf-8") as file:
    path_registry = yaml.safe_load(file) or {}

if not isinstance(path_registry, dict):
    raise TypeError(
        "The path registry must contain a YAML mapping."
    )

flattened_paths = flatten_mapping(path_registry)


# --------------------------------------------------------------------------------------------------
# Resolve Notebook 8 paths
# --------------------------------------------------------------------------------------------------

SOLUTION_ROOT = resolve_registered_path(
    flattened_paths,
    (
        "solution_root",
        "project_root",
        "workspace_root",
    ),
    CANONICAL_SOLUTION_ROOT,
)

DATA_ROOT = resolve_registered_path(
    flattened_paths,
    (
        "data_root",
        "dedicated_data_root",
    ),
    CANONICAL_DATA_ROOT,
)

API_ROOT = resolve_registered_path(
    flattened_paths,
    (
        "api_root",
        "api_directory",
        "api_dir",
    ),
    SOLUTION_ROOT / "api",
)

UI_ROOT = resolve_registered_path(
    flattened_paths,
    (
        "ui_root",
        "ui_directory",
        "ui_dir",
    ),
    SOLUTION_ROOT / "ui",
)

TEST_ROOT = resolve_registered_path(
    flattened_paths,
    (
        "test_root",
        "tests_root",
        "tests_directory",
        "tests_dir",
    ),
    SOLUTION_ROOT / "tests",
)

OUTPUT_ROOT = resolve_registered_path(
    flattened_paths,
    (
        "output_root",
        "outputs_root",
        "outputs_directory",
        "outputs_dir",
    ),
    DATA_ROOT / "outputs",
)

API_OUTPUT_ROOT = OUTPUT_ROOT / "api"
UI_OUTPUT_ROOT = OUTPUT_ROOT / "ui"

UI_ROOT.mkdir(parents=True, exist_ok=True)
UI_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------------------------------------------------------
# Configure the API address without starting either server
# --------------------------------------------------------------------------------------------------

API_BASE_URL = os.getenv(
    "CHEST_XRAY_API_BASE_URL",
    "http://127.0.0.1:8000",
).strip().rstrip("/")

if not API_BASE_URL:
    raise ValueError(
        "CHEST_XRAY_API_BASE_URL cannot be blank."
    )


# --------------------------------------------------------------------------------------------------
# Load the persisted Notebook 7 evidence
# --------------------------------------------------------------------------------------------------

API_READINESS_PATH = (
    API_OUTPUT_ROOT / "api_integration_readiness.json"
)

OPENAPI_PATH = API_OUTPUT_ROOT / "openapi.json"

for required_file in (
    API_READINESS_PATH,
    OPENAPI_PATH,
):
    if not required_file.is_file():
        raise FileNotFoundError(
            f"Required API artifact was not found: {required_file}"
        )

with API_READINESS_PATH.open("r", encoding="utf-8") as file:
    api_readiness = json.load(file)

with OPENAPI_PATH.open("r", encoding="utf-8") as file:
    openapi_document = json.load(file)

if not isinstance(api_readiness, dict):
    raise TypeError(
        "The API readiness artifact must contain a JSON object."
    )

if not isinstance(openapi_document, dict):
    raise TypeError(
        "The OpenAPI artifact must contain a JSON object."
    )


# --------------------------------------------------------------------------------------------------
# Validate the exact persisted readiness contract
# --------------------------------------------------------------------------------------------------

READINESS_VERSION = api_readiness.get("readiness_version")
READINESS_STATUS = api_readiness.get("status")

readiness_api_contract = api_readiness.get(
    "api_contract",
    {},
)

readiness_testing = api_readiness.get(
    "testing",
    {},
)

readiness_storage = api_readiness.get(
    "storage",
    {},
)

readiness_checks = api_readiness.get(
    "checks",
    {},
)

if not isinstance(readiness_api_contract, dict):
    raise TypeError(
        "The readiness api_contract field must be a mapping."
    )

if not isinstance(readiness_testing, dict):
    raise TypeError(
        "The readiness testing field must be a mapping."
    )

if not isinstance(readiness_storage, dict):
    raise TypeError(
        "The readiness storage field must be a mapping."
    )

if not isinstance(readiness_checks, dict) or not readiness_checks:
    raise ValueError(
        "The readiness artifact does not contain its validation checks."
    )

READINESS_VERSION_CONFIRMED = (
    READINESS_VERSION == EXPECTED_READINESS_VERSION
)

BACKEND_READINESS_CONFIRMED = (
    READINESS_STATUS == EXPECTED_READINESS_STATUS
)

READINESS_CHECK_COUNT = len(readiness_checks)

FAILED_READINESS_CHECKS = [
    check_name
    for check_name, passed in readiness_checks.items()
    if passed is not True
]

ALL_READINESS_CHECKS_PASSED = (
    len(FAILED_READINESS_CHECKS) == 0
)


# --------------------------------------------------------------------------------------------------
# Validate the persisted OpenAPI contract
# --------------------------------------------------------------------------------------------------

openapi_paths = openapi_document.get("paths")

if not isinstance(openapi_paths, dict):
    raise ValueError(
        "The OpenAPI document does not contain a valid paths mapping."
    )

OPENAPI_PATH_COUNT = len(openapi_paths)

OPENAPI_OPERATION_COUNT = sum(
    1
    for path_item in openapi_paths.values()
    if isinstance(path_item, dict)
    for method in path_item
    if method.lower()
    in {
        "get",
        "post",
        "put",
        "patch",
        "delete",
        "options",
        "head",
        "trace",
    }
)

READINESS_RECORDED_PATH_COUNT = (
    readiness_api_contract.get("openapi_paths")
)

READINESS_RECORDED_OPERATION_COUNT = (
    readiness_api_contract.get("openapi_operations")
)

OPENAPI_PATHS_CONFIRMED = (
    OPENAPI_PATH_COUNT
    == READINESS_RECORDED_PATH_COUNT
    == EXPECTED_API_PATH_COUNT
)

OPENAPI_OPERATIONS_CONFIRMED = (
    OPENAPI_OPERATION_COUNT
    == READINESS_RECORDED_OPERATION_COUNT
    == EXPECTED_API_OPERATION_COUNT
)


# --------------------------------------------------------------------------------------------------
# Validate package versions and current storage
# --------------------------------------------------------------------------------------------------

STREAMLIT_VERSION = version("streamlit")
HTTPX_VERSION = version("httpx")

storage_usage = shutil.disk_usage(DATA_ROOT)

FREE_STORAGE_GIB = (
    storage_usage.free / (1024 ** 3)
)

STORAGE_RESERVE_PRESERVED = (
    FREE_STORAGE_GIB >= MINIMUM_FREE_STORAGE_GIB
)

NOTEBOOK_7_STORAGE_RESERVE_CONFIRMED = (
    readiness_storage.get("reserve_available") is True
)

PYTEST_FAILED_TESTS = readiness_testing.get(
    "pytest_failed_tests"
)

PYTEST_VERIFICATION_CONFIRMED = (
    PYTEST_FAILED_TESTS == 0
)


# --------------------------------------------------------------------------------------------------
# Authoritative configuration gates
# --------------------------------------------------------------------------------------------------

if not READINESS_VERSION_CONFIRMED:
    raise RuntimeError(
        "Unexpected readiness artifact version: "
        f"{READINESS_VERSION!r}."
    )

if not BACKEND_READINESS_CONFIRMED:
    raise RuntimeError(
        "Unexpected API readiness status: "
        f"{READINESS_STATUS!r}."
    )

if not ALL_READINESS_CHECKS_PASSED:
    raise RuntimeError(
        "Notebook 7 readiness checks failed: "
        f"{FAILED_READINESS_CHECKS}"
    )

if not OPENAPI_PATHS_CONFIRMED:
    raise RuntimeError(
        "OpenAPI path validation failed: "
        f"document={OPENAPI_PATH_COUNT}, "
        f"readiness={READINESS_RECORDED_PATH_COUNT}, "
        f"expected={EXPECTED_API_PATH_COUNT}."
    )

if not OPENAPI_OPERATIONS_CONFIRMED:
    raise RuntimeError(
        "OpenAPI operation validation failed: "
        f"document={OPENAPI_OPERATION_COUNT}, "
        f"readiness={READINESS_RECORDED_OPERATION_COUNT}, "
        f"expected={EXPECTED_API_OPERATION_COUNT}."
    )

if not PYTEST_VERIFICATION_CONFIRMED:
    raise RuntimeError(
        "The persisted readiness artifact reports failed API tests."
    )

if not NOTEBOOK_7_STORAGE_RESERVE_CONFIRMED:
    raise RuntimeError(
        "Notebook 7 did not preserve the protected storage reserve."
    )

if not STORAGE_RESERVE_PRESERVED:
    raise RuntimeError(
        "The current protected storage reserve is unavailable: "
        f"{FREE_STORAGE_GIB:.2f} GiB free, "
        f"{MINIMUM_FREE_STORAGE_GIB:.2f} GiB required."
    )


# --------------------------------------------------------------------------------------------------
# Formal configuration summary
# --------------------------------------------------------------------------------------------------

summary_rows = [
    ("Path registry", PATH_REGISTRY_PATH),
    ("Solution root", SOLUTION_ROOT),
    ("Data root", DATA_ROOT),
    ("API root", API_ROOT),
    ("UI root", UI_ROOT),
    ("Test root", TEST_ROOT),
    ("Output root", OUTPUT_ROOT),
    ("UI output root", UI_OUTPUT_ROOT),
    ("API base URL", API_BASE_URL),
    ("API readiness artifact", API_READINESS_PATH),
    ("OpenAPI artifact", OPENAPI_PATH),
    ("Readiness version", READINESS_VERSION),
    ("Readiness status", READINESS_STATUS),
    ("Readiness checks", f"{READINESS_CHECK_COUNT} / {READINESS_CHECK_COUNT} passed"),
    ("Persisted Pytest failures", PYTEST_FAILED_TESTS),
    ("OpenAPI paths", f"{OPENAPI_PATH_COUNT} / {EXPECTED_API_PATH_COUNT}"),
    (
        "OpenAPI operations",
        f"{OPENAPI_OPERATION_COUNT} / {EXPECTED_API_OPERATION_COUNT}",
    ),
    ("Streamlit version", STREAMLIT_VERSION),
    ("HTTPX version", HTTPX_VERSION),
    ("Free data-volume storage", f"{FREE_STORAGE_GIB:.2f} GiB"),
    ("Minimum storage reserve", f"{MINIMUM_FREE_STORAGE_GIB:.2f} GiB"),
    ("Storage reserve preserved", "PASS"),
    ("Models loaded", "NO"),
    ("FastAPI started", "NO"),
    ("Streamlit started", "NO"),
]

print("STREAMLIT INTERFACE CONFIGURATION")
print("-" * 100)

for label, value in summary_rows:
    print(f"{label:<32}: {value}")

print()
print("FINAL STATUS")
print("-" * 100)
print("READY FOR PERSISTED API CONTRACT INSPECTION")


STREAMLIT INTERFACE CONFIGURATION
----------------------------------------------------------------------------------------------------
Path registry                   : /home/jovyan/chest-xray-ai-assistant/configs/paths.yaml
Solution root                   : /home/jovyan/chest-xray-ai-assistant
Data root                       : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data
API root                        : /home/jovyan/chest-xray-ai-assistant/api
UI root                         : /home/jovyan/chest-xray-ai-assistant/ui
Test root                       : /home/jovyan/chest-xray-ai-assistant/tests
Output root                     : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs
UI output root                  : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/ui
API base URL                    : http://127.0.0.1:8000
API readiness artifact          : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/api/api_integra

## 2. Persisted API and Source Contract Inspection

This section inspects the persisted OpenAPI document and the API source modules required by the interface. It identifies the authoritative HTTP methods, request content types, request schemas, response schemas, and persisted Python definitions without starting the backend or inferring fields from earlier notebook variables.


In [2]:
import ast
from pathlib import Path
from typing import Any


# --------------------------------------------------------------------------------------------------
# UI-relevant endpoint contract
# --------------------------------------------------------------------------------------------------

UI_ENDPOINT_CONTRACT = (
    ("GET", "/health", "Backend health and readiness"),
    ("GET", "/api/v1/model/info", "Model and contract lineage"),
    ("GET", "/api/v1/model/metrics", "Persisted evaluation metrics"),
    ("POST", "/api/v1/analyze-complete", "Primary image-analysis workflow"),
    ("POST", "/api/v1/question/answer", "Grounded follow-up question"),
    (
        "GET",
        "/api/v1/predictions/{prediction_id}",
        "Stored prediction retrieval",
    ),
    ("GET", "/api/v1/llmops/metrics", "Operational and LLMOps metrics"),
)


def schema_reference(schema: Any) -> str:
    """Return a readable OpenAPI schema reference or schema type."""
    if not isinstance(schema, dict):
        return "NONE"

    reference = schema.get("$ref")

    if isinstance(reference, str):
        return reference.rsplit("/", maxsplit=1)[-1]

    schema_type = schema.get("type")

    if isinstance(schema_type, str):
        return schema_type

    if "allOf" in schema:
        references = [
            schema_reference(item)
            for item in schema["allOf"]
        ]
        return "allOf(" + ", ".join(references) + ")"

    if "anyOf" in schema:
        references = [
            schema_reference(item)
            for item in schema["anyOf"]
        ]
        return "anyOf(" + ", ".join(references) + ")"

    return "INLINE_SCHEMA"


def inspect_openapi_operation(
    method: str,
    path: str,
) -> dict[str, Any]:
    """Extract the exact persisted contract for one OpenAPI operation."""
    path_item = openapi_paths.get(path)

    if not isinstance(path_item, dict):
        raise KeyError(
            f"OpenAPI path is unavailable: {path}"
        )

    operation = path_item.get(method.lower())

    if not isinstance(operation, dict):
        raise KeyError(
            f"OpenAPI operation is unavailable: {method} {path}"
        )

    request_body = operation.get("requestBody", {})
    request_content = request_body.get("content", {})

    if not isinstance(request_content, dict):
        request_content = {}

    request_media_types = tuple(request_content.keys())

    request_schemas = {
        media_type: schema_reference(
            media_contract.get("schema", {})
        )
        for media_type, media_contract in request_content.items()
        if isinstance(media_contract, dict)
    }

    success_response = operation.get(
        "responses",
        {},
    ).get("200", {})

    success_content = success_response.get(
        "content",
        {},
    )

    if not isinstance(success_content, dict):
        success_content = {}

    response_schemas = {
        media_type: schema_reference(
            media_contract.get("schema", {})
        )
        for media_type, media_contract in success_content.items()
        if isinstance(media_contract, dict)
    }

    return {
        "operation_id": operation.get("operationId"),
        "request_media_types": request_media_types,
        "request_schemas": request_schemas,
        "response_schemas": response_schemas,
        "documented_status_codes": tuple(
            operation.get("responses", {}).keys()
        ),
    }


# --------------------------------------------------------------------------------------------------
# Persisted source-contract inspection
# --------------------------------------------------------------------------------------------------

SOURCE_CONTRACT_FILES = (
    API_ROOT / "schemas" / "common.py",
    API_ROOT / "schemas" / "requests.py",
    API_ROOT / "schemas" / "prediction.py",
    API_ROOT / "schemas" / "explainability.py",
    API_ROOT / "schemas" / "language.py",
    API_ROOT / "schemas" / "system.py",
    API_ROOT / "schemas" / "aggregate.py",
    API_ROOT / "routes" / "workflows.py",
    API_ROOT / "routes" / "aggregate.py",
    API_ROOT / "routes" / "system.py",
)


def inspect_python_definitions(
    source_path: Path,
) -> dict[str, tuple[str, ...]]:
    """Inspect persisted class and function names through the Python AST."""
    if not source_path.is_file():
        raise FileNotFoundError(
            f"Required source contract was not found: {source_path}"
        )

    source_text = source_path.read_text(
        encoding="utf-8"
    )

    syntax_tree = ast.parse(
        source_text,
        filename=str(source_path),
    )

    class_names = tuple(
        node.name
        for node in syntax_tree.body
        if isinstance(node, ast.ClassDef)
    )

    function_names = tuple(
        node.name
        for node in syntax_tree.body
        if isinstance(
            node,
            (ast.FunctionDef, ast.AsyncFunctionDef),
        )
    )

    return {
        "classes": class_names,
        "functions": function_names,
    }


# --------------------------------------------------------------------------------------------------
# Execute the read-only contract inspection
# --------------------------------------------------------------------------------------------------

endpoint_contracts = {}

for method, path, purpose in UI_ENDPOINT_CONTRACT:
    endpoint_contracts[(method, path)] = {
        "purpose": purpose,
        **inspect_openapi_operation(method, path),
    }

source_definitions = {
    source_path.relative_to(SOLUTION_ROOT).as_posix():
    inspect_python_definitions(source_path)
    for source_path in SOURCE_CONTRACT_FILES
}


# --------------------------------------------------------------------------------------------------
# Formal inspection summary
# --------------------------------------------------------------------------------------------------

print("PERSISTED API CONTRACT INSPECTION")
print("-" * 120)

for method, path, _ in UI_ENDPOINT_CONTRACT:
    contract = endpoint_contracts[(method, path)]

    request_types = (
        ", ".join(contract["request_media_types"])
        if contract["request_media_types"]
        else "NONE"
    )

    request_schemas = (
        ", ".join(
            f"{media_type}: {schema_name}"
            for media_type, schema_name
            in contract["request_schemas"].items()
        )
        if contract["request_schemas"]
        else "NONE"
    )

    response_schemas = (
        ", ".join(
            f"{media_type}: {schema_name}"
            for media_type, schema_name
            in contract["response_schemas"].items()
        )
        if contract["response_schemas"]
        else "NONE"
    )

    print(f"{method:<5} {path}")
    print(
        f"{'':<6}Operation ID       : "
        f"{contract['operation_id']}"
    )
    print(
        f"{'':<6}Purpose            : "
        f"{contract['purpose']}"
    )
    print(
        f"{'':<6}Request media      : "
        f"{request_types}"
    )
    print(
        f"{'':<6}Request schema     : "
        f"{request_schemas}"
    )
    print(
        f"{'':<6}Success schema     : "
        f"{response_schemas}"
    )
    print(
        f"{'':<6}Documented statuses: "
        f"{', '.join(contract['documented_status_codes'])}"
    )
    print()


print("PERSISTED SOURCE DEFINITIONS")
print("-" * 120)

for relative_path, definitions in source_definitions.items():
    class_summary = (
        ", ".join(definitions["classes"])
        if definitions["classes"]
        else "NONE"
    )

    function_summary = (
        ", ".join(definitions["functions"])
        if definitions["functions"]
        else "NONE"
    )

    print(relative_path)
    print(f"  Classes   : {class_summary}")
    print(f"  Functions : {function_summary}")
    print()


print("CONTRACT INSPECTION STATUS")
print("-" * 120)
print(
    f"UI-relevant OpenAPI operations inspected : "
    f"{len(endpoint_contracts)}"
)
print(
    f"Persisted source modules inspected        : "
    f"{len(source_definitions)}"
)
print("FastAPI started                           : NO")
print("Streamlit started                         : NO")
print("Models loaded                             : NO")
print()
print("READY FOR UI REQUEST AND RESPONSE SCHEMA INSPECTION")


PERSISTED API CONTRACT INSPECTION
------------------------------------------------------------------------------------------------------------------------
GET   /health
      Operation ID       : health_health_get
      Purpose            : Backend health and readiness
      Request media      : NONE
      Request schema     : NONE
      Success schema     : application/json: HealthResponse
      Documented statuses: 200

GET   /api/v1/model/info
      Operation ID       : model_info_api_v1_model_info_get
      Purpose            : Model and contract lineage
      Request media      : NONE
      Request schema     : NONE
      Success schema     : application/json: ModelInfoResponse
      Documented statuses: 200

GET   /api/v1/model/metrics
      Operation ID       : model_metrics_api_v1_model_metrics_get
      Purpose            : Persisted evaluation metrics
      Request media      : NONE
      Request schema     : NONE
      Success schema     : application/json: ModelMetricsRespo

## 3. UI Request and Response Schema Inspection

This section reads the authoritative OpenAPI component schemas used by the Streamlit workflow. It identifies required fields, data types, nested schema references, multipart upload fields, response structures, and controlled error fields without manually reproducing the backend contracts.


In [3]:
import sys
from typing import Any


# --------------------------------------------------------------------------------------------------
# OpenAPI component registry
# --------------------------------------------------------------------------------------------------

openapi_components = openapi_document.get(
    "components",
    {},
)

component_schemas = openapi_components.get(
    "schemas",
    {},
)

if not isinstance(component_schemas, dict):
    raise ValueError(
        "The OpenAPI component schema registry is unavailable."
    )


# --------------------------------------------------------------------------------------------------
# Resolve endpoint-linked schemas from the persisted contract
# --------------------------------------------------------------------------------------------------

complete_request_schema_name = endpoint_contracts[
    ("POST", "/api/v1/analyze-complete")
]["request_schemas"]["multipart/form-data"]

complete_response_schema_name = endpoint_contracts[
    ("POST", "/api/v1/analyze-complete")
]["response_schemas"]["application/json"]

question_request_schema_name = endpoint_contracts[
    ("POST", "/api/v1/question/answer")
]["request_schemas"]["application/json"]

question_response_schema_name = endpoint_contracts[
    ("POST", "/api/v1/question/answer")
]["response_schemas"]["application/json"]

stored_prediction_schema_name = endpoint_contracts[
    ("GET", "/api/v1/predictions/{prediction_id}")
]["response_schemas"]["application/json"]

health_schema_name = endpoint_contracts[
    ("GET", "/health")
]["response_schemas"]["application/json"]

model_info_schema_name = endpoint_contracts[
    ("GET", "/api/v1/model/info")
]["response_schemas"]["application/json"]

model_metrics_schema_name = endpoint_contracts[
    ("GET", "/api/v1/model/metrics")
]["response_schemas"]["application/json"]

operational_metrics_schema_name = endpoint_contracts[
    ("GET", "/api/v1/llmops/metrics")
]["response_schemas"]["application/json"]


OPENAPI_UI_SCHEMA_NAMES = tuple(
    dict.fromkeys(
        (
            complete_request_schema_name,
            complete_response_schema_name,
            question_request_schema_name,
            question_response_schema_name,
            stored_prediction_schema_name,
            health_schema_name,
            model_info_schema_name,
            model_metrics_schema_name,
            operational_metrics_schema_name,
            "ImageMetadata",
            "FindingEvidence",
            "ExplainabilityContract",
            "GradCAMEvidence",
            "EmbeddedLanguageOutput",
            "ModelVersions",
        )
    )
)

available_openapi_schema_names = tuple(
    schema_name
    for schema_name in OPENAPI_UI_SCHEMA_NAMES
    if schema_name in component_schemas
)

unavailable_openapi_schema_names = tuple(
    schema_name
    for schema_name in OPENAPI_UI_SCHEMA_NAMES
    if schema_name not in component_schemas
)


# --------------------------------------------------------------------------------------------------
# Schema inspection helpers
# --------------------------------------------------------------------------------------------------

def describe_schema_type(
    schema: Any,
) -> str:
    """Produce a compact description of an OpenAPI field type."""
    if not isinstance(schema, dict):
        return "unknown"

    reference = schema.get("$ref")

    if isinstance(reference, str):
        return reference.rsplit("/", maxsplit=1)[-1]

    if "anyOf" in schema:
        return " | ".join(
            describe_schema_type(item)
            for item in schema["anyOf"]
        )

    if "allOf" in schema:
        return " & ".join(
            describe_schema_type(item)
            for item in schema["allOf"]
        )

    field_type = str(
        schema.get("type", "object")
    )

    if field_type == "array":
        item_type = describe_schema_type(
            schema.get("items", {})
        )
        field_type = f"array[{item_type}]"

    field_format = schema.get("format")

    if field_format:
        field_type = (
            f"{field_type} ({field_format})"
        )

    return field_type


def inspect_openapi_schema(
    schema_name: str,
) -> dict[str, Any]:
    """Inspect top-level fields without imposing additional gates."""
    schema = component_schemas[schema_name]

    required_fields = set(
        schema.get("required", [])
    )

    properties = schema.get(
        "properties",
        {},
    )

    if not isinstance(properties, dict):
        properties = {}

    fields = tuple(
        {
            "name": field_name,
            "type": describe_schema_type(field_schema),
            "required": field_name in required_fields,
        }
        for field_name, field_schema in properties.items()
    )

    return {
        "source": "OpenAPI",
        "fields": fields,
        "field_count": len(fields),
        "required_count": len(required_fields),
    }


openapi_ui_schema_contracts = {
    schema_name: inspect_openapi_schema(schema_name)
    for schema_name in available_openapi_schema_names
}


# --------------------------------------------------------------------------------------------------
# Inspect source-only Pydantic schemas
# --------------------------------------------------------------------------------------------------

solution_root_text = str(SOLUTION_ROOT)

if solution_root_text not in sys.path:
    sys.path.insert(0, solution_root_text)

from api.schemas.common import APIErrorResponse
from api.schemas.requests import CompleteAnalysisOptions


def readable_annotation(
    annotation: Any,
) -> str:
    annotation_text = str(annotation)

    return (
        annotation_text
        .replace("typing.", "")
        .replace("<class '", "")
        .replace("'>", "")
    )


def inspect_pydantic_schema(
    model_class: type,
) -> dict[str, Any]:
    """Inspect a persisted Pydantic model through model_fields."""
    model_fields = model_class.model_fields

    fields = tuple(
        {
            "name": field_name,
            "type": readable_annotation(
                field_info.annotation
            ),
            "required": field_info.is_required(),
        }
        for field_name, field_info in model_fields.items()
    )

    return {
        "source": (
            f"{model_class.__module__}."
            f"{model_class.__name__}"
        ),
        "fields": fields,
        "field_count": len(fields),
        "required_count": sum(
            field["required"]
            for field in fields
        ),
    }


source_only_schema_contracts = {
    "CompleteAnalysisOptions":
        inspect_pydantic_schema(
            CompleteAnalysisOptions
        ),
    "APIErrorResponse":
        inspect_pydantic_schema(
            APIErrorResponse
        ),
}


# --------------------------------------------------------------------------------------------------
# Derive the actual multipart upload contract
# --------------------------------------------------------------------------------------------------

complete_request_schema = component_schemas[
    complete_request_schema_name
]

complete_request_properties = complete_request_schema.get(
    "properties",
    {},
)

COMPLETE_ANALYSIS_IMAGE_FIELD = next(
    (
        field_name
        for field_name, field_schema
        in complete_request_properties.items()
        if isinstance(field_schema, dict)
        and field_schema.get("format") == "binary"
    ),
    None,
)

COMPLETE_ANALYSIS_QUESTION_FIELD = (
    "question"
    if "question" in complete_request_properties
    else None
)

COMPLETE_ANALYSIS_REQUIRED_FIELDS = tuple(
    complete_request_schema.get(
        "required",
        [],
    )
)

question_request_fields = tuple(
    component_schemas[
        question_request_schema_name
    ].get("properties", {}).keys()
)

api_error_fields = tuple(
    field["name"]
    for field in source_only_schema_contracts[
        "APIErrorResponse"
    ]["fields"]
)


# --------------------------------------------------------------------------------------------------
# Formal schema summary
# --------------------------------------------------------------------------------------------------

print("UI REQUEST AND RESPONSE SCHEMA INSPECTION")
print("-" * 120)

for schema_name, contract in openapi_ui_schema_contracts.items():
    print(
        f"{schema_name} "
        f"({contract['required_count']} required / "
        f"{contract['field_count']} total)"
    )
    print(f"  Source: {contract['source']}")

    for field in contract["fields"]:
        requirement = (
            "required"
            if field["required"]
            else "optional"
        )

        print(
            f"  - {field['name']}: "
            f"{field['type']} [{requirement}]"
        )

    if not contract["fields"]:
        print("  - No top-level properties")

    print()


print("SOURCE-ONLY SCHEMA INSPECTION")
print("-" * 120)

for schema_name, contract in source_only_schema_contracts.items():
    print(
        f"{schema_name} "
        f"({contract['required_count']} required / "
        f"{contract['field_count']} total)"
    )
    print(f"  Source: {contract['source']}")

    for field in contract["fields"]:
        requirement = (
            "required"
            if field["required"]
            else "optional"
        )

        print(
            f"  - {field['name']}: "
            f"{field['type']} [{requirement}]"
        )

    print()


print("AUTHORITATIVE UI PAYLOAD CONTRACT")
print("-" * 120)
print(
    f"Complete-analysis image field        : "
    f"{COMPLETE_ANALYSIS_IMAGE_FIELD}"
)
print(
    f"Complete-analysis question field     : "
    f"{COMPLETE_ANALYSIS_QUESTION_FIELD}"
)
print(
    f"Complete-analysis required fields    : "
    f"{', '.join(COMPLETE_ANALYSIS_REQUIRED_FIELDS)}"
)
print(
    f"Grounded-question request fields     : "
    f"{', '.join(question_request_fields)}"
)
print(
    f"Controlled API error fields          : "
    f"{', '.join(api_error_fields)}"
)
print(
    f"Unavailable requested OpenAPI schemas: "
    f"{unavailable_openapi_schema_names or 'NONE'}"
)
print("Client-supplied grounding context    : NOT EXPOSED")
print("Models loaded                        : NO")
print("FastAPI started                      : NO")
print("Streamlit started                    : NO")
print()
print("READY FOR UI-SAFE HTTP CLIENT IMPLEMENTATION")


UI REQUEST AND RESPONSE SCHEMA INSPECTION
------------------------------------------------------------------------------------------------------------------------
Body_analyze_complete_api_v1_analyze_complete_post (1 required / 2 total)
  Source: OpenAPI
  - image: string (binary) [required]
  - question: string | null [optional]

CompleteAnalysisResponse (13 required / 18 total)
  Source: OpenAPI
  - api_version: string [optional]
  - crossed_finding_names: array[string] [required]
  - educational_use_only: boolean [optional]
  - explainability: ExplainabilityContract [required]
  - findings: array[FindingEvidence] [required]
  - image: ImageMetadata [required]
  - interpretation: string [required]
  - language_outputs: array[EmbeddedLanguageOutput] [required]
  - latency_ms: number [required]
  - model_versions: ModelVersions [required]
  - no_target_finding: boolean [required]
  - prediction_id: string (uuid) [required]
  - prompt_registry_version: string | null [optional]
  - reque

## 4. UI-Safe FastAPI Client

This section creates the reusable HTTP client used by the Streamlit interface. The client follows the persisted OpenAPI contract, submits the image through the authoritative `image` multipart field, handles JSON responses defensively, converts backend and connectivity failures into a controlled UI-safe error model, and does not access any model or backend service directly.


In [4]:
import ast
import py_compile
import textwrap
from pathlib import Path


# --------------------------------------------------------------------------------------------------
# Persist the UI-safe FastAPI client
# --------------------------------------------------------------------------------------------------

API_CLIENT_PATH = UI_ROOT / "api_client.py"

api_client_source = textwrap.dedent(
    '''
    from __future__ import annotations

    import os
    from typing import Any
    from uuid import UUID

    import httpx


    DEFAULT_API_BASE_URL = "http://127.0.0.1:8000"
    DEFAULT_TIMEOUT_SECONDS = 120.0


    class APIClientError(RuntimeError):
        """Controlled error exposed by the UI HTTP boundary."""

        def __init__(
            self,
            *,
            message: str,
            error_code: str,
            status_code: int | None = None,
            request_id: str | None = None,
            details: dict[str, Any] | None = None,
        ) -> None:
            super().__init__(message)
            self.message = message
            self.error_code = error_code
            self.status_code = status_code
            self.request_id = request_id
            self.details = details or {}

        def to_display_dict(self) -> dict[str, Any]:
            """Return safe structured details for Streamlit rendering."""
            return {
                "error_code": self.error_code,
                "message": self.message,
                "status_code": self.status_code,
                "request_id": self.request_id,
                "details": self.details,
            }


    class ChestXRayAPIClient:
        """HTTP-only client for the persisted FastAPI application."""

        def __init__(
            self,
            *,
            base_url: str | None = None,
            timeout_seconds: float = DEFAULT_TIMEOUT_SECONDS,
        ) -> None:
            resolved_base_url = (
                base_url
                or os.getenv(
                    "CHEST_XRAY_API_BASE_URL",
                    DEFAULT_API_BASE_URL,
                )
            ).strip().rstrip("/")

            if not resolved_base_url:
                raise ValueError("The API base URL cannot be blank.")

            if timeout_seconds <= 0:
                raise ValueError(
                    "The HTTP timeout must be greater than zero."
                )

            self.base_url = resolved_base_url
            self.timeout_seconds = float(timeout_seconds)

            self._client = httpx.Client(
                base_url=self.base_url,
                timeout=self.timeout_seconds,
                follow_redirects=True,
            )

        @staticmethod
        def _decode_json_response(
            response: httpx.Response,
        ) -> dict[str, Any] | None:
            """Decode JSON only when the response declares JSON content."""
            content_type = response.headers.get(
                "content-type",
                "",
            ).lower()

            if "application/json" not in content_type:
                return None

            try:
                payload = response.json()
            except ValueError:
                return None

            return payload if isinstance(payload, dict) else None

        def _request_json(
            self,
            method: str,
            path: str,
            **kwargs: Any,
        ) -> dict[str, Any]:
            """Execute one request and enforce a UI-safe JSON boundary."""
            try:
                response = self._client.request(
                    method=method,
                    url=path,
                    **kwargs,
                )
            except httpx.TimeoutException as exc:
                raise APIClientError(
                    error_code="API_TIMEOUT",
                    message=(
                        "The backend did not respond within the "
                        "configured time limit."
                    ),
                ) from exc
            except httpx.RequestError as exc:
                raise APIClientError(
                    error_code="API_UNAVAILABLE",
                    message=(
                        "The FastAPI backend is currently unavailable."
                    ),
                ) from exc

            payload = self._decode_json_response(response)

            if not 200 <= response.status_code < 300:
                if payload is not None:
                    error_code = str(
                        payload.get(
                            "error_code",
                            "API_REQUEST_FAILED",
                        )
                    )
                    message = str(
                        payload.get(
                            "message",
                            "The API request could not be completed.",
                        )
                    )
                    request_id_value = payload.get("request_id")
                    details_value = payload.get("details")

                    raise APIClientError(
                        error_code=error_code,
                        message=message,
                        status_code=response.status_code,
                        request_id=(
                            str(request_id_value)
                            if request_id_value is not None
                            else None
                        ),
                        details=(
                            details_value
                            if isinstance(details_value, dict)
                            else {}
                        ),
                    )

                raise APIClientError(
                    error_code="NON_JSON_API_ERROR",
                    message=(
                        "The backend returned an unexpected "
                        "non-JSON error response."
                    ),
                    status_code=response.status_code,
                )

            if payload is None:
                raise APIClientError(
                    error_code="INVALID_API_RESPONSE",
                    message=(
                        "The backend returned an invalid or "
                        "non-JSON success response."
                    ),
                    status_code=response.status_code,
                )

            return payload

        def health(self) -> dict[str, Any]:
            return self._request_json(
                "GET",
                "/health",
            )

        def model_info(self) -> dict[str, Any]:
            return self._request_json(
                "GET",
                "/api/v1/model/info",
            )

        def model_metrics(self) -> dict[str, Any]:
            return self._request_json(
                "GET",
                "/api/v1/model/metrics",
            )

        def analyze_complete(
            self,
            *,
            filename: str,
            media_type: str,
            image_content: bytes,
            question: str | None = None,
        ) -> dict[str, Any]:
            if not filename.strip():
                raise ValueError("The image filename cannot be blank.")

            if not media_type.strip():
                raise ValueError("The image media type cannot be blank.")

            if not image_content:
                raise ValueError("The uploaded image cannot be empty.")

            files = {
                "image": (
                    filename,
                    image_content,
                    media_type,
                )
            }

            data: dict[str, str] = {}

            if question is not None and question.strip():
                data["question"] = question.strip()

            return self._request_json(
                "POST",
                "/api/v1/analyze-complete",
                files=files,
                data=data,
            )

        def answer_question(
            self,
            *,
            prediction_id: str | UUID,
            question: str,
        ) -> dict[str, Any]:
            normalized_question = question.strip()

            if not normalized_question:
                raise ValueError("The question cannot be blank.")

            return self._request_json(
                "POST",
                "/api/v1/question/answer",
                json={
                    "prediction_id": str(prediction_id),
                    "question": normalized_question,
                },
            )

        def get_prediction(
            self,
            prediction_id: str | UUID,
        ) -> dict[str, Any]:
            return self._request_json(
                "GET",
                f"/api/v1/predictions/{prediction_id}",
            )

        def llmops_metrics(self) -> dict[str, Any]:
            return self._request_json(
                "GET",
                "/api/v1/llmops/metrics",
            )

        def close(self) -> None:
            self._client.close()

        def __enter__(self) -> "ChestXRayAPIClient":
            return self

        def __exit__(
            self,
            exc_type: Any,
            exc_value: Any,
            traceback: Any,
        ) -> None:
            self.close()
    '''
).strip() + "\n"

API_CLIENT_PATH.write_text(
    api_client_source,
    encoding="utf-8",
)


# --------------------------------------------------------------------------------------------------
# Validate syntax and persisted public definitions
# --------------------------------------------------------------------------------------------------

py_compile.compile(
    str(API_CLIENT_PATH),
    doraise=True,
)

syntax_tree = ast.parse(
    API_CLIENT_PATH.read_text(encoding="utf-8"),
    filename=str(API_CLIENT_PATH),
)

persisted_classes = {
    node.name: tuple(
        child.name
        for child in node.body
        if isinstance(
            child,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            ),
        )
    )
    for node in syntax_tree.body
    if isinstance(node, ast.ClassDef)
}

api_client_methods = persisted_classes.get(
    "ChestXRayAPIClient",
    (),
)


# --------------------------------------------------------------------------------------------------
# Formal implementation summary
# --------------------------------------------------------------------------------------------------

print("UI-SAFE FASTAPI CLIENT")
print("-" * 100)
print(f"Persisted module          : {API_CLIENT_PATH}")
print(f"Syntax compilation        : PASS")
print(
    f"Controlled error model    : "
    f"{'APIClientError' in persisted_classes}"
)
print(
    f"HTTP client class         : "
    f"{'ChestXRayAPIClient' in persisted_classes}"
)
print(
    f"Complete-analysis field   : "
    f"{COMPLETE_ANALYSIS_IMAGE_FIELD}"
)
print(
    f"Public endpoint methods   : "
    f"{', '.join(method for method in api_client_methods if not method.startswith('_'))}"
)
print(f"Default API base URL      : {API_BASE_URL}")
print(f"HTTP timeout              : 120.0 seconds")
print(f"Models loaded             : NO")
print(f"FastAPI started           : NO")
print(f"Streamlit started         : NO")
print()
print("READY FOR HTTP CLIENT CONTRACT TESTING")


UI-SAFE FASTAPI CLIENT
----------------------------------------------------------------------------------------------------
Persisted module          : /home/jovyan/chest-xray-ai-assistant/ui/api_client.py
Syntax compilation        : PASS
Controlled error model    : True
HTTP client class         : True
Complete-analysis field   : image
Public endpoint methods   : health, model_info, model_metrics, analyze_complete, answer_question, get_prediction, llmops_metrics, close
Default API base URL      : http://127.0.0.1:8000
HTTP timeout              : 120.0 seconds
Models loaded             : NO
FastAPI started           : NO
Streamlit started         : NO

READY FOR HTTP CLIENT CONTRACT TESTING


## 5. HTTP Client Contract Testing

This section validates the persisted UI HTTP client without starting FastAPI or loading either model. Mock HTTP transport is used to verify successful JSON handling, the authoritative `image` multipart field, optional-question submission, controlled API errors, invalid success responses, and backend timeout handling.


In [5]:
import importlib.util
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable

import httpx


# --------------------------------------------------------------------------------------------------
# Import the persisted client module directly from its validated path
# --------------------------------------------------------------------------------------------------

module_spec = importlib.util.spec_from_file_location(
    "chest_xray_ui_api_client",
    API_CLIENT_PATH,
)

if module_spec is None or module_spec.loader is None:
    raise ImportError(
        f"Unable to load the API client module: {API_CLIENT_PATH}"
    )

api_client_module = importlib.util.module_from_spec(
    module_spec
)

module_spec.loader.exec_module(
    api_client_module
)

ChestXRayAPIClient = (
    api_client_module.ChestXRayAPIClient
)

APIClientError = (
    api_client_module.APIClientError
)


def build_mock_client(
    handler: Callable[[httpx.Request], httpx.Response],
) -> ChestXRayAPIClient:
    """Create the persisted client with an isolated HTTP mock transport."""
    client = ChestXRayAPIClient(
        base_url="http://ui-client-test",
        timeout_seconds=5.0,
    )

    client.close()

    client._client = httpx.Client(
        base_url="http://ui-client-test",
        timeout=5.0,
        transport=httpx.MockTransport(handler),
    )

    return client


# --------------------------------------------------------------------------------------------------
# Individual client-contract tests
# --------------------------------------------------------------------------------------------------

def test_successful_health_response() -> dict:
    expected_payload = {
        "status": "success",
        "service_name": "test-service",
    }

    def handler(
        request: httpx.Request,
    ) -> httpx.Response:
        if request.method != "GET":
            raise AssertionError(
                f"Unexpected method: {request.method}"
            )

        if request.url.path != "/health":
            raise AssertionError(
                f"Unexpected path: {request.url.path}"
            )

        return httpx.Response(
            200,
            json=expected_payload,
        )

    with build_mock_client(handler) as client:
        received_payload = client.health()

    if received_payload != expected_payload:
        raise AssertionError(
            "The successful JSON response was not preserved."
        )

    return {
        "status_code": 200,
        "response_preserved": True,
    }


def test_complete_analysis_multipart_contract() -> dict:
    expected_prediction_id = (
        "11111111-1111-1111-1111-111111111111"
    )

    def handler(
        request: httpx.Request,
    ) -> httpx.Response:
        if request.method != "POST":
            raise AssertionError(
                f"Unexpected method: {request.method}"
            )

        if request.url.path != "/api/v1/analyze-complete":
            raise AssertionError(
                f"Unexpected path: {request.url.path}"
            )

        content_type = request.headers.get(
            "content-type",
            "",
        )

        if "multipart/form-data" not in content_type:
            raise AssertionError(
                "Complete analysis was not submitted as multipart form data."
            )

        request_body = request.read().decode(
            "latin-1"
        )

        if 'name="image"' not in request_body:
            raise AssertionError(
                "The authoritative image multipart field is missing."
            )

        if 'filename="sample.png"' not in request_body:
            raise AssertionError(
                "The uploaded filename is missing."
            )

        if 'name="question"' not in request_body:
            raise AssertionError(
                "The optional question field is missing."
            )

        if "What findings crossed their thresholds?" not in request_body:
            raise AssertionError(
                "The question value was not preserved."
            )

        return httpx.Response(
            200,
            json={
                "status": "success",
                "prediction_id": expected_prediction_id,
            },
        )

    with build_mock_client(handler) as client:
        payload = client.analyze_complete(
            filename="sample.png",
            media_type="image/png",
            image_content=b"mock-png-content",
            question=(
                "What findings crossed their thresholds?"
            ),
        )

    if payload.get("prediction_id") != expected_prediction_id:
        raise AssertionError(
            "The complete-analysis response was not preserved."
        )

    return {
        "status_code": 200,
        "multipart_image_field": "image",
        "optional_question_preserved": True,
    }


def test_controlled_api_error() -> dict:
    expected_request_id = (
        "22222222-2222-2222-2222-222222222222"
    )

    def handler(
        request: httpx.Request,
    ) -> httpx.Response:
        return httpx.Response(
            415,
            json={
                "request_id": expected_request_id,
                "error_code": "UNSUPPORTED_MEDIA_TYPE",
                "message": (
                    "The declared media type does not match "
                    "the uploaded image."
                ),
                "details": {
                    "allowed_media_types": [
                        "image/png",
                        "image/jpeg",
                    ]
                },
            },
        )

    with build_mock_client(handler) as client:
        try:
            client.analyze_complete(
                filename="sample.jpg",
                media_type="image/jpeg",
                image_content=b"mock-png-content",
            )
        except APIClientError as exc:
            if exc.status_code != 415:
                raise AssertionError(
                    "The HTTP status code was not preserved."
                )

            if exc.error_code != "UNSUPPORTED_MEDIA_TYPE":
                raise AssertionError(
                    "The controlled error code was not preserved."
                )

            if exc.request_id != expected_request_id:
                raise AssertionError(
                    "The request ID was not preserved."
                )

            return exc.to_display_dict()

    raise AssertionError(
        "The controlled API error was not raised."
    )


def test_invalid_success_response() -> dict:
    def handler(
        request: httpx.Request,
    ) -> httpx.Response:
        return httpx.Response(
            200,
            text="unexpected non-json response",
            headers={
                "content-type": "text/plain",
            },
        )

    with build_mock_client(handler) as client:
        try:
            client.health()
        except APIClientError as exc:
            if exc.error_code != "INVALID_API_RESPONSE":
                raise AssertionError(
                    "The invalid-response code was not preserved."
                )

            return exc.to_display_dict()

    raise AssertionError(
        "The non-JSON success response was not rejected."
    )


def test_backend_timeout() -> dict:
    def handler(
        request: httpx.Request,
    ) -> httpx.Response:
        raise httpx.ReadTimeout(
            "Mock backend timeout",
            request=request,
        )

    with build_mock_client(handler) as client:
        try:
            client.health()
        except APIClientError as exc:
            if exc.error_code != "API_TIMEOUT":
                raise AssertionError(
                    "The timeout error code was not preserved."
                )

            return exc.to_display_dict()

    raise AssertionError(
        "The backend timeout was not converted to a controlled error."
    )


# --------------------------------------------------------------------------------------------------
# Execute and persist the client-contract test evidence
# --------------------------------------------------------------------------------------------------

client_contract_tests = (
    (
        "successful_health_response",
        test_successful_health_response,
    ),
    (
        "complete_analysis_multipart_contract",
        test_complete_analysis_multipart_contract,
    ),
    (
        "controlled_api_error",
        test_controlled_api_error,
    ),
    (
        "invalid_success_response",
        test_invalid_success_response,
    ),
    (
        "backend_timeout",
        test_backend_timeout,
    ),
)

client_test_results = []

for test_name, test_function in client_contract_tests:
    try:
        details = test_function()

        client_test_results.append(
            {
                "test_name": test_name,
                "status": "PASS",
                "details": details,
            }
        )

    except Exception as exc:
        client_test_results.append(
            {
                "test_name": test_name,
                "status": "FAIL",
                "details": {
                    "exception_type": type(exc).__name__,
                    "message": str(exc),
                },
            }
        )

passed_client_tests = sum(
    result["status"] == "PASS"
    for result in client_test_results
)

failed_client_tests = sum(
    result["status"] == "FAIL"
    for result in client_test_results
)

API_CLIENT_TEST_EVIDENCE_PATH = (
    UI_OUTPUT_ROOT / "api_client_contract_tests.json"
)

api_client_test_evidence = {
    "evidence_version": "ui-api-client-tests-v1",
    "evaluated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "api_base_url": "http://ui-client-test",
    "http_transport": "httpx.MockTransport",
    "total_tests": len(client_test_results),
    "passed_tests": passed_client_tests,
    "failed_tests": failed_client_tests,
    "results": client_test_results,
}

API_CLIENT_TEST_EVIDENCE_PATH.write_text(
    json.dumps(
        api_client_test_evidence,
        indent=2,
    ),
    encoding="utf-8",
)


# --------------------------------------------------------------------------------------------------
# Formal test summary
# --------------------------------------------------------------------------------------------------

print("HTTP CLIENT CONTRACT TESTING")
print("-" * 100)

for result in client_test_results:
    print(
        f"{result['test_name']:<45}: "
        f"{result['status']}"
    )

print()
print(f"Executed tests             : {len(client_test_results)}")
print(f"Passed tests               : {passed_client_tests}")
print(f"Failed tests               : {failed_client_tests}")
print(f"Evidence artifact          : {API_CLIENT_TEST_EVIDENCE_PATH}")
print(f"Authoritative upload field : {COMPLETE_ANALYSIS_IMAGE_FIELD}")
print(f"FastAPI started            : NO")
print(f"Streamlit started          : NO")
print(f"Models loaded              : NO")

if failed_client_tests:
    failed_details = [
        result
        for result in client_test_results
        if result["status"] == "FAIL"
    ]

    raise RuntimeError(
        "One or more HTTP client contract tests failed: "
        f"{failed_details}"
    )

print()
print("READY FOR LIVE BACKEND HEALTH AND MODEL-INFORMATION CALLS")


HTTP CLIENT CONTRACT TESTING
----------------------------------------------------------------------------------------------------
successful_health_response                   : PASS
complete_analysis_multipart_contract         : PASS
controlled_api_error                         : PASS
invalid_success_response                     : PASS
backend_timeout                              : PASS

Executed tests             : 5
Passed tests               : 5
Failed tests               : 0
Evidence artifact          : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/ui/api_client_contract_tests.json
Authoritative upload field : image
FastAPI started            : NO
Streamlit started          : NO
Models loaded              : NO

READY FOR LIVE BACKEND HEALTH AND MODEL-INFORMATION CALLS


## 6. Live Backend Readiness and System Information

This section starts the persisted FastAPI application only when no healthy backend is already available at the configured address. It then uses the UI HTTP client to retrieve the live health, model-lineage, and persisted evaluation-metrics responses. The Streamlit application is not started in this block.


In [6]:
import os
import subprocess
import time
from pathlib import Path
from typing import Any

import httpx


# --------------------------------------------------------------------------------------------------
# Backend launch configuration
# --------------------------------------------------------------------------------------------------

FASTAPI_HOST = "127.0.0.1"
FASTAPI_PORT = 8000

FASTAPI_LOG_PATH = (
    UI_OUTPUT_ROOT / "notebook8_fastapi_backend.log"
)

FASTAPI_STARTUP_TIMEOUT_SECONDS = 60

FASTAPI_PROCESS_STARTED_BY_NOTEBOOK = False
fastapi_process = None
fastapi_log_handle = None


def backend_health_available() -> bool:
    """Return True only when the configured backend health endpoint responds."""
    try:
        response = httpx.get(
            f"{API_BASE_URL}/health",
            timeout=3.0,
        )

        if response.status_code != 200:
            return False

        content_type = response.headers.get(
            "content-type",
            "",
        ).lower()

        if "application/json" not in content_type:
            return False

        payload = response.json()

        return isinstance(payload, dict)

    except (
        httpx.RequestError,
        ValueError,
    ):
        return False


def read_log_tail(
    log_path: Path,
    maximum_lines: int = 40,
) -> str:
    """Read a bounded backend-log tail for startup diagnostics."""
    if not log_path.is_file():
        return "Backend log is not available."

    lines = log_path.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()

    return "\n".join(
        lines[-maximum_lines:]
    )


# --------------------------------------------------------------------------------------------------
# Start FastAPI only when it is not already healthy
# --------------------------------------------------------------------------------------------------

if not backend_health_available():
    backend_environment = os.environ.copy()

    backend_environment.update(
        {
            "HF_HOME": str(
                DATA_ROOT / "hf-cache"
            ),
            "HF_DATASETS_CACHE": str(
                DATA_ROOT / "hf-cache" / "datasets"
            ),
            "TORCH_HOME": str(
                DATA_ROOT / "models" / "torch-cache"
            ),
            "PIP_CACHE_DIR": str(
                DATA_ROOT / "pip-cache"
            ),
            "MLFLOW_TRACKING_URI": (
                f"file://{DATA_ROOT / 'mlflow'}"
            ),
            "TOKENIZERS_PARALLELISM": "false",
            "CHEST_XRAY_API_BASE_URL": API_BASE_URL,
            "PYTHONUNBUFFERED": "1",
        }
    )

    existing_python_path = backend_environment.get(
        "PYTHONPATH",
        "",
    )

    backend_environment["PYTHONPATH"] = (
        f"{SOLUTION_ROOT}"
        if not existing_python_path
        else f"{SOLUTION_ROOT}:{existing_python_path}"
    )

    fastapi_log_handle = FASTAPI_LOG_PATH.open(
        "w",
        encoding="utf-8",
    )

    fastapi_process = subprocess.Popen(
        [
            "/opt/conda/bin/python",
            "-m",
            "uvicorn",
            "api.main:app",
            "--host",
            FASTAPI_HOST,
            "--port",
            str(FASTAPI_PORT),
            "--log-level",
            "info",
        ],
        cwd=str(SOLUTION_ROOT),
        env=backend_environment,
        stdout=fastapi_log_handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    FASTAPI_PROCESS_STARTED_BY_NOTEBOOK = True

    startup_deadline = (
        time.monotonic()
        + FASTAPI_STARTUP_TIMEOUT_SECONDS
    )

    while time.monotonic() < startup_deadline:
        if fastapi_process.poll() is not None:
            if fastapi_log_handle is not None:
                fastapi_log_handle.flush()

            startup_log_tail = read_log_tail(
                FASTAPI_LOG_PATH
            )

            raise RuntimeError(
                "The FastAPI process exited during startup.\n\n"
                f"{startup_log_tail}"
            )

        if backend_health_available():
            break

        time.sleep(1.0)

    if not backend_health_available():
        if fastapi_log_handle is not None:
            fastapi_log_handle.flush()

        startup_log_tail = read_log_tail(
            FASTAPI_LOG_PATH
        )

        raise RuntimeError(
            "FastAPI did not become healthy within "
            f"{FASTAPI_STARTUP_TIMEOUT_SECONDS} seconds.\n\n"
            f"{startup_log_tail}"
        )


# --------------------------------------------------------------------------------------------------
# Retrieve live system responses through the persisted UI client
# --------------------------------------------------------------------------------------------------

with ChestXRayAPIClient(
    base_url=API_BASE_URL,
    timeout_seconds=30.0,
) as live_api_client:
    live_health_response = (
        live_api_client.health()
    )

    live_model_info_response = (
        live_api_client.model_info()
    )

    live_model_metrics_response = (
        live_api_client.model_metrics()
    )


# --------------------------------------------------------------------------------------------------
# Extract only confirmed top-level response information
# --------------------------------------------------------------------------------------------------

health_model_versions = live_health_response.get(
    "model_versions",
    {},
)

model_limitations = live_model_info_response.get(
    "limitations",
    [],
)

computer_vision_metrics = (
    live_model_metrics_response.get(
        "computer_vision_metrics",
        {},
    )
)

language_metrics = (
    live_model_metrics_response.get(
        "language_metrics",
        {},
    )
)

guardrail_metrics = (
    live_model_metrics_response.get(
        "guardrail_metrics",
        {},
    )
)


# --------------------------------------------------------------------------------------------------
# Formal live-backend summary
# --------------------------------------------------------------------------------------------------

print("LIVE BACKEND READINESS AND SYSTEM INFORMATION")
print("-" * 110)
print(
    f"Backend base URL             : "
    f"{API_BASE_URL}"
)
print(
    f"Backend process source       : "
    f"{'STARTED BY NOTEBOOK 8' if FASTAPI_PROCESS_STARTED_BY_NOTEBOOK else 'EXISTING HEALTHY PROCESS'}"
)
print(
    f"Backend process ID           : "
    f"{fastapi_process.pid if fastapi_process is not None else 'EXTERNAL'}"
)
print(
    f"Health HTTP contract         : PASS"
)
print(
    f"Service status               : "
    f"{live_health_response.get('status')}"
)
print(
    f"Service name                 : "
    f"{live_health_response.get('service_name')}"
)
print(
    f"API version                  : "
    f"{live_health_response.get('api_version')}"
)
print(
    f"Educational use only         : "
    f"{live_health_response.get('educational_use_only')}"
)
print(
    f"Computer-vision version      : "
    f"{health_model_versions.get('computer_vision')}"
)
print(
    f"Explainability method        : "
    f"{health_model_versions.get('explainability_method')}"
)
print(
    f"Language-model version       : "
    f"{health_model_versions.get('language')}"
)
print(
    f"Prompt-registry version      : "
    f"{live_health_response.get('prompt_registry_version')}"
)
print(
    f"Model limitations returned   : "
    f"{len(model_limitations) if isinstance(model_limitations, list) else 'UNAVAILABLE'}"
)
print(
    f"Computer-vision metrics      : "
    f"{len(computer_vision_metrics) if isinstance(computer_vision_metrics, dict) else 'UNAVAILABLE'} fields"
)
print(
    f"Language metrics             : "
    f"{len(language_metrics) if isinstance(language_metrics, dict) else 'UNAVAILABLE'} fields"
)
print(
    f"Guardrail metrics            : "
    f"{len(guardrail_metrics) if isinstance(guardrail_metrics, dict) else 'UNAVAILABLE'} fields"
)
print(
    f"Backend log                  : "
    f"{FASTAPI_LOG_PATH}"
)
print(
    f"Streamlit started            : NO"
)
print()
print("READY FOR STREAMLIT PAGE AND SAFETY-NOTICE IMPLEMENTATION")


LIVE BACKEND READINESS AND SYSTEM INFORMATION
--------------------------------------------------------------------------------------------------------------
Backend base URL             : http://127.0.0.1:8000
Backend process source       : EXISTING HEALTHY PROCESS
Backend process ID           : EXTERNAL
Health HTTP contract         : PASS
Service status               : success
Service name                 : API-Driven Chest X-Ray Analysis and Explanation Assistant
API version                  : v1
Educational use only         : True
Computer-vision version      : resnet18-chestmnist-v1
Explainability method        : LayerGradCam
Language-model version       : flan-t5-small-chestmnist-v1
Prompt-registry version      : grounded-language-prompts-v1
Model limitations returned   : 2
Computer-vision metrics      : 2 fields
Language metrics             : 2 fields
Guardrail metrics            : 2 fields
Backend log                  : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-dat

## 7. Streamlit Page Configuration and Safety Boundary

This section establishes the persistent UI configuration and the initial Streamlit application shell. It presents the solution identity, API boundary, and exact educational-use limitation before any image can be submitted or any model-generated output can be displayed.


In [7]:
import ast
import py_compile
import textwrap
from pathlib import Path

import yaml


# --------------------------------------------------------------------------------------------------
# Persist the UI configuration
# --------------------------------------------------------------------------------------------------

UI_CONFIG_PATH = (
    SOLUTION_ROOT / "configs" / "ui_config.yaml"
)

STREAMLIT_APP_PATH = (
    UI_ROOT / "app.py"
)

EDUCATIONAL_LIMITATION = (
    "This output is generated by an educational decision-support prototype. "
    "It is not a diagnosis and should not replace review by a qualified "
    "healthcare professional."
)

GRADCAM_LIMITATION = (
    "Grad-CAM shows image regions that influenced a model output. "
    "It must not be interpreted as lesion segmentation, causal proof, "
    "anatomical confirmation, or clinical diagnosis."
)

ui_configuration = {
    "ui_version": "streamlit-interface-v1",
    "page": {
        "title": (
            "Chest X-Ray Analysis and Explanation Assistant"
        ),
        "subtitle": (
            "API-driven educational decision-support prototype"
        ),
        "icon": "🩻",
        "layout": "wide",
        "initial_sidebar_state": "expanded",
    },
    "api": {
        "base_url_environment_variable": (
            "CHEST_XRAY_API_BASE_URL"
        ),
        "default_base_url": API_BASE_URL,
    },
    "safety": {
        "educational_limitation": EDUCATIONAL_LIMITATION,
        "gradcam_limitation": GRADCAM_LIMITATION,
        "professional_review_guidance": (
            "Review all outputs with a qualified healthcare professional."
        ),
    },
}

UI_CONFIG_PATH.write_text(
    yaml.safe_dump(
        ui_configuration,
        sort_keys=False,
        allow_unicode=True,
    ),
    encoding="utf-8",
)


# --------------------------------------------------------------------------------------------------
# Persist the initial Streamlit application shell
# --------------------------------------------------------------------------------------------------

streamlit_app_source = textwrap.dedent(
    '''
    from __future__ import annotations

    import os
    from pathlib import Path
    from typing import Any

    import streamlit as st
    import yaml


    SOLUTION_ROOT = Path(__file__).resolve().parents[1]
    UI_CONFIG_PATH = SOLUTION_ROOT / "configs" / "ui_config.yaml"


    def load_ui_configuration(
        config_path: Path,
    ) -> dict[str, Any]:
        """Load the persisted Streamlit configuration."""
        if not config_path.is_file():
            raise FileNotFoundError(
                f"UI configuration was not found: {config_path}"
            )

        with config_path.open("r", encoding="utf-8") as file:
            configuration = yaml.safe_load(file) or {}

        if not isinstance(configuration, dict):
            raise TypeError(
                "The UI configuration must contain a YAML mapping."
            )

        return configuration


    ui_config = load_ui_configuration(
        UI_CONFIG_PATH
    )

    page_config = ui_config["page"]
    api_config = ui_config["api"]
    safety_config = ui_config["safety"]

    api_base_url = os.getenv(
        api_config["base_url_environment_variable"],
        api_config["default_base_url"],
    ).strip().rstrip("/")


    st.set_page_config(
        page_title=page_config["title"],
        page_icon=page_config["icon"],
        layout=page_config["layout"],
        initial_sidebar_state=(
            page_config["initial_sidebar_state"]
        ),
    )


    st.title(page_config["title"])
    st.caption(page_config["subtitle"])

    st.warning(
        safety_config["educational_limitation"],
        icon="⚠️",
    )

    st.markdown(
        """
        Upload a supported chest X-ray image to request a complete,
        API-driven analysis. Findings are determined only by the frozen
        computer-vision model and its persisted per-finding thresholds.
        Language outputs are generated only from the resulting structured
        evidence.
        """
    )


    with st.sidebar:
        st.header("System Boundary")

        st.info(
            "The interface communicates with the FastAPI backend through "
            "HTTP. It does not load the computer-vision model, language "
            "model, Grad-CAM service, or prediction store directly."
        )

        st.subheader("Backend Address")
        st.code(
            api_base_url,
            language=None,
        )

        st.subheader("Professional Review")
        st.caption(
            safety_config[
                "professional_review_guidance"
            ]
        )
    '''
).strip() + "\n"

STREAMLIT_APP_PATH.write_text(
    streamlit_app_source,
    encoding="utf-8",
)


# --------------------------------------------------------------------------------------------------
# Validate the persisted configuration and application source
# --------------------------------------------------------------------------------------------------

persisted_ui_configuration = yaml.safe_load(
    UI_CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)

py_compile.compile(
    str(STREAMLIT_APP_PATH),
    doraise=True,
)

streamlit_syntax_tree = ast.parse(
    STREAMLIT_APP_PATH.read_text(
        encoding="utf-8"
    ),
    filename=str(STREAMLIT_APP_PATH),
)

streamlit_function_names = tuple(
    node.name
    for node in streamlit_syntax_tree.body
    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        ),
    )
)

persisted_educational_limitation = (
    persisted_ui_configuration["safety"][
        "educational_limitation"
    ]
)

if persisted_educational_limitation != EDUCATIONAL_LIMITATION:
    raise RuntimeError(
        "The persisted educational limitation does not match "
        "the canonical safety boundary."
    )


# --------------------------------------------------------------------------------------------------
# Formal implementation summary
# --------------------------------------------------------------------------------------------------

print("STREAMLIT PAGE CONFIGURATION AND SAFETY BOUNDARY")
print("-" * 100)
print(f"UI configuration          : {UI_CONFIG_PATH}")
print(f"Streamlit application     : {STREAMLIT_APP_PATH}")
print(
    f"UI version                : "
    f"{persisted_ui_configuration['ui_version']}"
)
print(
    f"Page title                : "
    f"{persisted_ui_configuration['page']['title']}"
)
print(
    f"Page layout               : "
    f"{persisted_ui_configuration['page']['layout']}"
)
print(
    f"Educational limitation    : PRESERVED"
)
print(
    f"Grad-CAM limitation       : PRESERVED"
)
print(
    f"Backend boundary          : HTTP ONLY"
)
print(
    f"Persisted functions       : "
    f"{', '.join(streamlit_function_names)}"
)
print(f"Syntax compilation        : PASS")
print(f"FastAPI health            : AVAILABLE")
print(f"Streamlit started         : NO")
print()
print("READY FOR IMAGE UPLOAD AND PREVIEW IMPLEMENTATION")


STREAMLIT PAGE CONFIGURATION AND SAFETY BOUNDARY
----------------------------------------------------------------------------------------------------
UI configuration          : /home/jovyan/chest-xray-ai-assistant/configs/ui_config.yaml
Streamlit application     : /home/jovyan/chest-xray-ai-assistant/ui/app.py
UI version                : streamlit-interface-v1
Page title                : Chest X-Ray Analysis and Explanation Assistant
Page layout               : wide
Educational limitation    : PRESERVED
Grad-CAM limitation       : PRESERVED
Backend boundary          : HTTP ONLY
Persisted functions       : load_ui_configuration
Syntax compilation        : PASS
FastAPI health            : AVAILABLE
Streamlit started         : NO

READY FOR IMAGE UPLOAD AND PREVIEW IMPLEMENTATION


## 8. Persisted Image-Upload Contract Inspection

This section inspects the API contract and service configuration for the authoritative upload-size, media-type, image-format, and validation constraints. These persisted values will be used by the Streamlit uploader so that the interface does not introduce separate or conflicting image-validation rules.


In [8]:
import ast
import sys
from pathlib import Path
from typing import Any

import yaml


# --------------------------------------------------------------------------------------------------
# Persisted upload-related sources
# --------------------------------------------------------------------------------------------------

API_CONTRACT_PATH = (
    SOLUTION_ROOT / "configs" / "api_contract.yaml"
)

API_CONFIG_SOURCE_PATH = (
    API_ROOT / "core" / "config.py"
)

IMAGE_SERVICE_SOURCE_PATH = (
    SOLUTION_ROOT
    / "src"
    / "services"
    / "image_service.py"
)

required_upload_contract_files = (
    API_CONTRACT_PATH,
    API_CONFIG_SOURCE_PATH,
    IMAGE_SERVICE_SOURCE_PATH,
)

for required_file in required_upload_contract_files:
    if not required_file.is_file():
        raise FileNotFoundError(
            f"Required upload-contract source was not found: {required_file}"
        )


# --------------------------------------------------------------------------------------------------
# Load and flatten the persisted API contract
# --------------------------------------------------------------------------------------------------

with API_CONTRACT_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    api_contract_document = yaml.safe_load(file) or {}

if not isinstance(api_contract_document, dict):
    raise TypeError(
        "The API contract must contain a YAML mapping."
    )


def flatten_contract(
    value: Any,
    prefix: tuple[str, ...] = (),
) -> dict[str, Any]:
    """Flatten mappings and sequences while preserving their locations."""
    flattened: dict[str, Any] = {}

    if isinstance(value, dict):
        for key, nested_value in value.items():
            flattened.update(
                flatten_contract(
                    nested_value,
                    prefix + (str(key),),
                )
            )

    elif isinstance(value, (list, tuple)):
        flattened[".".join(prefix)] = list(value)

    else:
        flattened[".".join(prefix)] = value

    return flattened


flattened_api_contract = flatten_contract(
    api_contract_document
)

UPLOAD_KEYWORDS = (
    "upload",
    "image",
    "media",
    "mime",
    "size",
    "byte",
    "format",
    "extension",
    "dimension",
    "width",
    "height",
)

upload_contract_entries = {
    key: value
    for key, value in flattened_api_contract.items()
    if any(
        keyword in key.lower()
        for keyword in UPLOAD_KEYWORDS
    )
}


# --------------------------------------------------------------------------------------------------
# Inspect ServiceSettings fields without constructing backend services
# --------------------------------------------------------------------------------------------------

solution_root_text = str(SOLUTION_ROOT)

if solution_root_text not in sys.path:
    sys.path.insert(0, solution_root_text)

from api.core.config import ServiceSettings


service_setting_fields = getattr(
    ServiceSettings,
    "model_fields",
    {},
)

upload_setting_fields = {}

for field_name, field_info in service_setting_fields.items():
    normalized_name = field_name.lower()

    if any(
        keyword in normalized_name
        for keyword in UPLOAD_KEYWORDS
    ):
        upload_setting_fields[field_name] = {
            "annotation": str(
                field_info.annotation
            ).replace("typing.", ""),
            "required": field_info.is_required(),
            "default": (
                None
                if field_info.is_required()
                else field_info.default
            ),
        }


# --------------------------------------------------------------------------------------------------
# Inspect relevant persisted constants from image_service.py
# --------------------------------------------------------------------------------------------------

image_service_source = (
    IMAGE_SERVICE_SOURCE_PATH.read_text(
        encoding="utf-8"
    )
)

image_service_tree = ast.parse(
    image_service_source,
    filename=str(IMAGE_SERVICE_SOURCE_PATH),
)

image_service_assignments = {}

for node in ast.walk(image_service_tree):
    if isinstance(
        node,
        (
            ast.Assign,
            ast.AnnAssign,
        ),
    ):
        assignment_names = []

        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name):
                    assignment_names.append(
                        target.id
                    )
        elif isinstance(node.target, ast.Name):
            assignment_names.append(
                node.target.id
            )

        value_node = getattr(
            node,
            "value",
            None,
        )

        for assignment_name in assignment_names:
            if any(
                keyword in assignment_name.lower()
                for keyword in UPLOAD_KEYWORDS
            ):
                try:
                    assignment_value = ast.literal_eval(
                        value_node
                    )
                except (
                    ValueError,
                    TypeError,
                ):
                    assignment_value = "<runtime expression>"

                image_service_assignments[
                    assignment_name
                ] = assignment_value


# --------------------------------------------------------------------------------------------------
# Formal inspection summary
# --------------------------------------------------------------------------------------------------

print("PERSISTED IMAGE-UPLOAD CONTRACT INSPECTION")
print("-" * 110)
print(f"API contract              : {API_CONTRACT_PATH}")
print(f"Service configuration     : {API_CONFIG_SOURCE_PATH}")
print(f"Image-validation service  : {IMAGE_SERVICE_SOURCE_PATH}")

print()
print("API CONTRACT ENTRIES")
print("-" * 110)

if upload_contract_entries:
    for key, value in upload_contract_entries.items():
        print(f"{key:<65}: {value}")
else:
    print("No upload-related entries were identified.")

print()
print("SERVICE SETTINGS")
print("-" * 110)

if upload_setting_fields:
    for field_name, field_contract in upload_setting_fields.items():
        requirement = (
            "required"
            if field_contract["required"]
            else "optional"
        )

        print(
            f"{field_name:<35}: "
            f"type={field_contract['annotation']}, "
            f"{requirement}, "
            f"default={field_contract['default']!r}"
        )
else:
    print("No upload-related ServiceSettings fields were identified.")

print()
print("IMAGE-SERVICE CONSTANTS")
print("-" * 110)

if image_service_assignments:
    for name, value in image_service_assignments.items():
        print(f"{name:<35}: {value}")
else:
    print("No upload-related module constants were identified.")

print()
print("INSPECTION STATUS")
print("-" * 110)
print(
    f"API contract entries found       : "
    f"{len(upload_contract_entries)}"
)
print(
    f"Service setting fields found     : "
    f"{len(upload_setting_fields)}"
)
print(
    f"Image-service constants found    : "
    f"{len(image_service_assignments)}"
)
print("Backend services constructed     : NO")
print("Models loaded                    : NO")
print("Streamlit started                : NO")
print()
print("READY TO DEFINE THE UI UPLOAD CONTRACT FROM PERSISTED VALUES")


PERSISTED IMAGE-UPLOAD CONTRACT INSPECTION
--------------------------------------------------------------------------------------------------------------
API contract              : /home/jovyan/chest-xray-ai-assistant/configs/api_contract.yaml
Service configuration     : /home/jovyan/chest-xray-ai-assistant/api/core/config.py
Image-validation service  : /home/jovyan/chest-xray-ai-assistant/src/services/image_service.py

API CONTRACT ENTRIES
--------------------------------------------------------------------------------------------------------------
upload_contract.maximum_size_mib                                 : 10
upload_contract.supported_media_types                            : ['image/png', 'image/jpeg']
upload_contract.image_mode_handling                              : convert accepted images to RGB

SERVICE SETTINGS
--------------------------------------------------------------------------------------------------------------
maximum_upload_bytes               : type=<class 'i

## 9. Image Upload and Preview

This section adds the image-selection workflow using the persisted API upload contract. The interface accepts PNG and JPEG files, displays the selected image and file metadata, warns when the 10 MiB backend limit is exceeded, and preserves the selected upload in session state. The backend remains the authoritative image-validation boundary.


In [9]:
import ast
import py_compile
import textwrap
from pathlib import Path

import yaml


# --------------------------------------------------------------------------------------------------
# Read the confirmed upload contract
# --------------------------------------------------------------------------------------------------

persisted_upload_contract = api_contract_document[
    "upload_contract"
]

MAXIMUM_UPLOAD_MIB = persisted_upload_contract[
    "maximum_size_mib"
]

SUPPORTED_MEDIA_TYPES = tuple(
    persisted_upload_contract[
        "supported_media_types"
    ]
)

SUPPORTED_UPLOAD_EXTENSIONS = (
    "png",
    "jpg",
    "jpeg",
)

MAXIMUM_UPLOAD_BYTES = int(
    MAXIMUM_UPLOAD_MIB * 1024 * 1024
)


# --------------------------------------------------------------------------------------------------
# Align Streamlit's server limit with the persisted backend upload contract
# --------------------------------------------------------------------------------------------------

STREAMLIT_SERVER_CONFIG_PATH = (
    SOLUTION_ROOT / ".streamlit" / "config.toml"
)

STREAMLIT_SERVER_CONFIG_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

STREAMLIT_SERVER_CONFIG_PATH.write_text(
    "[server]\n"
    f"maxUploadSize = {MAXIMUM_UPLOAD_MIB}\n"
    "headless = true\n"
    "fileWatcherType = \"none\"\n"
    "\n[browser]\n"
    "gatherUsageStats = false\n",
    encoding="utf-8",
)


# --------------------------------------------------------------------------------------------------
# Extend the persistent UI configuration
# --------------------------------------------------------------------------------------------------

persisted_ui_configuration = yaml.safe_load(
    UI_CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)

persisted_ui_configuration["upload"] = {
    "maximum_size_mib": MAXIMUM_UPLOAD_MIB,
    "maximum_size_bytes": MAXIMUM_UPLOAD_BYTES,
    "supported_media_types": list(
        SUPPORTED_MEDIA_TYPES
    ),
    "supported_extensions": list(
        SUPPORTED_UPLOAD_EXTENSIONS
    ),
    "image_mode_handling": (
        persisted_upload_contract[
            "image_mode_handling"
        ]
    ),
}

UI_CONFIG_PATH.write_text(
    yaml.safe_dump(
        persisted_ui_configuration,
        sort_keys=False,
        allow_unicode=True,
    ),
    encoding="utf-8",
)


# --------------------------------------------------------------------------------------------------
# Extend the Streamlit application with upload and preview behavior
# --------------------------------------------------------------------------------------------------

streamlit_app_source = textwrap.dedent(
    '''
    from __future__ import annotations

    import hashlib
    import os
    from pathlib import Path
    from typing import Any

    import streamlit as st
    import yaml


    SOLUTION_ROOT = Path(__file__).resolve().parents[1]
    UI_CONFIG_PATH = SOLUTION_ROOT / "configs" / "ui_config.yaml"


    def load_ui_configuration(
        config_path: Path,
    ) -> dict[str, Any]:
        """Load the persisted Streamlit configuration."""
        if not config_path.is_file():
            raise FileNotFoundError(
                f"UI configuration was not found: {config_path}"
            )

        with config_path.open("r", encoding="utf-8") as file:
            configuration = yaml.safe_load(file) or {}

        if not isinstance(configuration, dict):
            raise TypeError(
                "The UI configuration must contain a YAML mapping."
            )

        return configuration


    def format_file_size(
        size_bytes: int,
    ) -> str:
        """Format an upload size using binary units."""
        size_mib = size_bytes / (1024 ** 2)
        return f"{size_mib:.2f} MiB"


    def initialize_session_state() -> None:
        """Initialize UI-local state without accessing backend services."""
        default_state = {
            "upload_signature": None,
            "uploaded_filename": None,
            "uploaded_media_type": None,
            "uploaded_image_bytes": None,
            "upload_ready": False,
            "analysis_response": None,
            "active_prediction_id": None,
            "question_response": None,
        }

        for key, default_value in default_state.items():
            if key not in st.session_state:
                st.session_state[key] = default_value


    ui_config = load_ui_configuration(
        UI_CONFIG_PATH
    )

    page_config = ui_config["page"]
    api_config = ui_config["api"]
    safety_config = ui_config["safety"]
    upload_config = ui_config["upload"]

    api_base_url = os.getenv(
        api_config["base_url_environment_variable"],
        api_config["default_base_url"],
    ).strip().rstrip("/")


    st.set_page_config(
        page_title=page_config["title"],
        page_icon=page_config["icon"],
        layout=page_config["layout"],
        initial_sidebar_state=(
            page_config["initial_sidebar_state"]
        ),
    )

    initialize_session_state()


    st.title(page_config["title"])
    st.caption(page_config["subtitle"])

    st.warning(
        safety_config["educational_limitation"],
        icon="⚠️",
    )

    st.markdown(
        """
        Upload a supported chest X-ray image to request a complete,
        API-driven analysis. Findings are determined only by the frozen
        computer-vision model and its persisted per-finding thresholds.
        Language outputs are generated only from the resulting structured
        evidence.
        """
    )


    with st.sidebar:
        st.header("System Boundary")

        st.info(
            "The interface communicates with the FastAPI backend through "
            "HTTP. It does not load the computer-vision model, language "
            "model, Grad-CAM service, or prediction store directly."
        )

        st.subheader("Backend Address")
        st.code(
            api_base_url,
            language=None,
        )

        st.subheader("Accepted Uploads")
        st.caption(
            "PNG or JPEG • "
            f"Maximum {upload_config['maximum_size_mib']} MiB"
        )

        st.subheader("Professional Review")
        st.caption(
            safety_config[
                "professional_review_guidance"
            ]
        )


    st.divider()
    st.subheader("1. Select a Chest X-Ray Image")

    uploaded_file = st.file_uploader(
        "Choose a PNG or JPEG image",
        type=upload_config[
            "supported_extensions"
        ],
        accept_multiple_files=False,
        help=(
            "The backend accepts image/png and image/jpeg files "
            f"up to {upload_config['maximum_size_mib']} MiB."
        ),
    )


    if uploaded_file is None:
        st.session_state.upload_signature = None
        st.session_state.uploaded_filename = None
        st.session_state.uploaded_media_type = None
        st.session_state.uploaded_image_bytes = None
        st.session_state.upload_ready = False
        st.session_state.analysis_response = None
        st.session_state.active_prediction_id = None
        st.session_state.question_response = None

        st.info(
            "Select an image to enable API submission."
        )

    else:
        uploaded_image_bytes = uploaded_file.getvalue()
        uploaded_size_bytes = len(
            uploaded_image_bytes
        )
        uploaded_media_type = (
            uploaded_file.type or ""
        )

        upload_signature = hashlib.sha256(
            uploaded_image_bytes
        ).hexdigest()

        if (
            st.session_state.upload_signature
            != upload_signature
        ):
            st.session_state.analysis_response = None
            st.session_state.active_prediction_id = None
            st.session_state.question_response = None

        st.session_state.upload_signature = (
            upload_signature
        )
        st.session_state.uploaded_filename = (
            uploaded_file.name
        )
        st.session_state.uploaded_media_type = (
            uploaded_media_type
        )
        st.session_state.uploaded_image_bytes = (
            uploaded_image_bytes
        )

        size_within_limit = (
            uploaded_size_bytes
            <= upload_config[
                "maximum_size_bytes"
            ]
        )

        media_type_supported = (
            uploaded_media_type
            in upload_config[
                "supported_media_types"
            ]
        )

        st.session_state.upload_ready = (
            size_within_limit
            and media_type_supported
        )

        preview_column, details_column = st.columns(
            [1.4, 1.0],
            gap="large",
        )

        with preview_column:
            st.image(
                uploaded_image_bytes,
                caption=uploaded_file.name,
                use_container_width=True,
            )

        with details_column:
            st.markdown("#### Selected Image")
            st.write(
                f"**Filename:** {uploaded_file.name}"
            )
            st.write(
                f"**Media type:** "
                f"{uploaded_media_type or 'Not reported'}"
            )
            st.write(
                f"**File size:** "
                f"{format_file_size(uploaded_size_bytes)}"
            )
            st.write(
                f"**Backend size limit:** "
                f"{upload_config['maximum_size_mib']} MiB"
            )

            if not size_within_limit:
                st.error(
                    "The selected file exceeds the backend's "
                    f"{upload_config['maximum_size_mib']} MiB "
                    "upload limit."
                )

            elif not media_type_supported:
                st.error(
                    "The selected file does not report a supported "
                    "PNG or JPEG media type."
                )

            else:
                st.success(
                    "The selected file is ready for secure "
                    "backend validation and analysis."
                )

        st.caption(
            "The preview and size check do not validate medical content. "
            "Image decoding and validation remain authoritative in the "
            "FastAPI backend."
        )
    '''
).strip() + "\n"

STREAMLIT_APP_PATH.write_text(
    streamlit_app_source,
    encoding="utf-8",
)


# --------------------------------------------------------------------------------------------------
# Validate the updated application source
# --------------------------------------------------------------------------------------------------

py_compile.compile(
    str(STREAMLIT_APP_PATH),
    doraise=True,
)

updated_streamlit_tree = ast.parse(
    STREAMLIT_APP_PATH.read_text(
        encoding="utf-8"
    ),
    filename=str(STREAMLIT_APP_PATH),
)

updated_function_names = tuple(
    node.name
    for node in updated_streamlit_tree.body
    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        ),
    )
)

persisted_upload_configuration = yaml.safe_load(
    UI_CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)["upload"]


# --------------------------------------------------------------------------------------------------
# Formal implementation summary
# --------------------------------------------------------------------------------------------------

print("IMAGE UPLOAD AND PREVIEW IMPLEMENTATION")
print("-" * 100)
print(f"Streamlit application       : {STREAMLIT_APP_PATH}")
print(f"UI configuration            : {UI_CONFIG_PATH}")
print(
    f"Supported media types       : "
    f"{', '.join(persisted_upload_configuration['supported_media_types'])}"
)
print(
    f"Supported extensions        : "
    f"{', '.join(persisted_upload_configuration['supported_extensions'])}"
)
print(
    f"Maximum upload size         : "
    f"{persisted_upload_configuration['maximum_size_mib']} MiB"
)
print(
    f"Maximum upload bytes        : "
    f"{persisted_upload_configuration['maximum_size_bytes']}"
)
print(
    f"Persisted UI functions      : "
    f"{', '.join(updated_function_names)}"
)
print(f"Session-state initialization: INCLUDED")
print(f"Image preview               : INCLUDED")
print(f"Backend validation boundary : PRESERVED")
print(f"Syntax compilation          : PASS")
print(f"Streamlit upload limit      : {MAXIMUM_UPLOAD_MIB} MiB")
print(f"Streamlit started           : NO")
print()
print("READY FOR COMPLETE-ANALYSIS SUBMISSION WORKFLOW")


IMAGE UPLOAD AND PREVIEW IMPLEMENTATION
----------------------------------------------------------------------------------------------------
Streamlit application       : /home/jovyan/chest-xray-ai-assistant/ui/app.py
UI configuration            : /home/jovyan/chest-xray-ai-assistant/configs/ui_config.yaml
Supported media types       : image/png, image/jpeg
Supported extensions        : png, jpg, jpeg
Maximum upload size         : 10 MiB
Maximum upload bytes        : 10485760
Persisted UI functions      : load_ui_configuration, format_file_size, initialize_session_state
Session-state initialization: INCLUDED
Image preview               : INCLUDED
Backend validation boundary : PRESERVED
Syntax compilation          : PASS
Streamlit upload limit      : 10 MiB
Streamlit started           : NO

READY FOR COMPLETE-ANALYSIS SUBMISSION WORKFLOW


## 10. Complete-Analysis Submission Workflow

This section connects the Streamlit interface to the primary `POST /api/v1/analyze-complete` endpoint. The selected image and optional grounded question are submitted only when the user explicitly starts the workflow. Successful responses are preserved in session state, while controlled API failures are rendered safely without exposing internal paths, tracebacks, or backend exception details.


In [10]:
import ast
import py_compile
import textwrap


# --------------------------------------------------------------------------------------------------
# Extend the Streamlit application with the complete-analysis workflow
# --------------------------------------------------------------------------------------------------

streamlit_app_source = textwrap.dedent(
    '''
    from __future__ import annotations

    import hashlib
    import os
    from pathlib import Path
    from typing import Any

    import streamlit as st
    import yaml

    from api_client import (
        APIClientError,
        ChestXRayAPIClient,
    )


    SOLUTION_ROOT = Path(__file__).resolve().parents[1]
    UI_CONFIG_PATH = SOLUTION_ROOT / "configs" / "ui_config.yaml"


    def load_ui_configuration(
        config_path: Path,
    ) -> dict[str, Any]:
        """Load the persisted Streamlit configuration."""
        if not config_path.is_file():
            raise FileNotFoundError(
                f"UI configuration was not found: {config_path}"
            )

        with config_path.open("r", encoding="utf-8") as file:
            configuration = yaml.safe_load(file) or {}

        if not isinstance(configuration, dict):
            raise TypeError(
                "The UI configuration must contain a YAML mapping."
            )

        return configuration


    def format_file_size(
        size_bytes: int,
    ) -> str:
        """Format an upload size using binary units."""
        size_mib = size_bytes / (1024 ** 2)
        return f"{size_mib:.2f} MiB"


    def initialize_session_state() -> None:
        """Initialize UI-local workflow state."""
        default_state = {
            "upload_signature": None,
            "uploaded_filename": None,
            "uploaded_media_type": None,
            "uploaded_image_bytes": None,
            "upload_ready": False,
            "analysis_response": None,
            "active_prediction_id": None,
            "question_response": None,
            "last_api_error": None,
        }

        for key, default_value in default_state.items():
            if key not in st.session_state:
                st.session_state[key] = default_value


    @st.cache_resource(show_spinner=False)
    def get_api_client(
        base_url: str,
    ) -> ChestXRayAPIClient:
        """Create one reusable HTTP client for the Streamlit process."""
        return ChestXRayAPIClient(
            base_url=base_url,
            timeout_seconds=120.0,
        )


    def render_api_error(
        error_details: dict[str, Any],
    ) -> None:
        """Render only controlled, UI-safe API error information."""
        message = error_details.get(
            "message",
            "The request could not be completed.",
        )

        st.error(message)

        with st.expander(
            "Technical error details",
            expanded=False,
        ):
            st.write(
                f"**Error code:** "
                f"{error_details.get('error_code', 'UNKNOWN')}"
            )

            status_code = error_details.get(
                "status_code"
            )

            if status_code is not None:
                st.write(
                    f"**HTTP status:** {status_code}"
                )

            request_id = error_details.get(
                "request_id"
            )

            if request_id:
                st.write(
                    f"**Request ID:** `{request_id}`"
                )

            safe_details = error_details.get(
                "details"
            )

            if isinstance(safe_details, dict) and safe_details:
                st.write("**Details:**")
                st.json(safe_details)


    ui_config = load_ui_configuration(
        UI_CONFIG_PATH
    )

    page_config = ui_config["page"]
    api_config = ui_config["api"]
    safety_config = ui_config["safety"]
    upload_config = ui_config["upload"]

    api_base_url = os.getenv(
        api_config["base_url_environment_variable"],
        api_config["default_base_url"],
    ).strip().rstrip("/")


    st.set_page_config(
        page_title=page_config["title"],
        page_icon=page_config["icon"],
        layout=page_config["layout"],
        initial_sidebar_state=(
            page_config["initial_sidebar_state"]
        ),
    )

    initialize_session_state()


    st.title(page_config["title"])
    st.caption(page_config["subtitle"])

    st.warning(
        safety_config["educational_limitation"],
        icon="⚠️",
    )

    st.markdown(
        """
        Upload a supported chest X-ray image to request a complete,
        API-driven analysis. Findings are determined only by the frozen
        computer-vision model and its persisted per-finding thresholds.
        Language outputs are generated only from the resulting structured
        evidence.
        """
    )


    with st.sidebar:
        st.header("System Boundary")

        st.info(
            "The interface communicates with the FastAPI backend through "
            "HTTP. It does not load the computer-vision model, language "
            "model, Grad-CAM service, or prediction store directly."
        )

        st.subheader("Backend Address")
        st.code(
            api_base_url,
            language=None,
        )

        st.subheader("Accepted Uploads")
        st.caption(
            "PNG or JPEG • "
            f"Maximum {upload_config['maximum_size_mib']} MiB"
        )

        st.subheader("Professional Review")
        st.caption(
            safety_config[
                "professional_review_guidance"
            ]
        )


    st.divider()
    st.subheader("1. Select a Chest X-Ray Image")

    uploaded_file = st.file_uploader(
        "Choose a PNG or JPEG image",
        type=upload_config[
            "supported_extensions"
        ],
        accept_multiple_files=False,
        help=(
            "The backend accepts image/png and image/jpeg files "
            f"up to {upload_config['maximum_size_mib']} MiB."
        ),
    )


    if uploaded_file is None:
        st.session_state.upload_signature = None
        st.session_state.uploaded_filename = None
        st.session_state.uploaded_media_type = None
        st.session_state.uploaded_image_bytes = None
        st.session_state.upload_ready = False
        st.session_state.analysis_response = None
        st.session_state.active_prediction_id = None
        st.session_state.question_response = None
        st.session_state.last_api_error = None

        st.info(
            "Select an image to enable API submission."
        )

    else:
        uploaded_image_bytes = uploaded_file.getvalue()
        uploaded_size_bytes = len(
            uploaded_image_bytes
        )
        uploaded_media_type = (
            uploaded_file.type or ""
        )

        upload_signature = hashlib.sha256(
            uploaded_image_bytes
        ).hexdigest()

        if (
            st.session_state.upload_signature
            != upload_signature
        ):
            st.session_state.analysis_response = None
            st.session_state.active_prediction_id = None
            st.session_state.question_response = None
            st.session_state.last_api_error = None

        st.session_state.upload_signature = (
            upload_signature
        )
        st.session_state.uploaded_filename = (
            uploaded_file.name
        )
        st.session_state.uploaded_media_type = (
            uploaded_media_type
        )
        st.session_state.uploaded_image_bytes = (
            uploaded_image_bytes
        )

        size_within_limit = (
            uploaded_size_bytes
            <= upload_config[
                "maximum_size_bytes"
            ]
        )

        media_type_supported = (
            uploaded_media_type
            in upload_config[
                "supported_media_types"
            ]
        )

        st.session_state.upload_ready = (
            size_within_limit
            and media_type_supported
        )

        preview_column, details_column = st.columns(
            [1.4, 1.0],
            gap="large",
        )

        with preview_column:
            st.image(
                uploaded_image_bytes,
                caption=uploaded_file.name,
                use_container_width=True,
            )

        with details_column:
            st.markdown("#### Selected Image")
            st.write(
                f"**Filename:** {uploaded_file.name}"
            )
            st.write(
                f"**Media type:** "
                f"{uploaded_media_type or 'Not reported'}"
            )
            st.write(
                f"**File size:** "
                f"{format_file_size(uploaded_size_bytes)}"
            )
            st.write(
                f"**Backend size limit:** "
                f"{upload_config['maximum_size_mib']} MiB"
            )

            if not size_within_limit:
                st.error(
                    "The selected file exceeds the backend's "
                    f"{upload_config['maximum_size_mib']} MiB "
                    "upload limit."
                )

            elif not media_type_supported:
                st.error(
                    "The selected file does not report a supported "
                    "PNG or JPEG media type."
                )

            else:
                st.success(
                    "The selected file is ready for secure "
                    "backend validation and analysis."
                )

        st.caption(
            "The preview and size check do not validate medical content. "
            "Image decoding and validation remain authoritative in the "
            "FastAPI backend."
        )


    st.divider()
    st.subheader("2. Run Complete Analysis")

    initial_question = st.text_area(
        "Optional grounded question",
        placeholder=(
            "Example: Which findings crossed their frozen thresholds?"
        ),
        help=(
            "The question is answered only from the model findings, "
            "probabilities, thresholds, approved descriptions, and "
            "safety limitations."
        ),
        disabled=not st.session_state.upload_ready,
    )

    submit_analysis = st.button(
        "Run Complete Analysis",
        type="primary",
        use_container_width=True,
        disabled=not st.session_state.upload_ready,
    )

    if submit_analysis:
        st.session_state.last_api_error = None
        st.session_state.question_response = None

        try:
            with st.spinner(
                "Submitting the image to the FastAPI analysis workflow..."
            ):
                api_client = get_api_client(
                    api_base_url
                )

                analysis_response = (
                    api_client.analyze_complete(
                        filename=st.session_state.uploaded_filename,
                        media_type=st.session_state.uploaded_media_type,
                        image_content=st.session_state.uploaded_image_bytes,
                        question=(
                            initial_question
                            if initial_question.strip()
                            else None
                        ),
                    )
                )

            st.session_state.analysis_response = (
                analysis_response
            )

            st.session_state.active_prediction_id = (
                analysis_response.get(
                    "prediction_id"
                )
            )

        except APIClientError as exc:
            st.session_state.analysis_response = None
            st.session_state.active_prediction_id = None
            st.session_state.last_api_error = (
                exc.to_display_dict()
            )

        except ValueError as exc:
            st.session_state.analysis_response = None
            st.session_state.active_prediction_id = None
            st.session_state.last_api_error = {
                "error_code": "INVALID_UI_REQUEST",
                "message": str(exc),
                "status_code": None,
                "request_id": None,
                "details": {},
            }

        except Exception:
            st.session_state.analysis_response = None
            st.session_state.active_prediction_id = None
            st.session_state.last_api_error = {
                "error_code": "UI_EXECUTION_ERROR",
                "message": (
                    "The analysis request could not be completed. "
                    "Confirm that the backend is healthy and try again."
                ),
                "status_code": None,
                "request_id": None,
                "details": {},
            }


    if st.session_state.last_api_error:
        render_api_error(
            st.session_state.last_api_error
        )


    if st.session_state.analysis_response:
        active_response = (
            st.session_state.analysis_response
        )

        st.success(
            "The complete-analysis response was received and "
            "preserved in this interface session."
        )

        metadata_columns = st.columns(4)

        metadata_columns[0].metric(
            "Status",
            str(
                active_response.get(
                    "status",
                    "success",
                )
            ),
        )

        metadata_columns[1].metric(
            "Request Latency",
            (
                f"{active_response.get('latency_ms', 0):.2f} ms"
                if isinstance(
                    active_response.get("latency_ms"),
                    (int, float),
                )
                else "Unavailable"
            ),
        )

        metadata_columns[2].metric(
            "Prediction ID",
            "Available"
            if st.session_state.active_prediction_id
            else "Unavailable",
        )

        metadata_columns[3].metric(
            "Language Outputs",
            len(
                active_response.get(
                    "language_outputs",
                    [],
                )
            ),
        )

        with st.expander(
            "Request and lineage identifiers",
            expanded=False,
        ):
            st.write(
                f"**Request ID:** "
                f"`{active_response.get('request_id', 'Unavailable')}`"
            )
            st.write(
                f"**Prediction ID:** "
                f"`{active_response.get('prediction_id', 'Unavailable')}`"
            )
            st.write(
                f"**API version:** "
                f"{active_response.get('api_version', 'Unavailable')}"
            )
            st.write(
                f"**Prompt-registry version:** "
                f"{active_response.get('prompt_registry_version', 'Unavailable')}"
            )
            st.write("**Model versions:**")
            st.json(
                active_response.get(
                    "model_versions",
                    {},
                )
            )
    '''
).strip() + "\n"

STREAMLIT_APP_PATH.write_text(
    streamlit_app_source,
    encoding="utf-8",
)


# --------------------------------------------------------------------------------------------------
# Validate the updated application source
# --------------------------------------------------------------------------------------------------

py_compile.compile(
    str(STREAMLIT_APP_PATH),
    doraise=True,
)

submission_tree = ast.parse(
    STREAMLIT_APP_PATH.read_text(
        encoding="utf-8"
    ),
    filename=str(STREAMLIT_APP_PATH),
)

submission_function_names = tuple(
    node.name
    for node in submission_tree.body
    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        ),
    )
)


# --------------------------------------------------------------------------------------------------
# Formal implementation summary
# --------------------------------------------------------------------------------------------------

print("COMPLETE-ANALYSIS SUBMISSION WORKFLOW")
print("-" * 100)
print(f"Streamlit application       : {STREAMLIT_APP_PATH}")
print(f"Primary API endpoint        : POST /api/v1/analyze-complete")
print(f"Multipart image field       : image")
print(f"Optional question field     : question")
print(f"HTTP client boundary        : ChestXRayAPIClient")
print(f"Controlled API errors       : INCLUDED")
print(f"Session response persistence: INCLUDED")
print(f"Prediction ID persistence   : INCLUDED")
print(f"Request and lineage details : INCLUDED")
print(
    f"Persisted UI functions      : "
    f"{', '.join(submission_function_names)}"
)
print(f"Syntax compilation          : PASS")
print(f"Duplicate inference calls   : NOT INTRODUCED")
print(f"Streamlit started           : NO")
print()
print("READY FOR FINDING-SUMMARY RENDERING")


COMPLETE-ANALYSIS SUBMISSION WORKFLOW
----------------------------------------------------------------------------------------------------
Streamlit application       : /home/jovyan/chest-xray-ai-assistant/ui/app.py
Primary API endpoint        : POST /api/v1/analyze-complete
Multipart image field       : image
Optional question field     : question
HTTP client boundary        : ChestXRayAPIClient
Controlled API errors       : INCLUDED
Session response persistence: INCLUDED
Prediction ID persistence   : INCLUDED
Request and lineage details : INCLUDED
Persisted UI functions      : load_ui_configuration, format_file_size, initialize_session_state, get_api_client, render_api_error
Syntax compilation          : PASS
Duplicate inference calls   : NOT INTRODUCED
Streamlit started           : NO

READY FOR FINDING-SUMMARY RENDERING


## 11. Finding Summary and Threshold Interpretation

This section adds a reusable finding-summary component for the complete-analysis response. It presents all fourteen findings with their display names, probabilities, frozen thresholds, crossed-threshold decisions, and confidence categories. The no-target-finding state is worded cautiously and is never presented as evidence of a clinically normal radiograph.


In [11]:
import ast
import py_compile
import textwrap
from pathlib import Path


# --------------------------------------------------------------------------------------------------
# Persist the finding-rendering component
# --------------------------------------------------------------------------------------------------

COMPONENTS_PATH = UI_ROOT / "components.py"

components_source = textwrap.dedent(
    '''
    from __future__ import annotations

    from typing import Any

    import pandas as pd
    import streamlit as st


    def build_finding_table(
        findings: list[dict[str, Any]],
    ) -> pd.DataFrame:
        """Build an ordered table from authoritative API finding records."""
        rows = []

        for finding in findings:
            rows.append(
                {
                    "Label ID": finding.get("label_id"),
                    "Finding": finding.get(
                        "display_name",
                        finding.get("label_name", "Unavailable"),
                    ),
                    "Probability": finding.get("probability"),
                    "Frozen Threshold": finding.get(
                        "frozen_threshold"
                    ),
                    "Crossed Threshold": finding.get(
                        "crossed_threshold"
                    ),
                    "Confidence": finding.get(
                        "confidence_category",
                        "Unavailable",
                    ),
                }
            )

        finding_table = pd.DataFrame(rows)

        if (
            not finding_table.empty
            and "Label ID" in finding_table.columns
        ):
            finding_table = (
                finding_table
                .sort_values("Label ID")
                .reset_index(drop=True)
            )

        return finding_table


    def render_finding_summary(
        response: dict[str, Any],
    ) -> None:
        """Render model findings without adding clinical interpretation."""
        st.divider()
        st.subheader("3. Finding Summary")

        findings = response.get(
            "findings",
            [],
        )

        crossed_finding_names = response.get(
            "crossed_finding_names",
            [],
        )

        no_target_finding = response.get(
            "no_target_finding",
            False,
        )

        interpretation = response.get(
            "interpretation",
            "",
        )

        if not isinstance(findings, list):
            st.error(
                "The API response does not contain a valid finding list."
            )
            return

        finding_table = build_finding_table(
            findings
        )

        summary_columns = st.columns(3)

        summary_columns[0].metric(
            "Target Findings Evaluated",
            len(findings),
        )

        summary_columns[1].metric(
            "Thresholds Crossed",
            len(crossed_finding_names)
            if isinstance(crossed_finding_names, list)
            else 0,
        )

        summary_columns[2].metric(
            "No Target Finding",
            "Yes" if no_target_finding else "No",
        )

        if no_target_finding:
            st.info(
                "None of the fourteen ChestMNIST target findings crossed "
                "their frozen thresholds. This state must not be interpreted "
                "as confirmation of a clinically normal chest radiograph."
            )
        else:
            crossed_display = (
                ", ".join(crossed_finding_names)
                if isinstance(crossed_finding_names, list)
                else "Unavailable"
            )

            st.warning(
                "One or more model outputs crossed their frozen "
                f"thresholds: {crossed_display}. Threshold crossing is "
                "a model decision and is not a clinical diagnosis."
            )

        if interpretation:
            st.markdown("#### Backend Interpretation")
            st.write(interpretation)

        if finding_table.empty:
            st.warning(
                "No finding records were returned by the backend."
            )
            return

        st.dataframe(
            finding_table,
            hide_index=True,
            use_container_width=True,
            column_config={
                "Label ID": st.column_config.NumberColumn(
                    "Label ID",
                    format="%d",
                ),
                "Finding": st.column_config.TextColumn(
                    "Finding",
                ),
                "Probability": st.column_config.NumberColumn(
                    "Probability",
                    format="%.4f",
                ),
                "Frozen Threshold": st.column_config.NumberColumn(
                    "Frozen Threshold",
                    format="%.4f",
                ),
                "Crossed Threshold": st.column_config.CheckboxColumn(
                    "Crossed Threshold",
                ),
                "Confidence": st.column_config.TextColumn(
                    "Confidence",
                ),
            },
        )

        crossed_findings = [
            finding
            for finding in findings
            if finding.get("crossed_threshold") is True
        ]

        if crossed_findings:
            with st.expander(
                "Approved descriptions for crossed findings",
                expanded=False,
            ):
                for finding in crossed_findings:
                    display_name = finding.get(
                        "display_name",
                        finding.get(
                            "label_name",
                            "Finding",
                        ),
                    )

                    approved_description = finding.get(
                        "approved_description",
                        "No approved description was returned.",
                    )

                    st.markdown(
                        f"**{display_name}**"
                    )
                    st.write(
                        approved_description
                    )
    '''
).strip() + "\n"

COMPONENTS_PATH.write_text(
    components_source,
    encoding="utf-8",
)

py_compile.compile(
    str(COMPONENTS_PATH),
    doraise=True,
)


# --------------------------------------------------------------------------------------------------
# Connect the component to the persisted Streamlit application
# --------------------------------------------------------------------------------------------------

current_app_source = STREAMLIT_APP_PATH.read_text(
    encoding="utf-8"
)

component_import = (
    "from components import render_finding_summary"
)

if component_import not in current_app_source:
    import_anchor = textwrap.dedent(
        '''
        from api_client import (
            APIClientError,
            ChestXRayAPIClient,
        )
        '''
    ).strip()

    import_replacement = (
        import_anchor
        + "\n\n"
        + component_import
    )

    if current_app_source.count(import_anchor) != 1:
        raise RuntimeError(
            "The API-client import location could not be identified "
            "unambiguously in the persisted Streamlit application."
        )

    current_app_source = current_app_source.replace(
        import_anchor,
        import_replacement,
        1,
    )

finding_render_call = (
    "        render_finding_summary(active_response)"
)

if finding_render_call not in current_app_source:
    current_app_source = (
        current_app_source.rstrip()
        + "\n\n"
        + finding_render_call
        + "\n"
    )

STREAMLIT_APP_PATH.write_text(
    current_app_source,
    encoding="utf-8",
)

py_compile.compile(
    str(STREAMLIT_APP_PATH),
    doraise=True,
)


# --------------------------------------------------------------------------------------------------
# Inspect the persisted component definitions
# --------------------------------------------------------------------------------------------------

components_tree = ast.parse(
    COMPONENTS_PATH.read_text(
        encoding="utf-8"
    ),
    filename=str(COMPONENTS_PATH),
)

component_function_names = tuple(
    node.name
    for node in components_tree.body
    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        ),
    )
)

updated_app_source = STREAMLIT_APP_PATH.read_text(
    encoding="utf-8"
)


# --------------------------------------------------------------------------------------------------
# Formal implementation summary
# --------------------------------------------------------------------------------------------------

print("FINDING SUMMARY AND THRESHOLD INTERPRETATION")
print("-" * 100)
print(f"Components module          : {COMPONENTS_PATH}")
print(f"Streamlit application      : {STREAMLIT_APP_PATH}")
print(
    f"Persisted component methods: "
    f"{', '.join(component_function_names)}"
)
print(f"Finding display name       : INCLUDED")
print(f"Probability                : INCLUDED")
print(f"Frozen threshold           : INCLUDED")
print(f"Crossed-threshold status   : INCLUDED")
print(f"Confidence category        : INCLUDED")
print(f"Approved descriptions      : CROSSED FINDINGS")
print(f"No-target wording          : CAUTIOUS")
print(f"Clinical-normality claim   : NOT USED")
print(
    f"Component connected to app: "
    f"{finding_render_call in updated_app_source}"
)
print(f"Component syntax           : PASS")
print(f"Application syntax         : PASS")
print(f"Streamlit started          : NO")
print()
print("READY FOR GRAD-CAM VISUAL-EVIDENCE RENDERING")


FINDING SUMMARY AND THRESHOLD INTERPRETATION
----------------------------------------------------------------------------------------------------
Components module          : /home/jovyan/chest-xray-ai-assistant/ui/components.py
Streamlit application      : /home/jovyan/chest-xray-ai-assistant/ui/app.py
Persisted component methods: build_finding_table, render_finding_summary
Finding display name       : INCLUDED
Probability                : INCLUDED
Frozen threshold           : INCLUDED
Crossed-threshold status   : INCLUDED
Confidence category        : INCLUDED
Approved descriptions      : CROSSED FINDINGS
No-target wording          : CAUTIOUS
Clinical-normality claim   : NOT USED
Component connected to app: True
Component syntax           : PASS
Application syntax         : PASS
Streamlit started          : NO

READY FOR GRAD-CAM VISUAL-EVIDENCE RENDERING


## 12. Grad-CAM Visual-Evidence Rendering

This section renders Grad-CAM overlays only for findings whose frozen thresholds were crossed. The component decodes the API-provided evidence without accessing the image model or generating attributions locally. The authoritative Grad-CAM limitation is displayed beside the evidence, and the visualizations are not described as lesion localization, anatomical confirmation, causal proof, or diagnosis.


In [12]:
import ast
import py_compile
import textwrap


# --------------------------------------------------------------------------------------------------
# Extend the persisted components module
# --------------------------------------------------------------------------------------------------

current_components_source = COMPONENTS_PATH.read_text(
    encoding="utf-8"
)

base64_import_anchor = (
    "from __future__ import annotations\n\n"
)

base64_import_block = (
    "from __future__ import annotations\n\n"
    "import base64\n"
    "import binascii\n"
)

if "import base64" not in current_components_source:
    if current_components_source.count(
        base64_import_anchor
    ) != 1:
        raise RuntimeError(
            "The components import location could not be identified "
            "unambiguously."
        )

    current_components_source = (
        current_components_source.replace(
            base64_import_anchor,
            base64_import_block,
            1,
        )
    )


visual_evidence_functions = textwrap.dedent(
    '''

    def decode_base64_image(
        encoded_image: Any,
    ) -> bytes | None:
        """Decode an API-provided base64 image without modifying it."""
        if not isinstance(encoded_image, str):
            return None

        normalized_value = encoded_image.strip()

        if not normalized_value:
            return None

        if "," in normalized_value and normalized_value.startswith(
            "data:"
        ):
            normalized_value = normalized_value.split(
                ",",
                maxsplit=1,
            )[1]

        try:
            decoded_image = base64.b64decode(
                normalized_value
            )
        except (
            ValueError,
            binascii.Error,
        ):
            return None

        return decoded_image or None


    def render_visual_evidence(
        response: dict[str, Any],
        fallback_limitation: str,
    ) -> None:
        """Render API-provided Grad-CAM only for crossed findings."""
        st.divider()
        st.subheader("4. Grad-CAM Visual Evidence")

        explainability = response.get(
            "explainability",
            {},
        )

        visual_evidence = response.get(
            "visual_evidence",
            [],
        )

        no_target_finding = response.get(
            "no_target_finding",
            False,
        )

        if not isinstance(explainability, dict):
            explainability = {}

        if not isinstance(visual_evidence, list):
            visual_evidence = []

        crossed_visual_evidence = [
            evidence
            for evidence in visual_evidence
            if isinstance(evidence, dict)
            and evidence.get("crossed_threshold") is True
        ]

        limitation = explainability.get(
            "limitation"
        )

        if not isinstance(limitation, str) or not limitation.strip():
            limitation = fallback_limitation

        st.warning(
            limitation,
            icon="⚠️",
        )

        method_columns = st.columns(2)

        method_columns[0].write(
            f"**Method:** "
            f"{explainability.get('method', 'Unavailable')}"
        )

        method_columns[1].write(
            f"**Target layer:** "
            f"{explainability.get('target_layer', 'Unavailable')}"
        )

        if no_target_finding:
            st.info(
                "No Grad-CAM overlay is displayed because none of the "
                "fourteen target findings crossed its frozen threshold."
            )
            return

        if not crossed_visual_evidence:
            st.warning(
                "No crossed-finding visual evidence was returned by "
                "the backend."
            )
            return

        st.caption(
            "The following images show regions that influenced each "
            "crossed model output. They do not identify or confirm a "
            "lesion or anatomical abnormality."
        )

        for evidence in crossed_visual_evidence:
            finding_name = evidence.get(
                "finding_name",
                "Finding",
            )

            probability = evidence.get(
                "probability"
            )

            frozen_threshold = evidence.get(
                "frozen_threshold"
            )

            with st.expander(
                f"{finding_name} visual evidence",
                expanded=True,
            ):
                evidence_columns = st.columns(
                    2,
                    gap="large",
                )

                overlay_bytes = decode_base64_image(
                    evidence.get(
                        "overlay_png_base64"
                    )
                )

                heatmap_bytes = decode_base64_image(
                    evidence.get(
                        "heatmap_png_base64"
                    )
                )

                with evidence_columns[0]:
                    st.markdown(
                        "**Grad-CAM Overlay**"
                    )

                    if overlay_bytes is not None:
                        st.image(
                            overlay_bytes,
                            caption=(
                                f"{finding_name} model-influence overlay"
                            ),
                            use_container_width=True,
                        )
                    else:
                        st.error(
                            "The overlay image could not be decoded."
                        )

                with evidence_columns[1]:
                    st.markdown(
                        "**Attribution Heatmap**"
                    )

                    if heatmap_bytes is not None:
                        st.image(
                            heatmap_bytes,
                            caption=(
                                f"{finding_name} attribution heatmap"
                            ),
                            use_container_width=True,
                        )
                    else:
                        st.error(
                            "The heatmap image could not be decoded."
                        )

                detail_columns = st.columns(3)

                detail_columns[0].metric(
                    "Probability",
                    (
                        f"{probability:.4f}"
                        if isinstance(
                            probability,
                            (int, float),
                        )
                        else "Unavailable"
                    ),
                )

                detail_columns[1].metric(
                    "Frozen Threshold",
                    (
                        f"{frozen_threshold:.4f}"
                        if isinstance(
                            frozen_threshold,
                            (int, float),
                        )
                        else "Unavailable"
                    ),
                )

                footprint = evidence.get(
                    "high_attribution_area_percent"
                )

                detail_columns[2].metric(
                    "High-Attribution Area",
                    (
                        f"{footprint:.2f}%"
                        if isinstance(
                            footprint,
                            (int, float),
                        )
                        else "Not provided"
                    ),
                )

                st.caption(limitation)
    '''
).rstrip()

if (
    "def render_visual_evidence("
    not in current_components_source
):
    current_components_source = (
        current_components_source.rstrip()
        + "\n"
        + visual_evidence_functions
        + "\n"
    )

COMPONENTS_PATH.write_text(
    current_components_source,
    encoding="utf-8",
)

py_compile.compile(
    str(COMPONENTS_PATH),
    doraise=True,
)


# --------------------------------------------------------------------------------------------------
# Connect visual-evidence rendering to the Streamlit application
# --------------------------------------------------------------------------------------------------

current_app_source = STREAMLIT_APP_PATH.read_text(
    encoding="utf-8"
)

existing_component_import = (
    "from components import render_finding_summary"
)

updated_component_import = textwrap.dedent(
    '''
    from components import (
        render_finding_summary,
        render_visual_evidence,
    )
    '''
).strip()

if existing_component_import in current_app_source:
    current_app_source = current_app_source.replace(
        existing_component_import,
        updated_component_import,
        1,
    )

visual_evidence_call = textwrap.dedent(
    '''
            render_visual_evidence(
                active_response,
                safety_config["gradcam_limitation"],
            )
    '''
).strip("\n")

if (
    "render_visual_evidence("
    not in current_app_source
):
    finding_call = (
        "        render_finding_summary(active_response)"
    )

    if current_app_source.count(finding_call) != 1:
        raise RuntimeError(
            "The finding-summary render location could not be "
            "identified unambiguously."
        )

    current_app_source = current_app_source.replace(
        finding_call,
        finding_call
        + "\n\n"
        + visual_evidence_call,
        1,
    )

STREAMLIT_APP_PATH.write_text(
    current_app_source,
    encoding="utf-8",
)

py_compile.compile(
    str(STREAMLIT_APP_PATH),
    doraise=True,
)


# --------------------------------------------------------------------------------------------------
# Inspect persisted definitions
# --------------------------------------------------------------------------------------------------

components_tree = ast.parse(
    COMPONENTS_PATH.read_text(
        encoding="utf-8"
    ),
    filename=str(COMPONENTS_PATH),
)

component_function_names = tuple(
    node.name
    for node in components_tree.body
    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        ),
    )
)

updated_app_source = STREAMLIT_APP_PATH.read_text(
    encoding="utf-8"
)


# --------------------------------------------------------------------------------------------------
# Formal implementation summary
# --------------------------------------------------------------------------------------------------

print("GRAD-CAM VISUAL-EVIDENCE RENDERING")
print("-" * 100)
print(f"Components module            : {COMPONENTS_PATH}")
print(f"Streamlit application        : {STREAMLIT_APP_PATH}")
print(f"Base64 evidence decoding     : INCLUDED")
print(f"Crossed-finding filter       : INCLUDED")
print(f"Grad-CAM overlays            : INCLUDED")
print(f"Attribution heatmaps         : INCLUDED")
print(f"Explainability metadata      : INCLUDED")
print(f"Grad-CAM limitation          : ADJACENT TO EVIDENCE")
print(f"Lesion-confirmation claim    : NOT USED")
print(f"Anatomical-confirmation claim: NOT USED")
print(
    f"Visual component connected  : "
    f"{'render_visual_evidence(' in updated_app_source}"
)
print(
    f"Persisted component methods  : "
    f"{', '.join(component_function_names)}"
)
print(f"Component syntax             : PASS")
print(f"Application syntax           : PASS")
print(f"Streamlit started            : NO")
print()
print("READY FOR GROUNDED-LANGUAGE OUTPUT RENDERING")


GRAD-CAM VISUAL-EVIDENCE RENDERING
----------------------------------------------------------------------------------------------------
Components module            : /home/jovyan/chest-xray-ai-assistant/ui/components.py
Streamlit application        : /home/jovyan/chest-xray-ai-assistant/ui/app.py
Base64 evidence decoding     : INCLUDED
Crossed-finding filter       : INCLUDED
Grad-CAM overlays            : INCLUDED
Attribution heatmaps         : INCLUDED
Explainability metadata      : INCLUDED
Grad-CAM limitation          : ADJACENT TO EVIDENCE
Lesion-confirmation claim    : NOT USED
Anatomical-confirmation claim: NOT USED
Visual component connected  : True
Persisted component methods  : build_finding_table, render_finding_summary, decode_base64_image, render_visual_evidence
Component syntax             : PASS
Application syntax           : PASS
Streamlit started            : NO

READY FOR GROUNDED-LANGUAGE OUTPUT RENDERING


## 13. Grounded-Language Output Rendering

This section presents the structured preliminary report, plain-language explanation, educational follow-up guidance, and optional grounded question answer returned by the complete-analysis endpoint. Each output includes its guardrail action, fallback triggers, token count, and generation latency in a separate technical area. No language output is generated or modified by the interface.


In [13]:
import ast
import py_compile
import textwrap
from typing import Any


# --------------------------------------------------------------------------------------------------
# Extend the persisted components module
# --------------------------------------------------------------------------------------------------

current_components_source = COMPONENTS_PATH.read_text(
    encoding="utf-8"
)

language_component_source = textwrap.dedent(
    '''

    LANGUAGE_TASK_LABELS = {
        "structured_report": "Structured Preliminary Report",
        "plain_language_explanation": "Plain-Language Explanation",
        "educational_follow_up": "Educational Follow-Up",
        "grounded_question_answering": "Grounded Question Answer",
    }

    LANGUAGE_TASK_ORDER = (
        "structured_report",
        "plain_language_explanation",
        "educational_follow_up",
        "grounded_question_answering",
    )


    def render_language_outputs(
        response: dict[str, Any],
    ) -> None:
        """Render API-grounded language outputs and guardrail metadata."""
        st.divider()
        st.subheader("5. Grounded Language Outputs")

        language_outputs = response.get(
            "language_outputs",
            [],
        )

        if not isinstance(language_outputs, list):
            st.error(
                "The API response does not contain a valid "
                "language-output collection."
            )
            return

        valid_outputs = [
            output
            for output in language_outputs
            if isinstance(output, dict)
        ]

        if not valid_outputs:
            st.info(
                "No grounded language outputs were returned "
                "for this prediction."
            )
            return

        task_order = {
            task_name: index
            for index, task_name in enumerate(
                LANGUAGE_TASK_ORDER
            )
        }

        ordered_outputs = sorted(
            valid_outputs,
            key=lambda output: task_order.get(
                output.get("task_type"),
                len(task_order),
            ),
        )

        tab_labels = [
            LANGUAGE_TASK_LABELS.get(
                output.get("task_type"),
                str(
                    output.get(
                        "task_type",
                        "Language Output",
                    )
                ).replace("_", " ").title(),
            )
            for output in ordered_outputs
        ]

        output_tabs = st.tabs(
            tab_labels
        )

        for output_tab, output in zip(
            output_tabs,
            ordered_outputs,
        ):
            with output_tab:
                task_type = output.get(
                    "task_type",
                    "unknown",
                )

                question = output.get(
                    "question"
                )

                if (
                    task_type
                    == "grounded_question_answering"
                    and isinstance(question, str)
                    and question.strip()
                ):
                    st.markdown(
                        "**Grounded question**"
                    )
                    st.write(question)

                st.markdown(
                    "**API-generated output**"
                )

                output_text = output.get(
                    "output_text",
                    "No output text was returned.",
                )

                st.write(output_text)

                guardrail_action = output.get(
                    "guardrail_action",
                    "Unavailable",
                )

                trigger_reasons = output.get(
                    "trigger_reasons",
                    [],
                )

                if (
                    guardrail_action
                    == "safe_template_fallback"
                ):
                    st.info(
                        "The deterministic language guardrail "
                        "replaced the raw model generation with "
                        "a grounded safety template."
                    )

                with st.expander(
                    "Generation and guardrail details",
                    expanded=False,
                ):
                    detail_columns = st.columns(3)

                    detail_columns[0].metric(
                        "Generated Tokens",
                        output.get(
                            "generated_tokens",
                            "Unavailable",
                        ),
                    )

                    generation_latency = output.get(
                        "generation_latency_ms"
                    )

                    detail_columns[1].metric(
                        "Generation Latency",
                        (
                            f"{generation_latency:.2f} ms"
                            if isinstance(
                                generation_latency,
                                (int, float),
                            )
                            else "Unavailable"
                        ),
                    )

                    detail_columns[2].metric(
                        "Guardrail Action",
                        str(guardrail_action),
                    )

                    st.write(
                        f"**Task type:** `{task_type}`"
                    )

                    if (
                        isinstance(trigger_reasons, list)
                        and trigger_reasons
                    ):
                        st.write(
                            "**Fallback trigger reasons:**"
                        )

                        for reason in trigger_reasons:
                            st.write(
                                f"- `{reason}`"
                            )
                    else:
                        st.write(
                            "**Fallback trigger reasons:** None"
                        )

        st.caption(
            "All displayed language outputs are grounded only in the "
            "structured API evidence and remain subject to the educational "
            "use limitation shown at the top of the interface."
        )
    '''
).rstrip()

if (
    "def render_language_outputs("
    not in current_components_source
):
    current_components_source = (
        current_components_source.rstrip()
        + "\n"
        + language_component_source
        + "\n"
    )

COMPONENTS_PATH.write_text(
    current_components_source,
    encoding="utf-8",
)

py_compile.compile(
    str(COMPONENTS_PATH),
    doraise=True,
)


# --------------------------------------------------------------------------------------------------
# Update imports and ensure render calls remain inside the response block
# --------------------------------------------------------------------------------------------------

current_app_source = STREAMLIT_APP_PATH.read_text(
    encoding="utf-8"
)

existing_component_import = textwrap.dedent(
    '''
    from components import (
        render_finding_summary,
        render_visual_evidence,
    )
    '''
).strip()

updated_component_import = textwrap.dedent(
    '''
    from components import (
        render_finding_summary,
        render_language_outputs,
        render_visual_evidence,
    )
    '''
).strip()

if existing_component_import in current_app_source:
    current_app_source = current_app_source.replace(
        existing_component_import,
        updated_component_import,
        1,
    )


# Correct the visual-evidence call scope if it was placed at module level.
incorrect_visual_call = textwrap.dedent(
    '''
    render_visual_evidence(
        active_response,
        safety_config["gradcam_limitation"],
    )
    '''
).strip()

correct_visual_call = textwrap.indent(
    incorrect_visual_call,
    "        ",
)

if (
    f"\n{incorrect_visual_call}\n"
    in current_app_source
):
    current_app_source = current_app_source.replace(
        f"\n{incorrect_visual_call}\n",
        f"\n{correct_visual_call}\n",
        1,
    )


language_render_call = (
    "        render_language_outputs(active_response)"
)

if language_render_call not in current_app_source:
    if current_app_source.count(
        correct_visual_call
    ) != 1:
        raise RuntimeError(
            "The visual-evidence render location could not be "
            "identified unambiguously."
        )

    current_app_source = current_app_source.replace(
        correct_visual_call,
        correct_visual_call
        + "\n\n"
        + language_render_call,
        1,
    )

STREAMLIT_APP_PATH.write_text(
    current_app_source,
    encoding="utf-8",
)

py_compile.compile(
    str(STREAMLIT_APP_PATH),
    doraise=True,
)


# --------------------------------------------------------------------------------------------------
# Verify that response renderers are not executed at module level
# --------------------------------------------------------------------------------------------------

application_tree = ast.parse(
    STREAMLIT_APP_PATH.read_text(
        encoding="utf-8"
    ),
    filename=str(STREAMLIT_APP_PATH),
)

response_renderer_names = {
    "render_finding_summary",
    "render_visual_evidence",
    "render_language_outputs",
}

module_level_renderer_calls = []

for node in application_tree.body:
    if (
        isinstance(node, ast.Expr)
        and isinstance(node.value, ast.Call)
        and isinstance(node.value.func, ast.Name)
        and node.value.func.id
        in response_renderer_names
    ):
        module_level_renderer_calls.append(
            node.value.func.id
        )

if module_level_renderer_calls:
    raise RuntimeError(
        "Response renderers were found outside the active-response "
        f"workflow: {module_level_renderer_calls}"
    )

all_renderer_calls = [
    node.func.id
    for node in ast.walk(application_tree)
    if isinstance(node, ast.Call)
    and isinstance(node.func, ast.Name)
    and node.func.id in response_renderer_names
]

components_tree = ast.parse(
    COMPONENTS_PATH.read_text(
        encoding="utf-8"
    ),
    filename=str(COMPONENTS_PATH),
)

component_function_names = tuple(
    node.name
    for node in components_tree.body
    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        ),
    )
)


# --------------------------------------------------------------------------------------------------
# Formal implementation summary
# --------------------------------------------------------------------------------------------------

print("GROUNDED-LANGUAGE OUTPUT RENDERING")
print("-" * 100)
print(f"Components module             : {COMPONENTS_PATH}")
print(f"Streamlit application         : {STREAMLIT_APP_PATH}")
print(f"Structured preliminary report : INCLUDED")
print(f"Plain-language explanation    : INCLUDED")
print(f"Educational follow-up         : INCLUDED")
print(f"Grounded question answer      : INCLUDED WHEN RETURNED")
print(f"Guardrail action              : INCLUDED")
print(f"Fallback trigger reasons      : INCLUDED")
print(f"Generated-token count         : INCLUDED")
print(f"Generation latency            : INCLUDED")
print(
    f"Connected response renderers  : "
    f"{', '.join(all_renderer_calls)}"
)
print(
    f"Module-level renderer calls   : "
    f"{len(module_level_renderer_calls)}"
)
print(
    f"Persisted component methods   : "
    f"{', '.join(component_function_names)}"
)
print(f"Component syntax              : PASS")
print(f"Application syntax            : PASS")
print(f"Streamlit started             : NO")
print()
print("READY FOR FOLLOW-UP QUESTION WORKFLOW")


GROUNDED-LANGUAGE OUTPUT RENDERING
----------------------------------------------------------------------------------------------------
Components module             : /home/jovyan/chest-xray-ai-assistant/ui/components.py
Streamlit application         : /home/jovyan/chest-xray-ai-assistant/ui/app.py
Structured preliminary report : INCLUDED
Plain-language explanation    : INCLUDED
Educational follow-up         : INCLUDED
Grounded question answer      : INCLUDED WHEN RETURNED
Guardrail action              : INCLUDED
Fallback trigger reasons      : INCLUDED
Generated-token count         : INCLUDED
Generation latency            : INCLUDED
Connected response renderers  : render_finding_summary, render_visual_evidence, render_language_outputs
Module-level renderer calls   : 0
Persisted component methods   : build_finding_table, render_finding_summary, decode_base64_image, render_visual_evidence, render_language_outputs
Component syntax              : PASS
Application syntax            : PASS

## 14. Grounded Follow-Up Question Workflow

This section enables additional questions for the active prediction through `POST /api/v1/question/answer`. Only the prediction identifier and user question are submitted; the client cannot provide or alter grounding evidence. The existing image prediction is reused, so the workflow does not repeat computer-vision inference or Grad-CAM generation.


In [14]:
import ast
import py_compile
import textwrap


# --------------------------------------------------------------------------------------------------
# Extend the components module with grounded-question response rendering
# --------------------------------------------------------------------------------------------------

current_components_source = COMPONENTS_PATH.read_text(
    encoding="utf-8"
)

question_response_component = textwrap.dedent(
    '''

    def render_grounded_question_response(
        response: dict[str, Any],
    ) -> None:
        """Render a grounded-question response and its guardrail details."""
        st.markdown("#### Grounded Answer")

        question = response.get(
            "question"
        )

        if isinstance(question, str) and question.strip():
            st.markdown("**Question**")
            st.write(question)

        st.markdown("**API-generated answer**")
        st.write(
            response.get(
                "output_text",
                "No answer text was returned.",
            )
        )

        guardrail_action = response.get(
            "guardrail_action",
            "Unavailable",
        )

        trigger_reasons = response.get(
            "trigger_reasons",
            [],
        )

        if guardrail_action == "safe_template_fallback":
            st.info(
                "The deterministic language guardrail replaced the raw "
                "model generation with a grounded safety template."
            )

        metric_columns = st.columns(3)

        metric_columns[0].metric(
            "Generated Tokens",
            response.get(
                "generated_tokens",
                "Unavailable",
            ),
        )

        generation_latency = response.get(
            "generation_latency_ms"
        )

        metric_columns[1].metric(
            "Generation Latency",
            (
                f"{generation_latency:.2f} ms"
                if isinstance(
                    generation_latency,
                    (int, float),
                )
                else "Unavailable"
            ),
        )

        metric_columns[2].metric(
            "Guardrail Action",
            str(guardrail_action),
        )

        with st.expander(
            "Question-response details",
            expanded=False,
        ):
            st.write(
                f"**Task type:** "
                f"`{response.get('task_type', 'Unavailable')}`"
            )
            st.write(
                f"**Request ID:** "
                f"`{response.get('request_id', 'Unavailable')}`"
            )
            st.write(
                f"**Prediction ID:** "
                f"`{response.get('prediction_id', 'Unavailable')}`"
            )
            st.write(
                f"**Request latency:** "
                f"{response.get('latency_ms', 'Unavailable')} ms"
            )

            if (
                isinstance(trigger_reasons, list)
                and trigger_reasons
            ):
                st.write(
                    "**Fallback trigger reasons:**"
                )

                for reason in trigger_reasons:
                    st.write(
                        f"- `{reason}`"
                    )
            else:
                st.write(
                    "**Fallback trigger reasons:** None"
                )

        st.caption(
            "This answer is grounded only in the active prediction's "
            "structured evidence and does not independently inspect "
            "the uploaded image."
        )
    '''
).rstrip()

if (
    "def render_grounded_question_response("
    not in current_components_source
):
    current_components_source = (
        current_components_source.rstrip()
        + "\n"
        + question_response_component
        + "\n"
    )

COMPONENTS_PATH.write_text(
    current_components_source,
    encoding="utf-8",
)

py_compile.compile(
    str(COMPONENTS_PATH),
    doraise=True,
)


# --------------------------------------------------------------------------------------------------
# Update the application component import
# --------------------------------------------------------------------------------------------------

current_app_source = STREAMLIT_APP_PATH.read_text(
    encoding="utf-8"
)

existing_component_import = textwrap.dedent(
    '''
    from components import (
        render_finding_summary,
        render_language_outputs,
        render_visual_evidence,
    )
    '''
).strip()

updated_component_import = textwrap.dedent(
    '''
    from components import (
        render_finding_summary,
        render_grounded_question_response,
        render_language_outputs,
        render_visual_evidence,
    )
    '''
).strip()

if existing_component_import in current_app_source:
    current_app_source = current_app_source.replace(
        existing_component_import,
        updated_component_import,
        1,
    )


# --------------------------------------------------------------------------------------------------
# Add the follow-up workflow inside the active-response block
# --------------------------------------------------------------------------------------------------

language_render_call = (
    "        render_language_outputs(active_response)"
)

follow_up_workflow = textwrap.indent(
    textwrap.dedent(
        '''
        st.divider()
        st.subheader("6. Ask a Grounded Follow-Up Question")

        st.caption(
            "The question is sent with the active prediction ID. "
            "The interface cannot submit its own findings, probabilities, "
            "thresholds, descriptions, or grounding context."
        )

        follow_up_question = st.text_input(
            "Question about the active prediction",
            placeholder=(
                "Example: What does the crossed-threshold result mean?"
            ),
            key="follow_up_question_input",
        )

        submit_follow_up = st.button(
            "Ask Grounded Question",
            use_container_width=True,
            disabled=(
                not st.session_state.active_prediction_id
                or not follow_up_question.strip()
            ),
        )

        if submit_follow_up:
            st.session_state.last_api_error = None

            try:
                with st.spinner(
                    "Generating an answer from the active "
                    "prediction evidence..."
                ):
                    api_client = get_api_client(
                        api_base_url
                    )

                    question_response = (
                        api_client.answer_question(
                            prediction_id=(
                                st.session_state.active_prediction_id
                            ),
                            question=follow_up_question,
                        )
                    )

                st.session_state.question_response = (
                    question_response
                )

            except APIClientError as exc:
                st.session_state.question_response = None
                st.session_state.last_api_error = (
                    exc.to_display_dict()
                )

                render_api_error(
                    st.session_state.last_api_error
                )

            except ValueError as exc:
                st.session_state.question_response = None
                st.session_state.last_api_error = {
                    "error_code": "INVALID_UI_REQUEST",
                    "message": str(exc),
                    "status_code": None,
                    "request_id": None,
                    "details": {},
                }

                render_api_error(
                    st.session_state.last_api_error
                )

            except Exception:
                st.session_state.question_response = None
                st.session_state.last_api_error = {
                    "error_code": "UI_EXECUTION_ERROR",
                    "message": (
                        "The grounded question could not be completed. "
                        "Confirm that the backend is healthy and try again."
                    ),
                    "status_code": None,
                    "request_id": None,
                    "details": {},
                }

                render_api_error(
                    st.session_state.last_api_error
                )

        if st.session_state.question_response:
            render_grounded_question_response(
                st.session_state.question_response
            )
        '''
    ).strip(),
    "        ",
)

if (
    "submit_follow_up = st.button("
    not in current_app_source
):
    if current_app_source.count(
        language_render_call
    ) != 1:
        raise RuntimeError(
            "The grounded-language render location could not be "
            "identified unambiguously."
        )

    current_app_source = current_app_source.replace(
        language_render_call,
        language_render_call
        + "\n\n"
        + follow_up_workflow,
        1,
    )

STREAMLIT_APP_PATH.write_text(
    current_app_source,
    encoding="utf-8",
)

py_compile.compile(
    str(STREAMLIT_APP_PATH),
    doraise=True,
)


# --------------------------------------------------------------------------------------------------
# Verify workflow placement and endpoint usage
# --------------------------------------------------------------------------------------------------

application_tree = ast.parse(
    STREAMLIT_APP_PATH.read_text(
        encoding="utf-8"
    ),
    filename=str(STREAMLIT_APP_PATH),
)

components_tree = ast.parse(
    COMPONENTS_PATH.read_text(
        encoding="utf-8"
    ),
    filename=str(COMPONENTS_PATH),
)

component_function_names = tuple(
    node.name
    for node in components_tree.body
    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        ),
    )
)

application_source = STREAMLIT_APP_PATH.read_text(
    encoding="utf-8"
)

answer_question_call_count = application_source.count(
    "api_client.answer_question("
)

analyze_complete_call_count = application_source.count(
    "api_client.analyze_complete("
)


# --------------------------------------------------------------------------------------------------
# Formal implementation summary
# --------------------------------------------------------------------------------------------------

print("GROUNDED FOLLOW-UP QUESTION WORKFLOW")
print("-" * 100)
print(f"Streamlit application       : {STREAMLIT_APP_PATH}")
print(f"Components module           : {COMPONENTS_PATH}")
print(f"Question endpoint           : POST /api/v1/question/answer")
print(f"Submitted prediction field  : prediction_id")
print(f"Submitted question field    : question")
print(f"Client grounding fields     : NOT EXPOSED")
print(f"Follow-up response state    : INCLUDED")
print(f"Controlled errors           : INCLUDED")
print(f"Guardrail details           : INCLUDED")
print(
    f"Complete-analysis call sites: "
    f"{analyze_complete_call_count}"
)
print(
    f"Question-answer call sites  : "
    f"{answer_question_call_count}"
)
print(f"Repeated image inference    : NOT INTRODUCED")
print(
    f"Question renderer available : "
    f"{'render_grounded_question_response' in component_function_names}"
)
print(f"Component syntax            : PASS")
print(f"Application syntax          : PASS")
print(f"Streamlit started           : NO")
print()
print("READY FOR STORED-PREDICTION RETRIEVAL")


GROUNDED FOLLOW-UP QUESTION WORKFLOW
----------------------------------------------------------------------------------------------------
Streamlit application       : /home/jovyan/chest-xray-ai-assistant/ui/app.py
Components module           : /home/jovyan/chest-xray-ai-assistant/ui/components.py
Question endpoint           : POST /api/v1/question/answer
Submitted prediction field  : prediction_id
Submitted question field    : question
Client grounding fields     : NOT EXPOSED
Follow-up response state    : INCLUDED
Controlled errors           : INCLUDED
Guardrail details           : INCLUDED
Complete-analysis call sites: 1
Question-answer call sites  : 1
Repeated image inference    : NOT INTRODUCED
Question renderer available : True
Component syntax            : PASS
Application syntax          : PASS
Streamlit started           : NO

READY FOR STORED-PREDICTION RETRIEVAL


## 15. Stored-Prediction Retrieval

This section adds retrieval of the active prediction through `GET /api/v1/predictions/{prediction_id}`. The workflow verifies that the in-memory backend record remains available and summarizes its stored findings, visual evidence, and language outputs without repeating image inference, Grad-CAM generation, or language generation.


In [15]:
import ast
import py_compile
import textwrap


# --------------------------------------------------------------------------------------------------
# Ensure the stored-prediction renderer exists
# --------------------------------------------------------------------------------------------------

current_components_source = COMPONENTS_PATH.read_text(
    encoding="utf-8"
)

stored_prediction_component = textwrap.dedent(
    '''

    def render_stored_prediction_summary(
        response: dict[str, Any],
    ) -> None:
        """Render a compact summary of a retrieved prediction record."""
        st.markdown("#### Retrieved Prediction Record")

        findings = response.get("findings", [])
        visual_evidence = response.get(
            "visual_evidence",
            [],
        )
        language_outputs = response.get(
            "language_outputs",
            [],
        )
        crossed_finding_names = response.get(
            "crossed_finding_names",
            [],
        )

        summary_columns = st.columns(4)

        summary_columns[0].metric(
            "Stored Findings",
            len(findings)
            if isinstance(findings, list)
            else 0,
        )

        summary_columns[1].metric(
            "Thresholds Crossed",
            len(crossed_finding_names)
            if isinstance(crossed_finding_names, list)
            else 0,
        )

        summary_columns[2].metric(
            "Visual Evidence",
            len(visual_evidence)
            if isinstance(visual_evidence, list)
            else 0,
        )

        summary_columns[3].metric(
            "Language Outputs",
            len(language_outputs)
            if isinstance(language_outputs, list)
            else 0,
        )

        with st.expander(
            "Stored-record identifiers",
            expanded=False,
        ):
            st.write(
                f"**Prediction ID:** "
                f"`{response.get('prediction_id', 'Unavailable')}`"
            )
            st.write(
                f"**Record created:** "
                f"{response.get('created_at_utc', 'Unavailable')}"
            )
            st.write(
                f"**Retrieval request ID:** "
                f"`{response.get('request_id', 'Unavailable')}`"
            )
            st.write(
                f"**Retrieval latency:** "
                f"{response.get('latency_ms', 'Unavailable')} ms"
            )
            st.write(
                f"**No target finding:** "
                f"{response.get('no_target_finding', 'Unavailable')}"
            )

        st.caption(
            "This view summarizes the existing backend record. "
            "Retrieval does not repeat model inference or generation."
        )
    '''
).rstrip()

if (
    "def render_stored_prediction_summary("
    not in current_components_source
):
    current_components_source = (
        current_components_source.rstrip()
        + "\n"
        + stored_prediction_component
        + "\n"
    )

COMPONENTS_PATH.write_text(
    current_components_source,
    encoding="utf-8",
)

py_compile.compile(
    str(COMPONENTS_PATH),
    doraise=True,
)


# --------------------------------------------------------------------------------------------------
# Rebuild app.py with explicit workflow functions
# --------------------------------------------------------------------------------------------------

streamlit_app_source = textwrap.dedent(
    '''
    from __future__ import annotations

    import hashlib
    import os
    from pathlib import Path
    from typing import Any

    import streamlit as st
    import yaml

    from api_client import (
        APIClientError,
        ChestXRayAPIClient,
    )
    from components import (
        render_finding_summary,
        render_grounded_question_response,
        render_language_outputs,
        render_stored_prediction_summary,
        render_visual_evidence,
    )


    SOLUTION_ROOT = Path(__file__).resolve().parents[1]
    UI_CONFIG_PATH = SOLUTION_ROOT / "configs" / "ui_config.yaml"


    def load_ui_configuration(
        config_path: Path,
    ) -> dict[str, Any]:
        """Load the persisted UI configuration."""
        if not config_path.is_file():
            raise FileNotFoundError(
                f"UI configuration was not found: {config_path}"
            )

        with config_path.open("r", encoding="utf-8") as file:
            configuration = yaml.safe_load(file) or {}

        if not isinstance(configuration, dict):
            raise TypeError(
                "The UI configuration must contain a YAML mapping."
            )

        return configuration


    def format_file_size(
        size_bytes: int,
    ) -> str:
        """Format a file size using binary units."""
        return f"{size_bytes / (1024 ** 2):.2f} MiB"


    def initialize_session_state() -> None:
        """Initialize all UI-local workflow state."""
        default_state = {
            "upload_signature": None,
            "uploaded_filename": None,
            "uploaded_media_type": None,
            "uploaded_image_bytes": None,
            "upload_ready": False,
            "analysis_response": None,
            "active_prediction_id": None,
            "question_response": None,
            "stored_prediction_response": None,
            "last_api_error": None,
        }

        for key, default_value in default_state.items():
            if key not in st.session_state:
                st.session_state[key] = default_value


    def reset_analysis_state() -> None:
        """Clear outputs when the selected image changes."""
        st.session_state.analysis_response = None
        st.session_state.active_prediction_id = None
        st.session_state.question_response = None
        st.session_state.stored_prediction_response = None
        st.session_state.last_api_error = None


    @st.cache_resource(show_spinner=False)
    def get_api_client(
        base_url: str,
    ) -> ChestXRayAPIClient:
        """Create one reusable HTTP client."""
        return ChestXRayAPIClient(
            base_url=base_url,
            timeout_seconds=120.0,
        )


    def render_api_error(
        error_details: dict[str, Any],
    ) -> None:
        """Render only controlled, UI-safe error information."""
        st.error(
            error_details.get(
                "message",
                "The request could not be completed.",
            )
        )

        with st.expander(
            "Technical error details",
            expanded=False,
        ):
            st.write(
                f"**Error code:** "
                f"{error_details.get('error_code', 'UNKNOWN')}"
            )

            status_code = error_details.get("status_code")

            if status_code is not None:
                st.write(
                    f"**HTTP status:** {status_code}"
                )

            request_id = error_details.get("request_id")

            if request_id:
                st.write(
                    f"**Request ID:** `{request_id}`"
                )

            safe_details = error_details.get("details")

            if isinstance(safe_details, dict) and safe_details:
                st.json(safe_details)


    def render_sidebar(
        api_base_url: str,
        upload_config: dict[str, Any],
        safety_config: dict[str, Any],
    ) -> None:
        """Render system-boundary information."""
        with st.sidebar:
            st.header("System Boundary")

            st.info(
                "The interface communicates with the FastAPI backend "
                "through HTTP. It does not load the models, Grad-CAM "
                "service, or prediction store directly."
            )

            st.subheader("Backend Address")
            st.code(
                api_base_url,
                language=None,
            )

            st.subheader("Accepted Uploads")
            st.caption(
                "PNG or JPEG • Maximum "
                f"{upload_config['maximum_size_mib']} MiB"
            )

            st.subheader("Professional Review")
            st.caption(
                safety_config[
                    "professional_review_guidance"
                ]
            )


    def render_upload(
        upload_config: dict[str, Any],
    ) -> None:
        """Render image selection, size awareness, and preview."""
        st.divider()
        st.subheader("1. Select a Chest X-Ray Image")

        uploaded_file = st.file_uploader(
            "Choose a PNG or JPEG image",
            type=upload_config[
                "supported_extensions"
            ],
            accept_multiple_files=False,
            help=(
                "The backend accepts image/png and image/jpeg files "
                f"up to {upload_config['maximum_size_mib']} MiB."
            ),
        )

        if uploaded_file is None:
            if st.session_state.upload_signature is not None:
                reset_analysis_state()

            st.session_state.upload_signature = None
            st.session_state.uploaded_filename = None
            st.session_state.uploaded_media_type = None
            st.session_state.uploaded_image_bytes = None
            st.session_state.upload_ready = False

            st.info(
                "Select an image to enable API submission."
            )
            return

        uploaded_image_bytes = uploaded_file.getvalue()
        uploaded_size_bytes = len(
            uploaded_image_bytes
        )
        uploaded_media_type = uploaded_file.type or ""

        upload_signature = hashlib.sha256(
            uploaded_image_bytes
        ).hexdigest()

        if (
            st.session_state.upload_signature
            != upload_signature
        ):
            reset_analysis_state()

        st.session_state.upload_signature = (
            upload_signature
        )
        st.session_state.uploaded_filename = (
            uploaded_file.name
        )
        st.session_state.uploaded_media_type = (
            uploaded_media_type
        )
        st.session_state.uploaded_image_bytes = (
            uploaded_image_bytes
        )

        size_within_limit = (
            uploaded_size_bytes
            <= upload_config["maximum_size_bytes"]
        )

        media_type_supported = (
            uploaded_media_type
            in upload_config["supported_media_types"]
        )

        st.session_state.upload_ready = (
            size_within_limit
            and media_type_supported
        )

        preview_column, details_column = st.columns(
            [1.4, 1.0],
            gap="large",
        )

        with preview_column:
            st.image(
                uploaded_image_bytes,
                caption=uploaded_file.name,
                use_container_width=True,
            )

        with details_column:
            st.markdown("#### Selected Image")
            st.write(
                f"**Filename:** {uploaded_file.name}"
            )
            st.write(
                f"**Media type:** "
                f"{uploaded_media_type or 'Not reported'}"
            )
            st.write(
                f"**File size:** "
                f"{format_file_size(uploaded_size_bytes)}"
            )
            st.write(
                f"**Backend size limit:** "
                f"{upload_config['maximum_size_mib']} MiB"
            )

            if not size_within_limit:
                st.error(
                    "The selected file exceeds the backend's "
                    f"{upload_config['maximum_size_mib']} MiB limit."
                )
            elif not media_type_supported:
                st.error(
                    "The selected file does not report a supported "
                    "PNG or JPEG media type."
                )
            else:
                st.success(
                    "The file is ready for secure backend "
                    "validation and analysis."
                )

        st.caption(
            "The preview and size check do not validate medical "
            "content. Image validation remains authoritative in "
            "the FastAPI backend."
        )


    def render_analysis_submission(
        api_base_url: str,
    ) -> None:
        """Submit the selected image only on explicit user action."""
        st.divider()
        st.subheader("2. Run Complete Analysis")

        initial_question = st.text_area(
            "Optional grounded question",
            placeholder=(
                "Example: Which findings crossed their "
                "frozen thresholds?"
            ),
            help=(
                "The question is answered only from the structured "
                "model evidence and safety limitations."
            ),
            disabled=not st.session_state.upload_ready,
        )

        submit_analysis = st.button(
            "Run Complete Analysis",
            type="primary",
            use_container_width=True,
            disabled=not st.session_state.upload_ready,
        )

        if not submit_analysis:
            return

        st.session_state.last_api_error = None
        st.session_state.question_response = None
        st.session_state.stored_prediction_response = None

        try:
            with st.spinner(
                "Submitting the image to the FastAPI "
                "analysis workflow..."
            ):
                api_client = get_api_client(
                    api_base_url
                )

                response = api_client.analyze_complete(
                    filename=st.session_state.uploaded_filename,
                    media_type=st.session_state.uploaded_media_type,
                    image_content=st.session_state.uploaded_image_bytes,
                    question=(
                        initial_question
                        if initial_question.strip()
                        else None
                    ),
                )

            st.session_state.analysis_response = response
            st.session_state.active_prediction_id = (
                response.get("prediction_id")
            )

        except APIClientError as exc:
            st.session_state.analysis_response = None
            st.session_state.active_prediction_id = None
            st.session_state.last_api_error = (
                exc.to_display_dict()
            )

        except ValueError as exc:
            st.session_state.analysis_response = None
            st.session_state.active_prediction_id = None
            st.session_state.last_api_error = {
                "error_code": "INVALID_UI_REQUEST",
                "message": str(exc),
                "status_code": None,
                "request_id": None,
                "details": {},
            }

        except Exception:
            st.session_state.analysis_response = None
            st.session_state.active_prediction_id = None
            st.session_state.last_api_error = {
                "error_code": "UI_EXECUTION_ERROR",
                "message": (
                    "The analysis request could not be completed. "
                    "Confirm that the backend is healthy and try again."
                ),
                "status_code": None,
                "request_id": None,
                "details": {},
            }


    def render_response_metadata(
        response: dict[str, Any],
    ) -> None:
        """Render identifiers, latency, and model lineage."""
        st.success(
            "The complete-analysis response was received and "
            "preserved in this interface session."
        )

        metadata_columns = st.columns(4)

        metadata_columns[0].metric(
            "Status",
            str(
                response.get(
                    "status",
                    "success",
                )
            ),
        )

        latency = response.get("latency_ms")

        metadata_columns[1].metric(
            "Request Latency",
            (
                f"{latency:.2f} ms"
                if isinstance(latency, (int, float))
                else "Unavailable"
            ),
        )

        metadata_columns[2].metric(
            "Prediction ID",
            "Available"
            if response.get("prediction_id")
            else "Unavailable",
        )

        language_outputs = response.get(
            "language_outputs",
            [],
        )

        metadata_columns[3].metric(
            "Language Outputs",
            len(language_outputs)
            if isinstance(language_outputs, list)
            else 0,
        )

        with st.expander(
            "Request and lineage identifiers",
            expanded=False,
        ):
            st.write(
                f"**Request ID:** "
                f"`{response.get('request_id', 'Unavailable')}`"
            )
            st.write(
                f"**Prediction ID:** "
                f"`{response.get('prediction_id', 'Unavailable')}`"
            )
            st.write(
                f"**API version:** "
                f"{response.get('api_version', 'Unavailable')}"
            )
            st.write(
                f"**Prompt-registry version:** "
                f"{response.get('prompt_registry_version', 'Unavailable')}"
            )
            st.write("**Model versions:**")
            st.json(
                response.get(
                    "model_versions",
                    {},
                )
            )


    def render_follow_up_workflow(
        api_base_url: str,
    ) -> None:
        """Submit a grounded question for the active prediction."""
        st.divider()
        st.subheader("6. Ask a Grounded Follow-Up Question")

        st.caption(
            "Only the active prediction ID and question are sent. "
            "The interface cannot submit grounding evidence."
        )

        question = st.text_input(
            "Question about the active prediction",
            placeholder=(
                "Example: What does the crossed-threshold "
                "result mean?"
            ),
            key="follow_up_question_input",
        )

        submit_question = st.button(
            "Ask Grounded Question",
            use_container_width=True,
            disabled=(
                not st.session_state.active_prediction_id
                or not question.strip()
            ),
        )

        if submit_question:
            try:
                with st.spinner(
                    "Generating an answer from the active "
                    "prediction evidence..."
                ):
                    api_client = get_api_client(
                        api_base_url
                    )

                    response = api_client.answer_question(
                        prediction_id=(
                            st.session_state.active_prediction_id
                        ),
                        question=question,
                    )

                st.session_state.question_response = response

            except APIClientError as exc:
                st.session_state.question_response = None
                render_api_error(
                    exc.to_display_dict()
                )

            except Exception:
                st.session_state.question_response = None
                render_api_error(
                    {
                        "error_code": "UI_EXECUTION_ERROR",
                        "message": (
                            "The grounded question could not be "
                            "completed. Try again after confirming "
                            "backend health."
                        ),
                        "status_code": None,
                        "request_id": None,
                        "details": {},
                    }
                )

        if st.session_state.question_response:
            render_grounded_question_response(
                st.session_state.question_response
            )


    def render_retrieval_workflow(
        api_base_url: str,
    ) -> None:
        """Retrieve the active stored prediction without rerunning models."""
        st.divider()
        st.subheader("7. Retrieve the Stored Prediction")

        st.caption(
            "Retrieve the active record from the backend's in-memory "
            "prediction store without repeating analysis."
        )

        retrieve_prediction = st.button(
            "Retrieve Active Prediction",
            use_container_width=True,
            disabled=not st.session_state.active_prediction_id,
        )

        if retrieve_prediction:
            try:
                with st.spinner(
                    "Retrieving the active prediction record..."
                ):
                    api_client = get_api_client(
                        api_base_url
                    )

                    response = api_client.get_prediction(
                        st.session_state.active_prediction_id
                    )

                st.session_state.stored_prediction_response = (
                    response
                )

                st.success(
                    "The prediction record was retrieved without "
                    "repeating model execution."
                )

            except APIClientError as exc:
                st.session_state.stored_prediction_response = None
                render_api_error(
                    exc.to_display_dict()
                )

            except Exception:
                st.session_state.stored_prediction_response = None
                render_api_error(
                    {
                        "error_code": "UI_EXECUTION_ERROR",
                        "message": (
                            "The stored prediction could not be "
                            "retrieved. It may have expired or the "
                            "backend may be unavailable."
                        ),
                        "status_code": None,
                        "request_id": None,
                        "details": {},
                    }
                )

        stored_response = (
            st.session_state.stored_prediction_response
        )

        if (
            isinstance(stored_response, dict)
            and stored_response.get("prediction_id")
            == st.session_state.active_prediction_id
        ):
            render_stored_prediction_summary(
                stored_response
            )


    def main() -> None:
        """Run the Streamlit interface."""
        ui_config = load_ui_configuration(
            UI_CONFIG_PATH
        )

        page_config = ui_config["page"]
        api_config = ui_config["api"]
        safety_config = ui_config["safety"]
        upload_config = ui_config["upload"]

        api_base_url = os.getenv(
            api_config[
                "base_url_environment_variable"
            ],
            api_config["default_base_url"],
        ).strip().rstrip("/")

        st.set_page_config(
            page_title=page_config["title"],
            page_icon=page_config["icon"],
            layout=page_config["layout"],
            initial_sidebar_state=(
                page_config[
                    "initial_sidebar_state"
                ]
            ),
        )

        initialize_session_state()

        st.title(page_config["title"])
        st.caption(page_config["subtitle"])

        st.warning(
            safety_config[
                "educational_limitation"
            ],
            icon="⚠️",
        )

        st.markdown(
            """
            Upload a supported chest X-ray image to request a complete,
            API-driven analysis. Findings are determined only by the
            frozen computer-vision model and persisted thresholds.
            Language outputs use only the resulting structured evidence.
            """
        )

        render_sidebar(
            api_base_url,
            upload_config,
            safety_config,
        )

        render_upload(
            upload_config
        )

        render_analysis_submission(
            api_base_url
        )

        if st.session_state.last_api_error:
            render_api_error(
                st.session_state.last_api_error
            )

        active_response = (
            st.session_state.analysis_response
        )

        if not isinstance(active_response, dict):
            return

        render_response_metadata(
            active_response
        )

        render_finding_summary(
            active_response
        )

        render_visual_evidence(
            active_response,
            safety_config[
                "gradcam_limitation"
            ],
        )

        render_language_outputs(
            active_response
        )

        render_follow_up_workflow(
            api_base_url
        )

        render_retrieval_workflow(
            api_base_url
        )


    if __name__ == "__main__":
        main()
    '''
).strip() + "\n"

STREAMLIT_APP_PATH.write_text(
    streamlit_app_source,
    encoding="utf-8",
)


# --------------------------------------------------------------------------------------------------
# Validate both persisted modules
# --------------------------------------------------------------------------------------------------

py_compile.compile(
    str(COMPONENTS_PATH),
    doraise=True,
)

py_compile.compile(
    str(STREAMLIT_APP_PATH),
    doraise=True,
)

application_tree = ast.parse(
    STREAMLIT_APP_PATH.read_text(
        encoding="utf-8"
    ),
    filename=str(STREAMLIT_APP_PATH),
)

application_function_names = tuple(
    node.name
    for node in application_tree.body
    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        ),
    )
)

application_source = STREAMLIT_APP_PATH.read_text(
    encoding="utf-8"
)

analyze_complete_call_count = (
    application_source.count(
        "api_client.analyze_complete("
    )
)

answer_question_call_count = (
    application_source.count(
        "api_client.answer_question("
    )
)

get_prediction_call_count = (
    application_source.count(
        "api_client.get_prediction("
    )
)


# --------------------------------------------------------------------------------------------------
# Formal implementation summary
# --------------------------------------------------------------------------------------------------

print("STORED-PREDICTION RETRIEVAL")
print("-" * 100)
print(f"Streamlit application       : {STREAMLIT_APP_PATH}")
print(f"Components module           : {COMPONENTS_PATH}")
print(
    "Retrieval endpoint          : "
    "GET /api/v1/predictions/{prediction_id}"
)
print(f"Active prediction ID reused : YES")
print(f"Stored response state       : INCLUDED")
print(f"Stored finding count        : INCLUDED")
print(f"Stored visual-evidence count: INCLUDED")
print(f"Stored language-output count: INCLUDED")
print(
    f"Complete-analysis call sites: "
    f"{analyze_complete_call_count}"
)
print(
    f"Question-answer call sites  : "
    f"{answer_question_call_count}"
)
print(
    f"Prediction retrieval calls  : "
    f"{get_prediction_call_count}"
)
print(f"Repeated model execution    : NOT INTRODUCED")
print(
    f"Application functions       : "
    f"{', '.join(application_function_names)}"
)
print(f"Component syntax            : PASS")
print(f"Application syntax          : PASS")
print(f"Streamlit started           : NO")
print()
print("READY FOR OPERATIONAL AND LLMOPS METRICS DISPLAY")


STORED-PREDICTION RETRIEVAL
----------------------------------------------------------------------------------------------------
Streamlit application       : /home/jovyan/chest-xray-ai-assistant/ui/app.py
Components module           : /home/jovyan/chest-xray-ai-assistant/ui/components.py
Retrieval endpoint          : GET /api/v1/predictions/{prediction_id}
Active prediction ID reused : YES
Stored response state       : INCLUDED
Stored finding count        : INCLUDED
Stored visual-evidence count: INCLUDED
Stored language-output count: INCLUDED
Complete-analysis call sites: 1
Question-answer call sites  : 1
Prediction retrieval calls  : 1
Repeated model execution    : NOT INTRODUCED
Application functions       : load_ui_configuration, format_file_size, initialize_session_state, reset_analysis_state, get_api_client, render_api_error, render_sidebar, render_upload, render_analysis_submission, render_response_metadata, render_follow_up_workflow, render_retrieval_workflow, main
Component sy

## 16. Operational and LLMOps Metrics Display

This section adds an on-demand sidebar panel for `GET /api/v1/llmops/metrics`. It displays request totals, successful and failed requests, language-generation activity, guardrail actions, endpoint request counts, and endpoint latency summaries. Metrics are refreshed only when requested and do not trigger model inference.


In [16]:
import ast
import py_compile
import textwrap
from typing import Any


# --------------------------------------------------------------------------------------------------
# Load and inspect the current structured application source
# --------------------------------------------------------------------------------------------------

current_app_source = STREAMLIT_APP_PATH.read_text(
    encoding="utf-8"
)

application_tree = ast.parse(
    current_app_source,
    filename=str(STREAMLIT_APP_PATH),
)


def find_top_level_function(
    syntax_tree: ast.Module,
    function_name: str,
) -> ast.FunctionDef | ast.AsyncFunctionDef | None:
    """Find one top-level function by its persisted name."""
    for node in syntax_tree.body:
        if (
            isinstance(
                node,
                (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                ),
            )
            and node.name == function_name
        ):
            return node

    return None


# --------------------------------------------------------------------------------------------------
# Insert the metrics workflow before main() using AST line locations
# --------------------------------------------------------------------------------------------------

metrics_function_name = (
    "render_operational_metrics_workflow"
)

metrics_function_source = textwrap.dedent(
    '''
    def render_operational_metrics_workflow(
        api_base_url: str,
    ) -> None:
        """Retrieve and render operational metrics on explicit request."""
        if "operational_metrics_response" not in st.session_state:
            st.session_state.operational_metrics_response = None

        with st.sidebar:
            st.divider()
            st.subheader("Operational and LLMOps Metrics")

            refresh_metrics = st.button(
                "Refresh Operational Metrics",
                use_container_width=True,
            )

            if refresh_metrics:
                try:
                    with st.spinner(
                        "Retrieving operational metrics..."
                    ):
                        api_client = get_api_client(
                            api_base_url
                        )

                        metrics_response = (
                            api_client.llmops_metrics()
                        )

                    st.session_state.operational_metrics_response = (
                        metrics_response
                    )

                except APIClientError as exc:
                    st.session_state.operational_metrics_response = None
                    render_api_error(
                        exc.to_display_dict()
                    )

                except Exception:
                    st.session_state.operational_metrics_response = None
                    render_api_error(
                        {
                            "error_code": "UI_EXECUTION_ERROR",
                            "message": (
                                "Operational metrics could not be "
                                "retrieved. Confirm backend health "
                                "and try again."
                            ),
                            "status_code": None,
                            "request_id": None,
                            "details": {},
                        }
                    )

            metrics_response = (
                st.session_state.operational_metrics_response
            )

            if not isinstance(metrics_response, dict):
                st.caption(
                    "Select refresh to retrieve the current "
                    "backend operational snapshot."
                )
                return

            total_requests = metrics_response.get(
                "total_requests",
                0,
            )

            successful_requests = metrics_response.get(
                "successful_requests",
                0,
            )

            failed_requests = metrics_response.get(
                "failed_requests",
                0,
            )

            language_requests = metrics_response.get(
                "language_generation_requests",
                0,
            )

            st.metric(
                "Total Requests",
                total_requests,
            )

            success_column, failure_column = st.columns(2)

            success_column.metric(
                "Successful",
                successful_requests,
            )

            failure_column.metric(
                "Failed",
                failed_requests,
            )

            st.metric(
                "Language Generations",
                language_requests,
            )

            guardrail_actions = metrics_response.get(
                "guardrail_action_counts",
                {},
            )

            endpoint_counts = metrics_response.get(
                "endpoint_request_counts",
                {},
            )

            endpoint_latency = metrics_response.get(
                "endpoint_average_latency_ms",
                {},
            )

            with st.expander(
                "Guardrail actions",
                expanded=False,
            ):
                if (
                    isinstance(guardrail_actions, dict)
                    and guardrail_actions
                ):
                    st.json(guardrail_actions)
                else:
                    st.caption(
                        "No guardrail actions have been recorded."
                    )

            with st.expander(
                "Endpoint request counts",
                expanded=False,
            ):
                if (
                    isinstance(endpoint_counts, dict)
                    and endpoint_counts
                ):
                    st.json(endpoint_counts)
                else:
                    st.caption(
                        "No endpoint request counts are available."
                    )

            with st.expander(
                "Average endpoint latency",
                expanded=False,
            ):
                if (
                    isinstance(endpoint_latency, dict)
                    and endpoint_latency
                ):
                    st.json(endpoint_latency)
                else:
                    st.caption(
                        "No endpoint latency records are available."
                    )

            st.caption(
                "Service started: "
                f"{metrics_response.get('service_started_at_utc', 'Unavailable')}"
            )

            st.caption(
                "Metrics request ID: "
                f"{metrics_response.get('request_id', 'Unavailable')}"
            )
    '''
).strip() + "\n\n"

if (
    find_top_level_function(
        application_tree,
        metrics_function_name,
    )
    is None
):
    main_function = find_top_level_function(
        application_tree,
        "main",
    )

    if main_function is None:
        raise RuntimeError(
            "The persisted main function is unavailable."
        )

    application_lines = current_app_source.splitlines(
        keepends=True
    )

    insertion_index = main_function.lineno - 1

    application_lines.insert(
        insertion_index,
        metrics_function_source,
    )

    current_app_source = "".join(
        application_lines
    )


# --------------------------------------------------------------------------------------------------
# Insert the metrics call in main() before active-response rendering
# --------------------------------------------------------------------------------------------------

application_tree = ast.parse(
    current_app_source,
    filename=str(STREAMLIT_APP_PATH),
)

main_function = find_top_level_function(
    application_tree,
    "main",
)

if main_function is None:
    raise RuntimeError(
        "The persisted main function is unavailable."
    )

metrics_call_exists = any(
    isinstance(node, ast.Call)
    and isinstance(node.func, ast.Name)
    and node.func.id == metrics_function_name
    for node in ast.walk(main_function)
)

if not metrics_call_exists:
    active_response_assignment = None

    for node in main_function.body:
        if not isinstance(node, ast.Assign):
            continue

        assigned_names = [
            target.id
            for target in node.targets
            if isinstance(target, ast.Name)
        ]

        if "active_response" in assigned_names:
            active_response_assignment = node
            break

    if active_response_assignment is None:
        raise RuntimeError(
            "The active-response assignment is unavailable in main()."
        )

    metrics_call_source = textwrap.indent(
        textwrap.dedent(
            '''
            render_operational_metrics_workflow(
                api_base_url
            )

            '''
        ),
        "    ",
    )

    application_lines = current_app_source.splitlines(
        keepends=True
    )

    insertion_index = (
        active_response_assignment.lineno - 1
    )

    application_lines.insert(
        insertion_index,
        metrics_call_source,
    )

    current_app_source = "".join(
        application_lines
    )


# --------------------------------------------------------------------------------------------------
# Persist and validate the updated application
# --------------------------------------------------------------------------------------------------

STREAMLIT_APP_PATH.write_text(
    current_app_source,
    encoding="utf-8",
)

py_compile.compile(
    str(STREAMLIT_APP_PATH),
    doraise=True,
)

validated_tree = ast.parse(
    STREAMLIT_APP_PATH.read_text(
        encoding="utf-8"
    ),
    filename=str(STREAMLIT_APP_PATH),
)

validated_main = find_top_level_function(
    validated_tree,
    "main",
)

validated_metrics_function = (
    find_top_level_function(
        validated_tree,
        metrics_function_name,
    )
)

metrics_endpoint_call_count = (
    STREAMLIT_APP_PATH.read_text(
        encoding="utf-8"
    ).count(
        "api_client.llmops_metrics("
    )
)

metrics_workflow_call_count = sum(
    1
    for node in ast.walk(validated_main)
    if isinstance(node, ast.Call)
    and isinstance(node.func, ast.Name)
    and node.func.id == metrics_function_name
)


# --------------------------------------------------------------------------------------------------
# Formal implementation summary
# --------------------------------------------------------------------------------------------------

print("OPERATIONAL AND LLMOPS METRICS DISPLAY")
print("-" * 100)
print(f"Streamlit application       : {STREAMLIT_APP_PATH}")
print(f"Metrics endpoint            : GET /api/v1/llmops/metrics")
print(f"Metrics refresh             : USER INITIATED")
print(f"Total request count         : INCLUDED")
print(f"Successful requests         : INCLUDED")
print(f"Failed requests             : INCLUDED")
print(f"Language generations        : INCLUDED")
print(f"Guardrail action counts     : INCLUDED")
print(f"Endpoint request counts     : INCLUDED")
print(f"Endpoint average latency    : INCLUDED")
print(f"Metrics session state       : INCLUDED")
print(
    f"Metrics endpoint call sites : "
    f"{metrics_endpoint_call_count}"
)
print(
    f"Metrics workflow call sites : "
    f"{metrics_workflow_call_count}"
)
print(
    f"Metrics function available  : "
    f"{validated_metrics_function is not None}"
)
print(f"Model inference introduced  : NO")
print(f"Application syntax          : PASS")
print(f"Streamlit started           : NO")
print()
print("READY FOR UI COMPONENT AND STATE-TRANSITION TESTING")


OPERATIONAL AND LLMOPS METRICS DISPLAY
----------------------------------------------------------------------------------------------------
Streamlit application       : /home/jovyan/chest-xray-ai-assistant/ui/app.py
Metrics endpoint            : GET /api/v1/llmops/metrics
Metrics refresh             : USER INITIATED
Total request count         : INCLUDED
Successful requests         : INCLUDED
Failed requests             : INCLUDED
Language generations        : INCLUDED
Guardrail action counts     : INCLUDED
Endpoint request counts     : INCLUDED
Endpoint average latency    : INCLUDED
Metrics session state       : INCLUDED
Metrics endpoint call sites : 1
Metrics workflow call sites : 1
Metrics function available  : True
Model inference introduced  : NO
Application syntax          : PASS
Streamlit started           : NO

READY FOR UI COMPONENT AND STATE-TRANSITION TESTING


## 17. UI Component and State-Transition Testing

This section validates the Streamlit interface without repeating backend integration tests. It checks initial page rendering, session-state initialization and reset behavior, finding-table construction, base64 evidence decoding, and preservation of the HTTP-only backend boundary.


In [17]:
import ast
import base64
import importlib.util
import json
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

from streamlit.testing.v1 import AppTest


# --------------------------------------------------------------------------------------------------
# Load the persisted UI modules
# --------------------------------------------------------------------------------------------------

ui_root_text = str(UI_ROOT)

if ui_root_text not in sys.path:
    sys.path.insert(0, ui_root_text)


def load_module_from_path(
    module_name: str,
    module_path: Path,
):
    """Load one persisted module from its exact path."""
    module_spec = importlib.util.spec_from_file_location(
        module_name,
        module_path,
    )

    if module_spec is None or module_spec.loader is None:
        raise ImportError(
            f"Unable to load module: {module_path}"
        )

    module = importlib.util.module_from_spec(
        module_spec
    )

    module_spec.loader.exec_module(
        module
    )

    return module


ui_components_module = load_module_from_path(
    "tested_ui_components",
    COMPONENTS_PATH,
)

ui_application_module = load_module_from_path(
    "tested_ui_application",
    STREAMLIT_APP_PATH,
)


# --------------------------------------------------------------------------------------------------
# Isolated session-state replacement
# --------------------------------------------------------------------------------------------------

class FakeSessionState(dict):
    """Dictionary with Streamlit-style attribute access."""

    def __getattr__(
        self,
        name: str,
    ) -> Any:
        try:
            return self[name]
        except KeyError as exc:
            raise AttributeError(name) from exc

    def __setattr__(
        self,
        name: str,
        value: Any,
    ) -> None:
        self[name] = value


class FakeStreamlit:
    """Minimal Streamlit replacement used by state functions."""

    def __init__(self) -> None:
        self.session_state = FakeSessionState()


# --------------------------------------------------------------------------------------------------
# Focused UI tests
# --------------------------------------------------------------------------------------------------

def test_initial_streamlit_render() -> dict[str, Any]:
    app_test = AppTest.from_file(
        str(STREAMLIT_APP_PATH)
    )

    app_test.run(
        timeout=30,
    )

    rendered_exceptions = list(
        app_test.exception
    )

    if rendered_exceptions:
        raise AssertionError(
            "The initial Streamlit page raised exceptions: "
            f"{[str(item.value) for item in rendered_exceptions]}"
        )

    rendered_titles = [
        item.value
        for item in app_test.title
    ]

    rendered_warnings = [
        item.value
        for item in app_test.warning
    ]

    if (
        "Chest X-Ray Analysis and Explanation Assistant"
        not in rendered_titles
    ):
        raise AssertionError(
            "The configured application title was not rendered."
        )

    if EDUCATIONAL_LIMITATION not in rendered_warnings:
        raise AssertionError(
            "The educational limitation was not rendered."
        )

    return {
        "rendered_titles": len(rendered_titles),
        "rendered_warnings": len(rendered_warnings),
        "exceptions": 0,
    }


def test_session_state_transitions() -> dict[str, Any]:
    fake_streamlit = FakeStreamlit()

    original_streamlit_reference = (
        ui_application_module.st
    )

    ui_application_module.st = fake_streamlit

    try:
        ui_application_module.initialize_session_state()

        expected_state_keys = {
            "upload_signature",
            "uploaded_filename",
            "uploaded_media_type",
            "uploaded_image_bytes",
            "upload_ready",
            "analysis_response",
            "active_prediction_id",
            "question_response",
            "stored_prediction_response",
            "last_api_error",
        }

        initialized_state_keys = set(
            fake_streamlit.session_state
        )

        if initialized_state_keys != expected_state_keys:
            raise AssertionError(
                "The initialized state keys differ from the "
                "persisted UI contract."
            )

        fake_streamlit.session_state.analysis_response = {
            "prediction_id": "test-prediction"
        }
        fake_streamlit.session_state.active_prediction_id = (
            "test-prediction"
        )
        fake_streamlit.session_state.question_response = {
            "output_text": "test"
        }
        fake_streamlit.session_state.stored_prediction_response = {
            "prediction_id": "test-prediction"
        }
        fake_streamlit.session_state.last_api_error = {
            "error_code": "TEST"
        }

        ui_application_module.reset_analysis_state()

        reset_keys = (
            "analysis_response",
            "active_prediction_id",
            "question_response",
            "stored_prediction_response",
            "last_api_error",
        )

        uncleared_keys = [
            key
            for key in reset_keys
            if fake_streamlit.session_state[key]
            is not None
        ]

        if uncleared_keys:
            raise AssertionError(
                "State reset did not clear: "
                f"{uncleared_keys}"
            )

        return {
            "initialized_keys": len(
                expected_state_keys
            ),
            "reset_values": len(
                reset_keys
            ),
        }

    finally:
        ui_application_module.st = (
            original_streamlit_reference
        )


def test_finding_table_construction() -> dict[str, Any]:
    sample_findings = [
        {
            "label_id": 1,
            "label_name": "cardiomegaly",
            "display_name": "Cardiomegaly",
            "probability": 0.25,
            "frozen_threshold": 0.8438,
            "crossed_threshold": False,
            "confidence_category": "below_threshold",
            "approved_description": "Test description.",
        },
        {
            "label_id": 0,
            "label_name": "atelectasis",
            "display_name": "Atelectasis",
            "probability": 0.75,
            "frozen_threshold": 0.6797,
            "crossed_threshold": True,
            "confidence_category": "above_threshold",
            "approved_description": "Test description.",
        },
    ]

    finding_table = (
        ui_components_module.build_finding_table(
            sample_findings
        )
    )

    expected_columns = [
        "Label ID",
        "Finding",
        "Probability",
        "Frozen Threshold",
        "Crossed Threshold",
        "Confidence",
    ]

    if list(finding_table.columns) != expected_columns:
        raise AssertionError(
            "The finding-table columns differ from the "
            "authoritative UI mapping."
        )

    label_order = finding_table[
        "Label ID"
    ].tolist()

    if label_order != [0, 1]:
        raise AssertionError(
            "Finding rows were not ordered by label ID."
        )

    crossed_decision = finding_table.iloc[0][
        "Crossed Threshold"
    ]

    if not bool(crossed_decision):
        raise AssertionError(
            "The crossed-threshold decision was not preserved."
        )

    second_decision = finding_table.iloc[1][
        "Crossed Threshold"
    ]

    if bool(second_decision):
        raise AssertionError(
            "The below-threshold decision was not preserved."
        )

    return {
        "rows": len(finding_table),
        "columns": len(finding_table.columns),
        "label_order": label_order,
        "crossed_decisions": [
            bool(crossed_decision),
            bool(second_decision),
        ],
    }


def test_base64_evidence_decoding() -> dict[str, Any]:
    expected_bytes = (
        b"persisted-gradcam-png-evidence"
    )

    encoded_value = base64.b64encode(
        expected_bytes
    ).decode("ascii")

    decoded_value = (
        ui_components_module.decode_base64_image(
            encoded_value
        )
    )

    if decoded_value != expected_bytes:
        raise AssertionError(
            "Valid base64 evidence was not preserved."
        )

    malformed_result = (
        ui_components_module.decode_base64_image(
            "%%%invalid-base64%%%"
        )
    )

    if malformed_result is not None:
        raise AssertionError(
            "Malformed base64 evidence was not rejected."
        )

    return {
        "valid_round_trip": True,
        "malformed_rejected": True,
    }


def test_http_only_application_boundary() -> dict[str, Any]:
    application_source = STREAMLIT_APP_PATH.read_text(
        encoding="utf-8"
    )

    syntax_tree = ast.parse(
        application_source,
        filename=str(STREAMLIT_APP_PATH),
    )

    imported_roots = set()

    for node in ast.walk(syntax_tree):
        if isinstance(node, ast.Import):
            imported_roots.update(
                alias.name.split(".", maxsplit=1)[0]
                for alias in node.names
            )

        elif (
            isinstance(node, ast.ImportFrom)
            and node.module
        ):
            imported_roots.add(
                node.module.split(".", maxsplit=1)[0]
            )

    prohibited_model_imports = {
        "torch",
        "torchvision",
        "transformers",
        "captum",
        "src",
        "api",
    }

    detected_prohibited_imports = sorted(
        imported_roots
        & prohibited_model_imports
    )

    if detected_prohibited_imports:
        raise AssertionError(
            "The UI imports prohibited model or backend packages: "
            f"{detected_prohibited_imports}"
        )

    expected_client_calls = {
        "analyze_complete": 1,
        "answer_question": 1,
        "get_prediction": 1,
        "llmops_metrics": 1,
    }

    observed_client_calls = {
        method_name: application_source.count(
            f"api_client.{method_name}("
        )
        for method_name in expected_client_calls
    }

    if observed_client_calls != expected_client_calls:
        raise AssertionError(
            "The UI HTTP call sites differ from the expected "
            f"boundary: {observed_client_calls}"
        )

    return {
        "prohibited_imports": [],
        "http_call_sites": observed_client_calls,
    }


# --------------------------------------------------------------------------------------------------
# Execute the focused tests
# --------------------------------------------------------------------------------------------------

ui_test_cases: tuple[
    tuple[str, Callable[[], dict[str, Any]]],
    ...,
] = (
    (
        "initial_streamlit_render",
        test_initial_streamlit_render,
    ),
    (
        "session_state_transitions",
        test_session_state_transitions,
    ),
    (
        "finding_table_construction",
        test_finding_table_construction,
    ),
    (
        "base64_evidence_decoding",
        test_base64_evidence_decoding,
    ),
    (
        "http_only_application_boundary",
        test_http_only_application_boundary,
    ),
)

ui_test_results = []

for test_name, test_function in ui_test_cases:
    try:
        test_details = test_function()

        ui_test_results.append(
            {
                "test_name": test_name,
                "status": "PASS",
                "details": test_details,
            }
        )

    except Exception as exc:
        ui_test_results.append(
            {
                "test_name": test_name,
                "status": "FAIL",
                "details": {
                    "exception_type": type(exc).__name__,
                    "message": str(exc),
                },
            }
        )

passed_ui_tests = sum(
    result["status"] == "PASS"
    for result in ui_test_results
)

failed_ui_tests = sum(
    result["status"] == "FAIL"
    for result in ui_test_results
)


# --------------------------------------------------------------------------------------------------
# Persist test evidence
# --------------------------------------------------------------------------------------------------

UI_COMPONENT_TEST_EVIDENCE_PATH = (
    UI_OUTPUT_ROOT
    / "ui_component_state_tests.json"
)

ui_test_evidence = {
    "evidence_version": (
        "streamlit-component-state-tests-v1"
    ),
    "evaluated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "total_tests": len(ui_test_results),
    "passed_tests": passed_ui_tests,
    "failed_tests": failed_ui_tests,
    "results": ui_test_results,
}

UI_COMPONENT_TEST_EVIDENCE_PATH.write_text(
    json.dumps(
        ui_test_evidence,
        indent=2,
    ),
    encoding="utf-8",
)


# --------------------------------------------------------------------------------------------------
# Formal test summary
# --------------------------------------------------------------------------------------------------

print("UI COMPONENT AND STATE-TRANSITION TESTING")
print("-" * 100)

for result in ui_test_results:
    print(
        f"{result['test_name']:<42}: "
        f"{result['status']}"
    )

print()
print(f"Executed tests             : {len(ui_test_results)}")
print(f"Passed tests               : {passed_ui_tests}")
print(f"Failed tests               : {failed_ui_tests}")
print(f"Evidence artifact          : {UI_COMPONENT_TEST_EVIDENCE_PATH}")
print(f"Streamlit test runtime     : AppTest")
print(f"Backend inference repeated : NO")

if failed_ui_tests:
    failed_details = [
        result
        for result in ui_test_results
        if result["status"] == "FAIL"
    ]

    raise RuntimeError(
        "One or more focused UI tests failed: "
        f"{failed_details}"
    )

print()
print("READY FOR INDEPENDENT STREAMLIT STARTUP VALIDATION")


2026-08-09 01:39:04.472 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


UI COMPONENT AND STATE-TRANSITION TESTING
----------------------------------------------------------------------------------------------------
initial_streamlit_render                  : PASS
session_state_transitions                 : PASS
finding_table_construction                : PASS
base64_evidence_decoding                  : PASS
http_only_application_boundary            : PASS

Executed tests             : 5
Passed tests               : 5
Failed tests               : 0
Evidence artifact          : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/ui/ui_component_state_tests.json
Streamlit test runtime     : AppTest
Backend inference repeated : NO

READY FOR INDEPENDENT STREAMLIT STARTUP VALIDATION


## 18. Independent Streamlit Startup Validation

This section starts the persisted Streamlit application as an independent process while retaining the already-running FastAPI backend. A healthy Streamlit process is reused only when its persisted application and configuration signature matches the current files; an older instance of this exact application is replaced safely. The block validates Streamlit’s health endpoint, root HTML response, backend connectivity, process stability, and startup logs without submitting an image or triggering model inference.


In [18]:
import hashlib
import json
import os
import signal
import subprocess
import time
from pathlib import Path

import httpx


# --------------------------------------------------------------------------------------------------
# Independent Streamlit launch configuration
# --------------------------------------------------------------------------------------------------

STREAMLIT_HOST = "127.0.0.1"
STREAMLIT_PORT = 8501

STREAMLIT_BASE_URL = (
    f"http://{STREAMLIT_HOST}:{STREAMLIT_PORT}"
)

STREAMLIT_HEALTH_URL = (
    f"{STREAMLIT_BASE_URL}/_stcore/health"
)

STREAMLIT_LOG_PATH = (
    UI_OUTPUT_ROOT
    / "notebook8_streamlit_startup.log"
)

STREAMLIT_STARTUP_TIMEOUT_SECONDS = 60

STREAMLIT_RUNTIME_STATE_PATH = (
    UI_OUTPUT_ROOT / "streamlit_runtime_state.json"
)


def streamlit_deployment_signature() -> str:
    """Hash the persisted application and its two configurations."""
    digest = hashlib.sha256()

    for path in (
        STREAMLIT_APP_PATH,
        UI_CONFIG_PATH,
        STREAMLIT_SERVER_CONFIG_PATH,
    ):
        digest.update(str(path).encode("utf-8"))
        digest.update(path.read_bytes())

    return digest.hexdigest()


STREAMLIT_DEPLOYMENT_SIGNATURE = (
    streamlit_deployment_signature()
)

STREAMLIT_PROCESS_STARTED_BY_NOTEBOOK = False
streamlit_process = None
streamlit_log_handle = None


def streamlit_health_available() -> bool:
    """Check the independent Streamlit health endpoint."""
    try:
        response = httpx.get(
            STREAMLIT_HEALTH_URL,
            timeout=3.0,
        )

        return (
            response.status_code == 200
            and response.text.strip().lower()
            == "ok"
        )

    except httpx.RequestError:
        return False


def read_streamlit_log_tail(
    maximum_lines: int = 50,
) -> str:
    """Read a bounded Streamlit startup-log tail."""
    if not STREAMLIT_LOG_PATH.is_file():
        return "Streamlit log is not available."

    log_lines = STREAMLIT_LOG_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()

    return "\n".join(
        log_lines[-maximum_lines:]
    )


# --------------------------------------------------------------------------------------------------
# Reuse only a matching deployment; replace an older instance of this exact app
# --------------------------------------------------------------------------------------------------


def managed_streamlit_process_ids() -> list[int]:
    """Return only processes serving this exact app on the configured port."""
    process_ids = []

    for process_root in Path("/proc").iterdir():
        if not process_root.name.isdigit():
            continue

        command_path = process_root / "cmdline"

        try:
            command_parts = command_path.read_bytes().split(b"\0")
            command_text = " ".join(
                part.decode("utf-8", errors="replace")
                for part in command_parts
                if part
            )
        except (FileNotFoundError, PermissionError, ProcessLookupError):
            continue

        if (
            "streamlit" in command_text
            and " run " in f" {command_text} "
            and str(STREAMLIT_APP_PATH) in command_text
            and str(STREAMLIT_PORT) in command_text
        ):
            process_ids.append(int(process_root.name))

    return sorted(process_ids)


runtime_state = {}

if STREAMLIT_RUNTIME_STATE_PATH.is_file():
    try:
        runtime_state = json.loads(
            STREAMLIT_RUNTIME_STATE_PATH.read_text(
                encoding="utf-8"
            )
        )
    except (json.JSONDecodeError, OSError):
        runtime_state = {}

STREAMLIT_DEPLOYMENT_CURRENT = (
    streamlit_health_available()
    and runtime_state.get("deployment_signature")
    == STREAMLIT_DEPLOYMENT_SIGNATURE
    and runtime_state.get("application_path")
    == str(STREAMLIT_APP_PATH)
)

if streamlit_health_available() and not STREAMLIT_DEPLOYMENT_CURRENT:
    stale_process_ids = managed_streamlit_process_ids()

    if not stale_process_ids:
        raise RuntimeError(
            "Port 8501 is healthy, but the process could not be "
            "identified as this Streamlit application. It was not stopped."
        )

    for process_id in stale_process_ids:
        os.kill(process_id, signal.SIGTERM)

    shutdown_deadline = time.monotonic() + 20

    while (
        streamlit_health_available()
        and time.monotonic() < shutdown_deadline
    ):
        time.sleep(0.5)

    if streamlit_health_available():
        raise RuntimeError(
            "The older Streamlit process did not stop cleanly."
        )


# --------------------------------------------------------------------------------------------------
# Start Streamlit only when the current deployment is not already healthy
# --------------------------------------------------------------------------------------------------

if not streamlit_health_available():
    streamlit_environment = os.environ.copy()

    streamlit_environment.update(
        {
            "CHEST_XRAY_API_BASE_URL": API_BASE_URL,
            "TOKENIZERS_PARALLELISM": "false",
            "PYTHONUNBUFFERED": "1",
        }
    )

    existing_python_path = (
        streamlit_environment.get(
            "PYTHONPATH",
            "",
        )
    )

    streamlit_environment["PYTHONPATH"] = (
        f"{SOLUTION_ROOT}"
        if not existing_python_path
        else f"{SOLUTION_ROOT}:{existing_python_path}"
    )

    streamlit_log_handle = STREAMLIT_LOG_PATH.open(
        "w",
        encoding="utf-8",
    )

    streamlit_process = subprocess.Popen(
        [
            "/opt/conda/bin/python",
            "-m",
            "streamlit",
            "run",
            str(STREAMLIT_APP_PATH),
            "--server.address",
            STREAMLIT_HOST,
            "--server.port",
            str(STREAMLIT_PORT),
            "--server.maxUploadSize",
            str(MAXIMUM_UPLOAD_MIB),
            "--server.headless",
            "true",
            "--server.fileWatcherType",
            "none",
            "--browser.gatherUsageStats",
            "false",
        ],
        cwd=str(SOLUTION_ROOT),
        env=streamlit_environment,
        stdout=streamlit_log_handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    STREAMLIT_PROCESS_STARTED_BY_NOTEBOOK = True

    startup_deadline = (
        time.monotonic()
        + STREAMLIT_STARTUP_TIMEOUT_SECONDS
    )

    while time.monotonic() < startup_deadline:
        if streamlit_process.poll() is not None:
            if streamlit_log_handle is not None:
                streamlit_log_handle.flush()

            raise RuntimeError(
                "The independent Streamlit process exited during "
                "startup.\n\n"
                f"{read_streamlit_log_tail()}"
            )

        if streamlit_health_available():
            break

        time.sleep(1.0)

    if not streamlit_health_available():
        if streamlit_log_handle is not None:
            streamlit_log_handle.flush()

        raise RuntimeError(
            "Streamlit did not become healthy within "
            f"{STREAMLIT_STARTUP_TIMEOUT_SECONDS} seconds.\n\n"
            f"{read_streamlit_log_tail()}"
        )


STREAMLIT_RUNTIME_STATE_PATH.write_text(
    json.dumps(
        {
            "deployment_signature": STREAMLIT_DEPLOYMENT_SIGNATURE,
            "application_path": str(STREAMLIT_APP_PATH),
            "streamlit_base_url": STREAMLIT_BASE_URL,
            "process_id": (
                streamlit_process.pid
                if streamlit_process is not None
                else runtime_state.get("process_id")
            ),
        },
        indent=2,
    ),
    encoding="utf-8",
)


# --------------------------------------------------------------------------------------------------
# Validate root HTML, backend health, process state, and startup log
# --------------------------------------------------------------------------------------------------

streamlit_health_response = httpx.get(
    STREAMLIT_HEALTH_URL,
    timeout=5.0,
)

streamlit_root_response = httpx.get(
    STREAMLIT_BASE_URL,
    timeout=10.0,
)

fastapi_health_response = httpx.get(
    f"{API_BASE_URL}/health",
    timeout=10.0,
)

STREAMLIT_HEALTH_CONFIRMED = (
    streamlit_health_response.status_code == 200
    and streamlit_health_response.text.strip().lower()
    == "ok"
)

STREAMLIT_ROOT_CONFIRMED = (
    streamlit_root_response.status_code == 200
    and "text/html"
    in streamlit_root_response.headers.get(
        "content-type",
        "",
    ).lower()
)

FASTAPI_HEALTH_CONFIRMED = (
    fastapi_health_response.status_code == 200
)

STREAMLIT_PROCESS_ALIVE = (
    streamlit_process is None
    or streamlit_process.poll() is None
)

if streamlit_log_handle is not None:
    streamlit_log_handle.flush()

streamlit_log_text = (
    STREAMLIT_LOG_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    )
    if STREAMLIT_LOG_PATH.is_file()
    else ""
)

STREAMLIT_TRACEBACK_FREE = (
    "Traceback (most recent call last)"
    not in streamlit_log_text
)


# --------------------------------------------------------------------------------------------------
# Startup validation gates
# --------------------------------------------------------------------------------------------------

if not STREAMLIT_HEALTH_CONFIRMED:
    raise RuntimeError(
        "The independent Streamlit health endpoint did not pass."
    )

if not STREAMLIT_ROOT_CONFIRMED:
    raise RuntimeError(
        "The Streamlit root page did not return an HTML response."
    )

if not FASTAPI_HEALTH_CONFIRMED:
    raise RuntimeError(
        "The FastAPI backend was unavailable during "
        "Streamlit startup validation."
    )

if not STREAMLIT_PROCESS_ALIVE:
    raise RuntimeError(
        "The Streamlit process is no longer running."
    )

if not STREAMLIT_TRACEBACK_FREE:
    raise RuntimeError(
        "The Streamlit startup log contains a traceback.\n\n"
        f"{read_streamlit_log_tail()}"
    )


# --------------------------------------------------------------------------------------------------
# Formal startup summary
# --------------------------------------------------------------------------------------------------

print("INDEPENDENT STREAMLIT STARTUP VALIDATION")
print("-" * 100)
print(f"Streamlit application      : {STREAMLIT_APP_PATH}")
print(f"Streamlit base URL         : {STREAMLIT_BASE_URL}")
print(f"Streamlit health URL       : {STREAMLIT_HEALTH_URL}")
print(
    f"Process source             : "
    f"{'STARTED BY NOTEBOOK 8' if STREAMLIT_PROCESS_STARTED_BY_NOTEBOOK else 'EXISTING HEALTHY PROCESS'}"
)
print(
    f"Streamlit process ID       : "
    f"{streamlit_process.pid if streamlit_process is not None else 'EXTERNAL'}"
)
print(f"Streamlit health response  : PASS")
print(f"Streamlit root HTML        : PASS")
print(f"FastAPI backend health     : PASS")
print(f"Startup traceback          : NONE")
print(f"Startup log                : {STREAMLIT_LOG_PATH}")
print(f"Image submitted            : NO")
print(f"Model inference triggered  : NO")
print(f"Streamlit process retained : YES")
print()
print("READY FOR LIVE INTERFACE WORKFLOW VALIDATION")


INDEPENDENT STREAMLIT STARTUP VALIDATION
----------------------------------------------------------------------------------------------------
Streamlit application      : /home/jovyan/chest-xray-ai-assistant/ui/app.py
Streamlit base URL         : http://127.0.0.1:8501
Streamlit health URL       : http://127.0.0.1:8501/_stcore/health
Process source             : STARTED BY NOTEBOOK 8
Streamlit process ID       : 1227
Streamlit health response  : PASS
Streamlit root HTML        : PASS
FastAPI backend health     : PASS
Startup traceback          : NONE
Startup log                : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/ui/notebook8_streamlit_startup.log
Image submitted            : NO
Model inference triggered  : NO
Streamlit process retained : YES

READY FOR LIVE INTERFACE WORKFLOW VALIDATION


## 19. Live Workflow Fixture and Interaction Inspection

This section inspects the persisted API test fixture and the installed Streamlit testing interface before selecting a live-workflow validation method. It avoids regenerating dataset samples, accessing the ChestMNIST archive, or assuming that the current Streamlit test runtime supports file-uploader interaction.


In [19]:
import ast
from pathlib import Path

from streamlit.testing.v1 import AppTest


# --------------------------------------------------------------------------------------------------
# Inspect the installed Streamlit test interaction surface
# --------------------------------------------------------------------------------------------------

interaction_test = AppTest.from_file(
    str(STREAMLIT_APP_PATH)
)

available_app_test_attributes = tuple(
    sorted(
        attribute_name
        for attribute_name in dir(
            interaction_test
        )
        if not attribute_name.startswith("_")
    )
)

FILE_UPLOADER_INTERACTION_AVAILABLE = (
    "file_uploader"
    in available_app_test_attributes
)

BUTTON_INTERACTION_AVAILABLE = (
    "button"
    in available_app_test_attributes
)

TEXT_INPUT_INTERACTION_AVAILABLE = (
    "text_input"
    in available_app_test_attributes
)

TEXT_AREA_INTERACTION_AVAILABLE = (
    "text_area"
    in available_app_test_attributes
)


# --------------------------------------------------------------------------------------------------
# Inspect the persisted API test fixture source
# --------------------------------------------------------------------------------------------------

API_TEST_FIXTURE_PATH = (
    TEST_ROOT / "api" / "conftest.py"
)

if not API_TEST_FIXTURE_PATH.is_file():
    raise FileNotFoundError(
        f"API test fixture was not found: {API_TEST_FIXTURE_PATH}"
    )

api_fixture_source = API_TEST_FIXTURE_PATH.read_text(
    encoding="utf-8"
)

api_fixture_tree = ast.parse(
    api_fixture_source,
    filename=str(API_TEST_FIXTURE_PATH),
)

fixture_functions = []

for node in api_fixture_tree.body:
    if not isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        ),
    ):
        continue

    decorator_names = []

    for decorator in node.decorator_list:
        try:
            decorator_names.append(
                ast.unparse(decorator)
            )
        except Exception:
            decorator_names.append(
                "<unavailable>"
            )

    fixture_functions.append(
        {
            "name": node.name,
            "decorators": tuple(
                decorator_names
            ),
            "source": ast.get_source_segment(
                api_fixture_source,
                node,
            ),
        }
    )


# --------------------------------------------------------------------------------------------------
# Identify image-related persisted fixture definitions
# --------------------------------------------------------------------------------------------------

IMAGE_FIXTURE_KEYWORDS = (
    "image",
    "png",
    "jpeg",
    "jpg",
    "bytesio",
    "pil",
)

image_related_fixtures = [
    fixture
    for fixture in fixture_functions
    if any(
        keyword in fixture["name"].lower()
        or keyword in fixture["source"].lower()
        for keyword in IMAGE_FIXTURE_KEYWORDS
    )
]

image_related_source_lines = []

for line_number, line in enumerate(
    api_fixture_source.splitlines(),
    start=1,
):
    normalized_line = line.lower()

    if any(
        keyword in normalized_line
        for keyword in IMAGE_FIXTURE_KEYWORDS
    ):
        image_related_source_lines.append(
            {
                "line_number": line_number,
                "text": line.strip(),
            }
        )


# --------------------------------------------------------------------------------------------------
# Formal inspection summary
# --------------------------------------------------------------------------------------------------

print("LIVE WORKFLOW FIXTURE AND INTERACTION INSPECTION")
print("-" * 110)
print(f"Streamlit application       : {STREAMLIT_APP_PATH}")
print(f"API fixture source          : {API_TEST_FIXTURE_PATH}")
print(
    f"AppTest file uploader       : "
    f"{'AVAILABLE' if FILE_UPLOADER_INTERACTION_AVAILABLE else 'NOT AVAILABLE'}"
)
print(
    f"AppTest button interaction  : "
    f"{'AVAILABLE' if BUTTON_INTERACTION_AVAILABLE else 'NOT AVAILABLE'}"
)
print(
    f"AppTest text input          : "
    f"{'AVAILABLE' if TEXT_INPUT_INTERACTION_AVAILABLE else 'NOT AVAILABLE'}"
)
print(
    f"AppTest text area           : "
    f"{'AVAILABLE' if TEXT_AREA_INTERACTION_AVAILABLE else 'NOT AVAILABLE'}"
)
print(
    f"Persisted fixture functions : "
    f"{len(fixture_functions)}"
)
print(
    f"Image-related fixtures      : "
    f"{len(image_related_fixtures)}"
)

print()
print("IMAGE-RELATED FIXTURE DEFINITIONS")
print("-" * 110)

if image_related_fixtures:
    for fixture in image_related_fixtures:
        print(
            f"{fixture['name']} "
            f"[{', '.join(fixture['decorators']) or 'no decorator'}]"
        )
        print(fixture["source"])
        print()
else:
    print("No image-related fixture function was identified.")

print("IMAGE-RELATED SOURCE LINES")
print("-" * 110)

if image_related_source_lines:
    for source_line in image_related_source_lines:
        print(
            f"{source_line['line_number']:>4}: "
            f"{source_line['text']}"
        )
else:
    print("No image-related source lines were identified.")

print()
print("INSPECTION STATUS")
print("-" * 110)
print("Dataset archive accessed      : NO")
print("Memory-mapped dataset accessed: NO")
print("Image generated               : NO")
print("Model inference triggered     : NO")
print()
print("READY TO SELECT THE SUPPORTED LIVE WORKFLOW VALIDATION METHOD")


2026-08-09 01:39:05.897 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


LIVE WORKFLOW FIXTURE AND INTERACTION INSPECTION
--------------------------------------------------------------------------------------------------------------
Streamlit application       : /home/jovyan/chest-xray-ai-assistant/ui/app.py
API fixture source          : /home/jovyan/chest-xray-ai-assistant/tests/api/conftest.py
AppTest file uploader       : NOT AVAILABLE
AppTest button interaction  : AVAILABLE
AppTest text input          : AVAILABLE
AppTest text area           : AVAILABLE
Persisted fixture functions : 3
Image-related fixtures      : 2

IMAGE-RELATED FIXTURE DEFINITIONS
--------------------------------------------------------------------------------------------------------------
valid_png_bytes [pytest.fixture(scope='session')]
def valid_png_bytes():
    horizontal = np.linspace(
        32,
        224,
        224,
        dtype=np.uint8,
    )

    image_array = np.tile(
        horizontal,
        (224, 1),
    )

    image = Image.fromarray(
        image_array,
      

## 20. Live Interface Workflow Validation

This section uses the exact persisted API test-image construction to exercise the running FastAPI backend through the UI HTTP client. It validates complete analysis, grounded follow-up questioning, stored-prediction retrieval, operational metrics, and rendering of the live responses through an isolated Streamlit component harness. The ChestMNIST dataset is not accessed.


In [20]:
import io
import json
import sys
import textwrap
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
from PIL import Image
from streamlit.testing.v1 import AppTest


# --------------------------------------------------------------------------------------------------
# Reproduce the exact persisted API test fixture
# --------------------------------------------------------------------------------------------------

horizontal = np.linspace(
    32,
    224,
    224,
    dtype=np.uint8,
)

live_image_array = np.tile(
    horizontal,
    (224, 1),
)

live_test_image = Image.fromarray(
    live_image_array,
    mode="L",
)

live_image_buffer = io.BytesIO()

live_test_image.save(
    live_image_buffer,
    format="PNG",
)

live_png_bytes = live_image_buffer.getvalue()

LIVE_TEST_FILENAME = (
    "streamlit-live-workflow.png"
)

LIVE_TEST_MEDIA_TYPE = "image/png"

LIVE_INITIAL_QUESTION = (
    "Which model findings crossed their frozen thresholds?"
)

LIVE_FOLLOW_UP_QUESTION = (
    "What does this result mean in plain language?"
)


# --------------------------------------------------------------------------------------------------
# Execute the live workflow through the persisted UI HTTP client
# --------------------------------------------------------------------------------------------------

with ChestXRayAPIClient(
    base_url=API_BASE_URL,
    timeout_seconds=120.0,
) as live_workflow_client:
    live_complete_response = (
        live_workflow_client.analyze_complete(
            filename=LIVE_TEST_FILENAME,
            media_type=LIVE_TEST_MEDIA_TYPE,
            image_content=live_png_bytes,
            question=LIVE_INITIAL_QUESTION,
        )
    )

    live_prediction_id = (
        live_complete_response.get(
            "prediction_id"
        )
    )

    live_question_response = (
        live_workflow_client.answer_question(
            prediction_id=live_prediction_id,
            question=LIVE_FOLLOW_UP_QUESTION,
        )
    )

    live_stored_response = (
        live_workflow_client.get_prediction(
            live_prediction_id
        )
    )

    live_metrics_response = (
        live_workflow_client.llmops_metrics()
    )


# --------------------------------------------------------------------------------------------------
# Validate responses against the persisted OpenAPI requirements
# --------------------------------------------------------------------------------------------------

complete_required_fields = set(
    component_schemas[
        complete_response_schema_name
    ].get(
        "required",
        [],
    )
)

question_required_fields = set(
    component_schemas[
        question_response_schema_name
    ].get(
        "required",
        [],
    )
)

stored_required_fields = set(
    component_schemas[
        stored_prediction_schema_name
    ].get(
        "required",
        [],
    )
)

metrics_required_fields = set(
    component_schemas[
        operational_metrics_schema_name
    ].get(
        "required",
        [],
    )
)

missing_complete_fields = sorted(
    complete_required_fields
    - set(live_complete_response)
)

missing_question_fields = sorted(
    question_required_fields
    - set(live_question_response)
)

missing_stored_fields = sorted(
    stored_required_fields
    - set(live_stored_response)
)

missing_metrics_fields = sorted(
    metrics_required_fields
    - set(live_metrics_response)
)

if missing_complete_fields:
    raise RuntimeError(
        "The live complete-analysis response is missing required "
        f"fields: {missing_complete_fields}"
    )

if missing_question_fields:
    raise RuntimeError(
        "The live question response is missing required fields: "
        f"{missing_question_fields}"
    )

if missing_stored_fields:
    raise RuntimeError(
        "The stored-prediction response is missing required fields: "
        f"{missing_stored_fields}"
    )

if missing_metrics_fields:
    raise RuntimeError(
        "The operational-metrics response is missing required fields: "
        f"{missing_metrics_fields}"
    )


# --------------------------------------------------------------------------------------------------
# Validate workflow identity and evidence relationships
# --------------------------------------------------------------------------------------------------

live_findings = live_complete_response.get(
    "findings",
    [],
)

live_visual_evidence = live_complete_response.get(
    "visual_evidence",
    [],
)

live_language_outputs = live_complete_response.get(
    "language_outputs",
    [],
)

live_language_task_types = {
    output.get("task_type")
    for output in live_language_outputs
    if isinstance(output, dict)
}

expected_complete_task_types = {
    "structured_report",
    "plain_language_explanation",
    "educational_follow_up",
    "grounded_question_answering",
}

unexpected_visual_records = [
    evidence
    for evidence in live_visual_evidence
    if not isinstance(evidence, dict)
    or evidence.get("crossed_threshold") is not True
]

if len(live_findings) != 14:
    raise RuntimeError(
        "The complete-analysis response did not preserve all "
        f"fourteen findings: found {len(live_findings)}."
    )

if live_language_task_types != expected_complete_task_types:
    raise RuntimeError(
        "The complete-analysis language tasks differ from the "
        f"persisted contract: {sorted(live_language_task_types)}"
    )

if unexpected_visual_records:
    raise RuntimeError(
        "Visual evidence was returned for a record that was not "
        "marked as crossed-threshold."
    )

if (
    live_question_response.get("prediction_id")
    != live_prediction_id
):
    raise RuntimeError(
        "The grounded-question response changed the prediction ID."
    )

if (
    live_stored_response.get("prediction_id")
    != live_prediction_id
):
    raise RuntimeError(
        "Stored-prediction retrieval returned a different prediction ID."
    )

if (
    live_complete_response.get(
        "educational_use_only"
    )
    is not True
):
    raise RuntimeError(
        "The complete-analysis response did not preserve the "
        "educational-use boundary."
    )


# --------------------------------------------------------------------------------------------------
# Persist live response evidence
# --------------------------------------------------------------------------------------------------

LIVE_COMPLETE_RESPONSE_PATH = (
    UI_OUTPUT_ROOT
    / "live_complete_analysis_response.json"
)

LIVE_QUESTION_RESPONSE_PATH = (
    UI_OUTPUT_ROOT
    / "live_grounded_question_response.json"
)

LIVE_STORED_RESPONSE_PATH = (
    UI_OUTPUT_ROOT
    / "live_stored_prediction_response.json"
)

LIVE_METRICS_RESPONSE_PATH = (
    UI_OUTPUT_ROOT
    / "live_operational_metrics_response.json"
)

response_artifacts = (
    (
        LIVE_COMPLETE_RESPONSE_PATH,
        live_complete_response,
    ),
    (
        LIVE_QUESTION_RESPONSE_PATH,
        live_question_response,
    ),
    (
        LIVE_STORED_RESPONSE_PATH,
        live_stored_response,
    ),
    (
        LIVE_METRICS_RESPONSE_PATH,
        live_metrics_response,
    ),
)

for artifact_path, payload in response_artifacts:
    artifact_path.write_text(
        json.dumps(
            payload,
            indent=2,
        ),
        encoding="utf-8",
    )


# --------------------------------------------------------------------------------------------------
# Create an isolated Streamlit renderer for the live responses
# --------------------------------------------------------------------------------------------------

LIVE_RENDER_HARNESS_PATH = (
    UI_OUTPUT_ROOT
    / "live_response_render_harness.py"
)

live_render_harness_source = textwrap.dedent(
    f'''
    from __future__ import annotations

    import json
    import sys
    from pathlib import Path

    import streamlit as st
    import yaml


    UI_ROOT = Path({str(UI_ROOT)!r})
    UI_CONFIG_PATH = Path({str(UI_CONFIG_PATH)!r})

    if str(UI_ROOT) not in sys.path:
        sys.path.insert(0, str(UI_ROOT))

    from components import (
        render_finding_summary,
        render_grounded_question_response,
        render_language_outputs,
        render_stored_prediction_summary,
        render_visual_evidence,
    )


    def load_json(path_value: str) -> dict:
        with Path(path_value).open(
            "r",
            encoding="utf-8",
        ) as file:
            return json.load(file)


    ui_config = yaml.safe_load(
        UI_CONFIG_PATH.read_text(
            encoding="utf-8"
        )
    )

    complete_response = load_json(
        {str(LIVE_COMPLETE_RESPONSE_PATH)!r}
    )

    question_response = load_json(
        {str(LIVE_QUESTION_RESPONSE_PATH)!r}
    )

    stored_response = load_json(
        {str(LIVE_STORED_RESPONSE_PATH)!r}
    )


    st.set_page_config(
        page_title="Live Response Rendering Validation",
        layout="wide",
    )

    st.title("Live Response Rendering Validation")

    render_finding_summary(
        complete_response
    )

    render_visual_evidence(
        complete_response,
        ui_config["safety"]["gradcam_limitation"],
    )

    render_language_outputs(
        complete_response
    )

    st.divider()
    render_grounded_question_response(
        question_response
    )

    st.divider()
    render_stored_prediction_summary(
        stored_response
    )
    '''
).strip() + "\n"

LIVE_RENDER_HARNESS_PATH.write_text(
    live_render_harness_source,
    encoding="utf-8",
)


# --------------------------------------------------------------------------------------------------
# Validate live response rendering through Streamlit AppTest
# --------------------------------------------------------------------------------------------------

live_render_test = AppTest.from_file(
    str(LIVE_RENDER_HARNESS_PATH)
)

live_render_test.run(
    timeout=60,
)

live_render_exceptions = list(
    live_render_test.exception
)

if live_render_exceptions:
    raise RuntimeError(
        "The live response rendering harness raised exceptions: "
        f"{[str(item.value) for item in live_render_exceptions]}"
    )


# --------------------------------------------------------------------------------------------------
# Persist compact workflow evidence
# --------------------------------------------------------------------------------------------------

LIVE_WORKFLOW_EVIDENCE_PATH = (
    UI_OUTPUT_ROOT
    / "live_interface_workflow_validation.json"
)

live_workflow_evidence = {
    "evidence_version": (
        "streamlit-live-workflow-validation-v1"
    ),
    "evaluated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "fixture_source": str(
        API_TEST_FIXTURE_PATH
    ),
    "dataset_accessed": False,
    "prediction_id": live_prediction_id,
    "finding_count": len(live_findings),
    "crossed_finding_count": len(
        live_complete_response.get(
            "crossed_finding_names",
            [],
        )
    ),
    "no_target_finding": (
        live_complete_response.get(
            "no_target_finding"
        )
    ),
    "visual_evidence_count": len(
        live_visual_evidence
    ),
    "language_task_types": sorted(
        live_language_task_types
    ),
    "question_guardrail_action": (
        live_question_response.get(
            "guardrail_action"
        )
    ),
    "stored_prediction_verified": True,
    "operational_metrics_verified": True,
    "streamlit_render_exceptions": 0,
    "response_artifacts": [
        str(path)
        for path, _ in response_artifacts
    ],
}

LIVE_WORKFLOW_EVIDENCE_PATH.write_text(
    json.dumps(
        live_workflow_evidence,
        indent=2,
    ),
    encoding="utf-8",
)


# --------------------------------------------------------------------------------------------------
# Formal live-workflow summary
# --------------------------------------------------------------------------------------------------

print("LIVE INTERFACE WORKFLOW VALIDATION")
print("-" * 110)
print(f"Fixture source                : {API_TEST_FIXTURE_PATH}")
print(f"Fixture media type            : {LIVE_TEST_MEDIA_TYPE}")
print(f"Fixture image size            : {len(live_png_bytes)} bytes")
print(f"Prediction ID                 : {live_prediction_id}")
print(f"Findings returned             : {len(live_findings)}")
print(
    f"Thresholds crossed            : "
    f"{len(live_complete_response.get('crossed_finding_names', []))}"
)
print(
    f"No target finding             : "
    f"{live_complete_response.get('no_target_finding')}"
)
print(f"Visual-evidence records       : {len(live_visual_evidence)}")
print(f"Language tasks returned       : {len(live_language_task_types)}")
print(
    f"Question guardrail action     : "
    f"{live_question_response.get('guardrail_action')}"
)
print(f"Stored prediction retrieved   : PASS")
print(f"Operational metrics retrieved : PASS")
print(f"Live response rendering       : PASS")
print(f"Streamlit render exceptions   : 0")
print(f"Workflow evidence             : {LIVE_WORKFLOW_EVIDENCE_PATH}")
print(f"Dataset archive accessed      : NO")
print(f"Memory-mapped dataset accessed: NO")
print()
print("READY FOR SCREENSHOT-READY INTERFACE VALIDATION")


2026-08-09 01:39:12.322 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


LIVE INTERFACE WORKFLOW VALIDATION
--------------------------------------------------------------------------------------------------------------
Fixture source                : /home/jovyan/chest-xray-ai-assistant/tests/api/conftest.py
Fixture media type            : image/png
Fixture image size            : 332 bytes
Prediction ID                 : 01b21772-4c48-4acc-96ac-a45bbf6a2a45
Findings returned             : 14
Thresholds crossed            : 0
No target finding             : True
Visual-evidence records       : 0
Language tasks returned       : 4
Question guardrail action     : accepted_model_generation
Stored prediction retrieved   : PASS
Operational metrics retrieved : PASS
Live response rendering       : PASS
Streamlit render exceptions   : 0
Workflow evidence             : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/ui/live_interface_workflow_validation.json
Dataset archive accessed      : NO
Memory-mapped dataset accessed: NO

READY FOR SCREENSH

## 21. Screenshot-Capture Capability Inspection

This section inspects the existing runtime for browser executables, supported browser-automation packages, and Jupyter proxy routing. It does not install additional software, start another application process, or modify the validated interface.


In [21]:
import importlib.util
import os
import shutil
from pathlib import Path


# --------------------------------------------------------------------------------------------------
# Inspect available browser executables
# --------------------------------------------------------------------------------------------------

BROWSER_EXECUTABLE_CANDIDATES = (
    "chromium",
    "chromium-browser",
    "google-chrome",
    "google-chrome-stable",
    "firefox",
)

available_browser_executables = {
    executable_name: shutil.which(
        executable_name
    )
    for executable_name
    in BROWSER_EXECUTABLE_CANDIDATES
    if shutil.which(executable_name)
}


# --------------------------------------------------------------------------------------------------
# Inspect installed browser-automation packages
# --------------------------------------------------------------------------------------------------

BROWSER_AUTOMATION_PACKAGES = (
    "playwright",
    "selenium",
    "pyppeteer",
)

available_browser_packages = {
    package_name: (
        importlib.util.find_spec(
            package_name
        )
        is not None
    )
    for package_name
    in BROWSER_AUTOMATION_PACKAGES
}


# --------------------------------------------------------------------------------------------------
# Inspect Jupyter proxy-routing context
# --------------------------------------------------------------------------------------------------

proxy_environment_keys = (
    "JUPYTERHUB_SERVICE_PREFIX",
    "JUPYTERHUB_BASE_URL",
    "NB_PREFIX",
)

proxy_environment = {
    key: os.getenv(key)
    for key in proxy_environment_keys
    if os.getenv(key)
}

service_prefix = (
    proxy_environment.get(
        "JUPYTERHUB_SERVICE_PREFIX"
    )
    or proxy_environment.get(
        "NB_PREFIX"
    )
)

if service_prefix:
    normalized_service_prefix = (
        service_prefix.rstrip("/")
    )

    STREAMLIT_PROXY_PATH = (
        f"{normalized_service_prefix}/proxy/"
        f"{STREAMLIT_PORT}/"
    )
else:
    STREAMLIT_PROXY_PATH = None


# --------------------------------------------------------------------------------------------------
# Confirm that the validated processes remain healthy
# --------------------------------------------------------------------------------------------------

FASTAPI_REMAINS_HEALTHY = (
    backend_health_available()
)

STREAMLIT_REMAINS_HEALTHY = (
    streamlit_health_available()
)

current_storage_usage = shutil.disk_usage(
    DATA_ROOT
)

CURRENT_FREE_STORAGE_GIB = (
    current_storage_usage.free / (1024 ** 3)
)

CURRENT_STORAGE_RESERVE_PRESERVED = (
    CURRENT_FREE_STORAGE_GIB
    >= MINIMUM_FREE_STORAGE_GIB
)


# --------------------------------------------------------------------------------------------------
# Formal capability summary
# --------------------------------------------------------------------------------------------------

print("SCREENSHOT-CAPTURE CAPABILITY INSPECTION")
print("-" * 100)

print("Browser executables")
print("-" * 100)

if available_browser_executables:
    for browser_name, browser_path in (
        available_browser_executables.items()
    ):
        print(
            f"{browser_name:<25}: {browser_path}"
        )
else:
    print("No supported browser executable was identified.")

print()
print("Browser-automation packages")
print("-" * 100)

for package_name, available in (
    available_browser_packages.items()
):
    print(
        f"{package_name:<25}: "
        f"{'AVAILABLE' if available else 'NOT AVAILABLE'}"
    )

print()
print("Jupyter proxy context")
print("-" * 100)

if proxy_environment:
    for key, value in proxy_environment.items():
        print(f"{key:<30}: {value}")
else:
    print("No Jupyter proxy environment value was identified.")

print(
    f"Derived Streamlit proxy path : "
    f"{STREAMLIT_PROXY_PATH or 'UNAVAILABLE'}"
)

print()
print("Validated service state")
print("-" * 100)
print(
    f"FastAPI health               : "
    f"{'PASS' if FASTAPI_REMAINS_HEALTHY else 'FAIL'}"
)
print(
    f"Streamlit health             : "
    f"{'PASS' if STREAMLIT_REMAINS_HEALTHY else 'FAIL'}"
)
print(
    f"Free storage                 : "
    f"{CURRENT_FREE_STORAGE_GIB:.2f} GiB"
)
print(
    f"Storage reserve              : "
    f"{'PASS' if CURRENT_STORAGE_RESERVE_PRESERVED else 'FAIL'}"
)
print()
print("READY TO SELECT THE AVAILABLE SCREENSHOT VALIDATION METHOD")


SCREENSHOT-CAPTURE CAPABILITY INSPECTION
----------------------------------------------------------------------------------------------------
Browser executables
----------------------------------------------------------------------------------------------------
No supported browser executable was identified.

Browser-automation packages
----------------------------------------------------------------------------------------------------
playwright               : NOT AVAILABLE
selenium                 : NOT AVAILABLE
pyppeteer                : NOT AVAILABLE

Jupyter proxy context
----------------------------------------------------------------------------------------------------
NB_PREFIX                     : /notebook/2024ac05653-s2-25-aimlczg536/apicdsa2
Derived Streamlit proxy path : /notebook/2024ac05653-s2-25-aimlczg536/apicdsa2/proxy/8501/

Validated service state
----------------------------------------------------------------------------------------------------
FastAPI health 

## 22. Screenshot-Ready Interface Validation

This section validates that the live interface contains the required workflow sections, safety notices, technical details, and rendering components. It preserves a small non-clinical demonstration image and provides the confirmed Jupyter proxy link for manual browser review and screenshot capture.


In [22]:
import ast
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

import yaml
from IPython.display import HTML, display


# --------------------------------------------------------------------------------------------------
# Preserve the validated non-clinical demonstration fixture
# --------------------------------------------------------------------------------------------------

REPORTS_ROOT = SOLUTION_ROOT / "reports"

REPORTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

UI_DEMO_INPUT_PATH = (
    REPORTS_ROOT
    / "streamlit_demo_input.png"
)

UI_DEMO_INPUT_PATH.write_bytes(
    live_png_bytes
)


# --------------------------------------------------------------------------------------------------
# Parse the authoritative UI configuration and Python modules
# --------------------------------------------------------------------------------------------------

parsed_ui_configuration = yaml.safe_load(
    UI_CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)

application_source = STREAMLIT_APP_PATH.read_text(
    encoding="utf-8"
)

components_source = COMPONENTS_PATH.read_text(
    encoding="utf-8"
)

application_tree = ast.parse(
    application_source,
    filename=str(STREAMLIT_APP_PATH),
)

components_tree = ast.parse(
    components_source,
    filename=str(COMPONENTS_PATH),
)


def collect_function_names(
    syntax_tree: ast.Module,
) -> set[str]:
    """Collect persisted function names from a module."""
    return {
        node.name
        for node in syntax_tree.body
        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            ),
        )
    }


def collect_string_literals(
    syntax_tree: ast.Module,
) -> tuple[str, ...]:
    """Collect compiled Python string constants."""
    return tuple(
        node.value
        for node in ast.walk(syntax_tree)
        if isinstance(node, ast.Constant)
        and isinstance(node.value, str)
    )


def collect_streamlit_calls(
    syntax_tree: ast.Module,
) -> set[str]:
    """Collect Streamlit method names used by a module."""
    call_names = set()

    for node in ast.walk(syntax_tree):
        if not isinstance(node, ast.Call):
            continue

        if (
            isinstance(node.func, ast.Attribute)
            and isinstance(node.func.value, ast.Name)
            and node.func.value.id == "st"
        ):
            call_names.add(
                node.func.attr
            )

    return call_names


def contains_literal_text(
    literals: tuple[str, ...],
    expected_text: str,
) -> bool:
    """Search compiled string constants using normalized whitespace."""
    normalized_expected = " ".join(
        expected_text.split()
    ).lower()

    return any(
        normalized_expected
        in " ".join(value.split()).lower()
        for value in literals
    )


application_function_names = (
    collect_function_names(
        application_tree
    )
)

component_function_names = (
    collect_function_names(
        components_tree
    )
)

application_literals = (
    collect_string_literals(
        application_tree
    )
)

component_literals = (
    collect_string_literals(
        components_tree
    )
)

application_streamlit_calls = (
    collect_streamlit_calls(
        application_tree
    )
)

component_streamlit_calls = (
    collect_streamlit_calls(
        components_tree
    )
)

application_imported_names = {
    alias.name
    for node in application_tree.body
    if isinstance(node, ast.ImportFrom)
    for alias in node.names
}


# --------------------------------------------------------------------------------------------------
# Validate screenshot-required interface capabilities
# --------------------------------------------------------------------------------------------------

configured_page_title = (
    parsed_ui_configuration["page"]["title"]
)

configured_educational_limitation = (
    parsed_ui_configuration["safety"][
        "educational_limitation"
    ]
)

configured_gradcam_limitation = (
    parsed_ui_configuration["safety"][
        "gradcam_limitation"
    ]
)

screenshot_readiness_checks = {
    "application_title_present": (
        configured_page_title
        == "Chest X-Ray Analysis and Explanation Assistant"
    ),
    "educational_limitation_present": (
        configured_educational_limitation
        == EDUCATIONAL_LIMITATION
    ),
    "image_upload_present": (
        "file_uploader"
        in application_streamlit_calls
    ),
    "image_preview_present": (
        "image"
        in application_streamlit_calls
    ),
    "complete_analysis_present": (
        contains_literal_text(
            application_literals,
            "Run Complete Analysis",
        )
    ),
    "request_lineage_present": (
        contains_literal_text(
            application_literals,
            "Request and lineage identifiers",
        )
    ),
    "finding_summary_present": (
        "render_finding_summary"
        in component_function_names
    ),
    "no_target_wording_present": (
        contains_literal_text(
            component_literals,
            "must not be interpreted as confirmation "
            "of a clinically normal chest radiograph",
        )
    ),
    "gradcam_evidence_present": (
        "render_visual_evidence"
        in component_function_names
    ),
    "gradcam_limitation_present": (
        configured_gradcam_limitation
        == GRADCAM_LIMITATION
    ),
    "grounded_language_present": (
        "render_language_outputs"
        in component_function_names
    ),
    "guardrail_details_present": (
        contains_literal_text(
            component_literals,
            "Generation and guardrail details",
        )
    ),
    "follow_up_question_present": (
        contains_literal_text(
            application_literals,
            "Ask a Grounded Follow-Up Question",
        )
    ),
    "stored_prediction_present": (
        contains_literal_text(
            application_literals,
            "Retrieve the Stored Prediction",
        )
    ),
    "operational_metrics_present": (
        "render_operational_metrics_workflow"
        in application_function_names
    ),
    "controlled_error_rendering_present": (
        "render_api_error"
        in application_function_names
    ),
    "session_state_present": (
        "initialize_session_state"
        in application_function_names
    ),
    "http_only_client_present": (
        "ChestXRayAPIClient"
        in application_imported_names
    ),
    "live_workflow_evidence_present": (
        LIVE_WORKFLOW_EVIDENCE_PATH.is_file()
    ),
    "streamlit_health_confirmed": (
        streamlit_health_available()
    ),
    "fastapi_health_confirmed": (
        backend_health_available()
    ),
    "jupyter_proxy_path_available": (
        STREAMLIT_PROXY_PATH is not None
    ),
}

failed_screenshot_checks = [
    check_name
    for check_name, passed
    in screenshot_readiness_checks.items()
    if not passed
]

if failed_screenshot_checks:
    raise RuntimeError(
        "Screenshot-readiness validation failed: "
        f"{failed_screenshot_checks}"
    )


# --------------------------------------------------------------------------------------------------
# Validate the protected storage reserve
# --------------------------------------------------------------------------------------------------

current_storage_usage = shutil.disk_usage(
    DATA_ROOT
)

SCREENSHOT_FREE_STORAGE_GIB = (
    current_storage_usage.free / (1024 ** 3)
)

SCREENSHOT_STORAGE_RESERVE_PRESERVED = (
    SCREENSHOT_FREE_STORAGE_GIB
    >= MINIMUM_FREE_STORAGE_GIB
)

if not SCREENSHOT_STORAGE_RESERVE_PRESERVED:
    raise RuntimeError(
        "The protected storage reserve is no longer available."
    )


# --------------------------------------------------------------------------------------------------
# Persist screenshot-readiness evidence
# --------------------------------------------------------------------------------------------------

SCREENSHOT_READINESS_PATH = (
    UI_OUTPUT_ROOT
    / "screenshot_readiness_validation.json"
)

screenshot_readiness_evidence = {
    "evidence_version": (
        "streamlit-screenshot-readiness-v1"
    ),
    "evaluated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "validation_method": (
        "parsed_configuration_ast_validation_"
        "and_manual_jupyter_proxy_review"
    ),
    "browser_automation_available": False,
    "streamlit_base_url": STREAMLIT_BASE_URL,
    "jupyter_proxy_path": STREAMLIT_PROXY_PATH,
    "demonstration_input": str(
        UI_DEMO_INPUT_PATH
    ),
    "checks": screenshot_readiness_checks,
    "free_storage_gib": round(
        SCREENSHOT_FREE_STORAGE_GIB,
        2,
    ),
    "protected_reserve_gib": (
        MINIMUM_FREE_STORAGE_GIB
    ),
    "reserve_preserved": True,
}

SCREENSHOT_READINESS_PATH.write_text(
    json.dumps(
        screenshot_readiness_evidence,
        indent=2,
    ),
    encoding="utf-8",
)


# --------------------------------------------------------------------------------------------------
# Display the confirmed Jupyter proxy link
# --------------------------------------------------------------------------------------------------

display(
    HTML(
        f"""
        <div style="
            padding: 14px 16px;
            border: 1px solid #d0d7de;
            border-radius: 8px;
            background: #f6f8fa;
            margin: 8px 0 14px 0;
        ">
            <strong>Streamlit interface:</strong><br>
            <a
                href="{STREAMLIT_PROXY_PATH}"
                target="_blank"
                rel="noopener noreferrer"
                style="font-size: 16px;"
            >
                Open the Chest X-Ray Analysis and Explanation Assistant
            </a>
        </div>
        """
    )
)


# --------------------------------------------------------------------------------------------------
# Formal screenshot-readiness summary
# --------------------------------------------------------------------------------------------------

print("SCREENSHOT-READY INTERFACE VALIDATION")
print("-" * 110)

for check_name, passed in (
    screenshot_readiness_checks.items()
):
    print(
        f"{check_name:<48}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

print()
print(
    f"Validated checks             : "
    f"{len(screenshot_readiness_checks)}"
)
print(
    f"Failed checks                : "
    f"{len(failed_screenshot_checks)}"
)
print(
    f"Streamlit proxy path         : "
    f"{STREAMLIT_PROXY_PATH}"
)
print(
    f"Demonstration input          : "
    f"{UI_DEMO_INPUT_PATH}"
)
print(
    f"Readiness evidence           : "
    f"{SCREENSHOT_READINESS_PATH}"
)
print(
    f"Free storage                 : "
    f"{SCREENSHOT_FREE_STORAGE_GIB:.2f} GiB"
)
print(f"Protected storage reserve    : PASS")
print(f"Browser package installed    : NO")
print(f"Manual screenshot path ready : YES")
print()
print("READY FOR MANUAL INTERFACE REVIEW AND SCREENSHOT CAPTURE")


SCREENSHOT-READY INTERFACE VALIDATION
--------------------------------------------------------------------------------------------------------------
application_title_present                       : PASS
educational_limitation_present                  : PASS
image_upload_present                            : PASS
image_preview_present                           : PASS
complete_analysis_present                       : PASS
request_lineage_present                         : PASS
finding_summary_present                         : PASS
no_target_wording_present                       : PASS
gradcam_evidence_present                        : PASS
gradcam_limitation_present                      : PASS
grounded_language_present                       : PASS
guardrail_details_present                       : PASS
follow_up_question_present                      : PASS
stored_prediction_present                       : PASS
operational_metrics_present                     : PASS
controlled_error_rendering


## 23. UI Artifact Registry

This section records the persisted Streamlit source modules, configuration files, focused test evidence, live-workflow evidence, screenshot-readiness evidence, and demonstration input in a checksum-protected UI artifact registry. Runtime logs and transient process identifiers are intentionally excluded.


In [23]:

import hashlib
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path


UI_ARTIFACT_REGISTRY_PATH = (
    UI_OUTPUT_ROOT / "ui_artifact_registry.json"
)

UI_ARTIFACT_REGISTRY_VERSION = (
    "streamlit-ui-artifact-registry-v1"
)

registered_ui_artifacts = (
    ("source_module", API_CLIENT_PATH),
    ("source_module", COMPONENTS_PATH),
    ("source_module", STREAMLIT_APP_PATH),
    ("configuration", UI_CONFIG_PATH),
    ("configuration", STREAMLIT_SERVER_CONFIG_PATH),
    ("test_evidence", API_CLIENT_TEST_EVIDENCE_PATH),
    ("test_evidence", UI_COMPONENT_TEST_EVIDENCE_PATH),
    ("workflow_evidence", LIVE_WORKFLOW_EVIDENCE_PATH),
    ("readiness_evidence", SCREENSHOT_READINESS_PATH),
    ("demonstration_input", UI_DEMO_INPUT_PATH),
)

missing_ui_artifacts = [
    str(path)
    for _, path in registered_ui_artifacts
    if not path.is_file()
]

if missing_ui_artifacts:
    raise FileNotFoundError(
        "Required UI artifacts are unavailable: "
        f"{missing_ui_artifacts}"
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


artifact_records = [
    {
        "category": category,
        "path": str(path),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for category, path in registered_ui_artifacts
]

category_counts = {}

for record in artifact_records:
    category = record["category"]
    category_counts[category] = (
        category_counts.get(category, 0) + 1
    )

ui_artifact_registry = {
    "registry_version": UI_ARTIFACT_REGISTRY_VERSION,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "ui_version": persisted_ui_configuration[
        "ui_version"
    ],
    "backend_boundary": "http_only",
    "api_readiness_version": READINESS_VERSION,
    "api_readiness_status": READINESS_STATUS,
    "api_contract": {
        "openapi_paths": OPENAPI_PATH_COUNT,
        "openapi_operations": OPENAPI_OPERATION_COUNT,
        "complete_analysis_image_field": (
            COMPLETE_ANALYSIS_IMAGE_FIELD
        ),
    },
    "testing": {
        "api_client_tests_passed": passed_client_tests,
        "api_client_tests_failed": failed_client_tests,
        "ui_component_tests_passed": passed_ui_tests,
        "ui_component_tests_failed": failed_ui_tests,
        "screenshot_readiness_checks": len(
            screenshot_readiness_checks
        ),
    },
    "registered_artifact_files": len(
        artifact_records
    ),
    "category_counts": category_counts,
    "artifacts": artifact_records,
}

UI_ARTIFACT_REGISTRY_PATH.write_text(
    json.dumps(
        ui_artifact_registry,
        indent=2,
    ),
    encoding="utf-8",
)

persisted_ui_artifact_registry = json.loads(
    UI_ARTIFACT_REGISTRY_PATH.read_text(
        encoding="utf-8"
    )
)

checksum_failures = [
    record["path"]
    for record in persisted_ui_artifact_registry[
        "artifacts"
    ]
    if sha256_file(Path(record["path"]))
    != record["sha256"]
]

if checksum_failures:
    raise RuntimeError(
        "UI artifact checksum validation failed: "
        f"{checksum_failures}"
    )

registry_free_storage_gib = (
    shutil.disk_usage(DATA_ROOT).free
    / (1024 ** 3)
)

if registry_free_storage_gib < MINIMUM_FREE_STORAGE_GIB:
    raise RuntimeError(
        "The protected data-volume storage reserve is unavailable."
    )

print("UI ARTIFACT REGISTRY")
print("-" * 100)
print(f"Registry path             : {UI_ARTIFACT_REGISTRY_PATH}")
print(f"Registry version          : {UI_ARTIFACT_REGISTRY_VERSION}")
print(f"Registered artifact files : {len(artifact_records)}")
print(f"Source modules            : {category_counts.get('source_module', 0)}")
print(f"Configuration files       : {category_counts.get('configuration', 0)}")
print(f"Evidence files            : {sum(value for key, value in category_counts.items() if 'evidence' in key)}")
print(f"Demonstration inputs      : {category_counts.get('demonstration_input', 0)}")
print(f"Artifact checksums        : PASS")
print(f"Backend boundary          : HTTP ONLY")
print(f"Free storage             : {registry_free_storage_gib:.2f} GiB")
print(f"Protected storage reserve : PASS")
print()
print("READY FOR MLFLOW UI INTEGRATION REGISTRATION")


UI ARTIFACT REGISTRY
----------------------------------------------------------------------------------------------------
Registry path             : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/ui/ui_artifact_registry.json
Registry version          : streamlit-ui-artifact-registry-v1
Registered artifact files : 10
Source modules            : 3
Configuration files       : 2
Evidence files            : 4
Demonstration inputs      : 1
Artifact checksums        : PASS
Backend boundary          : HTTP ONLY
Free storage             : 8.71 GiB
Protected storage reserve : PASS

READY FOR MLFLOW UI INTEGRATION REGISTRATION



## 24. MLflow UI Integration Registration

This section registers the completed Streamlit integration, focused UI-test results, live workflow validation, screenshot readiness, and artifact registry in the project’s persisted MLflow tracking store. It records interface lineage only and does not load, retrain, or reevaluate either model.


In [24]:

import json
import os
from datetime import datetime, timezone

import mlflow
from mlflow.tracking import MlflowClient


MLFLOW_TRACKING_URI = (
    "file:///home/jovyan/apicdsa2-datavol-1/"
    "chest-xray-ai-assistant-data/mlflow"
)

UI_MLFLOW_EXPERIMENT_NAME = (
    "chestmnist-streamlit-interface-integration"
)

UI_MLFLOW_REGISTRATION_PATH = (
    UI_OUTPUT_ROOT / "ui_mlflow_registration.json"
)

os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

mlflow_client = MlflowClient(
    tracking_uri=MLFLOW_TRACKING_URI
)

ui_experiment = mlflow_client.get_experiment_by_name(
    UI_MLFLOW_EXPERIMENT_NAME
)

if ui_experiment is None:
    UI_MLFLOW_EXPERIMENT_ID = (
        mlflow_client.create_experiment(
            UI_MLFLOW_EXPERIMENT_NAME
        )
    )
else:
    UI_MLFLOW_EXPERIMENT_ID = (
        ui_experiment.experiment_id
    )

with mlflow.start_run(
    experiment_id=UI_MLFLOW_EXPERIMENT_ID,
    run_name="notebook8-streamlit-interface-integration",
    tags={
        "notebook": "08_streamlit_interface_integration",
        "ui_version": persisted_ui_configuration[
            "ui_version"
        ],
        "backend_boundary": "http_only",
        "educational_use_only": "true",
    },
) as ui_mlflow_run:
    UI_MLFLOW_RUN_ID = ui_mlflow_run.info.run_id

    mlflow.log_params(
        {
            "api_readiness_version": READINESS_VERSION,
            "api_readiness_status": READINESS_STATUS,
            "openapi_paths": OPENAPI_PATH_COUNT,
            "openapi_operations": OPENAPI_OPERATION_COUNT,
            "complete_analysis_image_field": (
                COMPLETE_ANALYSIS_IMAGE_FIELD
            ),
            "streamlit_version": STREAMLIT_VERSION,
            "httpx_version": HTTPX_VERSION,
            "maximum_upload_mib": MAXIMUM_UPLOAD_MIB,
            "artifact_registry_version": (
                UI_ARTIFACT_REGISTRY_VERSION
            ),
        }
    )

    mlflow.log_metrics(
        {
            "api_client_tests_passed": passed_client_tests,
            "api_client_tests_failed": failed_client_tests,
            "ui_component_tests_passed": passed_ui_tests,
            "ui_component_tests_failed": failed_ui_tests,
            "screenshot_checks_passed": sum(
                value is True
                for value in screenshot_readiness_checks.values()
            ),
            "screenshot_checks_failed": sum(
                value is not True
                for value in screenshot_readiness_checks.values()
            ),
            "live_findings_returned": len(live_findings),
            "live_visual_evidence_records": len(
                live_visual_evidence
            ),
            "registered_ui_artifacts": len(
                artifact_records
            ),
        }
    )

    for artifact_path in (
        UI_ARTIFACT_REGISTRY_PATH,
        API_CLIENT_TEST_EVIDENCE_PATH,
        UI_COMPONENT_TEST_EVIDENCE_PATH,
        LIVE_WORKFLOW_EVIDENCE_PATH,
        SCREENSHOT_READINESS_PATH,
    ):
        mlflow.log_artifact(
            str(artifact_path),
            artifact_path="ui_integration_evidence",
        )

ui_run_record = mlflow_client.get_run(
    UI_MLFLOW_RUN_ID
)

if ui_run_record.info.status != "FINISHED":
    raise RuntimeError(
        "The MLflow UI integration run did not finish successfully."
    )

ui_mlflow_registration = {
    "registration_version": (
        "streamlit-ui-mlflow-registration-v1"
    ),
    "registered_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "tracking_uri": MLFLOW_TRACKING_URI,
    "experiment_name": UI_MLFLOW_EXPERIMENT_NAME,
    "experiment_id": UI_MLFLOW_EXPERIMENT_ID,
    "run_id": UI_MLFLOW_RUN_ID,
    "run_status": ui_run_record.info.status,
    "ui_version": persisted_ui_configuration[
        "ui_version"
    ],
    "backend_boundary": "http_only",
    "educational_use_only": True,
    "models_loaded_by_registration": False,
    "models_retrained": False,
}

UI_MLFLOW_REGISTRATION_PATH.write_text(
    json.dumps(
        ui_mlflow_registration,
        indent=2,
    ),
    encoding="utf-8",
)

print("MLFLOW UI INTEGRATION REGISTRATION")
print("-" * 100)
print(f"Tracking URI              : {MLFLOW_TRACKING_URI}")
print(f"Experiment name           : {UI_MLFLOW_EXPERIMENT_NAME}")
print(f"Experiment ID             : {UI_MLFLOW_EXPERIMENT_ID}")
print(f"Run ID                    : {UI_MLFLOW_RUN_ID}")
print(f"Run status                : {ui_run_record.info.status}")
print(f"Registration artifact     : {UI_MLFLOW_REGISTRATION_PATH}")
print(f"UI artifact registry      : LOGGED")
print(f"Focused test evidence     : LOGGED")
print(f"Live workflow evidence    : LOGGED")
print(f"Screenshot readiness      : LOGGED")
print(f"Models loaded             : NO")
print(f"Models retrained          : NO")
print()
print("READY FOR FINAL NOTEBOOK 8 READINESS VALIDATION")


/opt/conda/lib/python3.11/site-packages/mlflow/utils/requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251


MLFLOW UI INTEGRATION REGISTRATION
----------------------------------------------------------------------------------------------------
Tracking URI              : file:///home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/mlflow
Experiment name           : chestmnist-streamlit-interface-integration
Experiment ID             : 727502364069826069
Run ID                    : fba173c494144546bf8cf12998bb48da
Run status                : FINISHED
Registration artifact     : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/ui/ui_mlflow_registration.json
UI artifact registry      : LOGGED
Focused test evidence     : LOGGED
Live workflow evidence    : LOGGED
Screenshot readiness      : LOGGED
Models loaded             : NO
Models retrained          : NO

READY FOR FINAL NOTEBOOK 8 READINESS VALIDATION



## 25. Final Notebook 8 Readiness Gate

This section performs the final evidence-based readiness check for the Streamlit integration. It verifies the persisted API contract, focused test results, live HTTP workflow, screenshot-ready interface, checksum-protected UI artifacts, MLflow lineage, running service health, HTTP-only backend boundary, and protected storage reserve before declaring Notebook 8 complete.


In [25]:

import ast
import hashlib
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

import httpx


UI_INTEGRATION_READINESS_PATH = (
    UI_OUTPUT_ROOT
    / "streamlit_interface_integration_readiness.json"
)


def current_sha256(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


persisted_registry = json.loads(
    UI_ARTIFACT_REGISTRY_PATH.read_text(
        encoding="utf-8"
    )
)

persisted_mlflow_registration = json.loads(
    UI_MLFLOW_REGISTRATION_PATH.read_text(
        encoding="utf-8"
    )
)

persisted_live_workflow = json.loads(
    LIVE_WORKFLOW_EVIDENCE_PATH.read_text(
        encoding="utf-8"
    )
)

persisted_screenshot_readiness = json.loads(
    SCREENSHOT_READINESS_PATH.read_text(
        encoding="utf-8"
    )
)

registry_checksums_valid = all(
    Path(record["path"]).is_file()
    and current_sha256(Path(record["path"]))
    == record["sha256"]
    for record in persisted_registry["artifacts"]
)

api_client_tests_valid = (
    passed_client_tests == len(client_test_results)
    and failed_client_tests == 0
)

ui_component_tests_valid = (
    passed_ui_tests == len(ui_test_results)
    and failed_ui_tests == 0
)

screenshot_checks = (
    persisted_screenshot_readiness["checks"]
)

screenshot_readiness_valid = (
    bool(screenshot_checks)
    and all(value is True for value in screenshot_checks.values())
)

live_workflow_valid = all(
    (
        persisted_live_workflow.get(
            "stored_prediction_verified"
        ) is True,
        persisted_live_workflow.get(
            "operational_metrics_verified"
        ) is True,
        persisted_live_workflow.get(
            "streamlit_render_exceptions"
        ) == 0,
        persisted_live_workflow.get(
            "dataset_accessed"
        ) is False,
    )
)

mlflow_run_valid = (
    persisted_mlflow_registration.get(
        "run_status"
    ) == "FINISHED"
    and mlflow_client.get_run(
        persisted_mlflow_registration["run_id"]
    ).info.status == "FINISHED"
)

forbidden_ui_import_roots = {
    "torch",
    "torchvision",
    "transformers",
    "captum",
}

observed_ui_import_roots = set()

for source_path in (
    API_CLIENT_PATH,
    COMPONENTS_PATH,
    STREAMLIT_APP_PATH,
):
    source_tree = ast.parse(
        source_path.read_text(encoding="utf-8"),
        filename=str(source_path),
    )

    for node in ast.walk(source_tree):
        if isinstance(node, ast.Import):
            observed_ui_import_roots.update(
                alias.name.split(".")[0]
                for alias in node.names
            )
        elif isinstance(node, ast.ImportFrom) and node.module:
            observed_ui_import_roots.add(
                node.module.split(".")[0]
            )

http_only_boundary_valid = not (
    forbidden_ui_import_roots
    & observed_ui_import_roots
)

with httpx.Client(timeout=10.0) as final_http_client:
    final_fastapi_health = final_http_client.get(
        f"{API_BASE_URL}/health"
    )
    final_streamlit_health = final_http_client.get(
        STREAMLIT_HEALTH_URL
    )

service_health_valid = (
    final_fastapi_health.status_code == 200
    and final_streamlit_health.status_code == 200
    and final_streamlit_health.text.strip().lower() == "ok"
)

final_free_storage_gib = (
    shutil.disk_usage(DATA_ROOT).free
    / (1024 ** 3)
)

final_storage_reserve_valid = (
    final_free_storage_gib
    >= MINIMUM_FREE_STORAGE_GIB
)

final_readiness_checks = {
    "Notebook 7 API readiness is preserved": (
        BACKEND_READINESS_CONFIRMED
    ),
    "Twelve OpenAPI paths are preserved": (
        OPENAPI_PATH_COUNT == 12
    ),
    "Twelve OpenAPI operations are preserved": (
        OPENAPI_OPERATION_COUNT == 12
    ),
    "UI HTTP client tests passed": api_client_tests_valid,
    "UI component and state tests passed": (
        ui_component_tests_valid
    ),
    "Live interface workflow passed": live_workflow_valid,
    "Screenshot-ready checks passed": (
        screenshot_readiness_valid
    ),
    "UI artifact checksums are complete": (
        registry_checksums_valid
    ),
    "MLflow UI registration is finished": (
        mlflow_run_valid
    ),
    "UI preserves the HTTP-only backend boundary": (
        http_only_boundary_valid
    ),
    "FastAPI and Streamlit are healthy": (
        service_health_valid
    ),
    "Educational-use boundary remains enabled": (
        persisted_ui_configuration["safety"][
            "educational_limitation"
        ] == EDUCATIONAL_LIMITATION
    ),
    "Protected storage reserve remains available": (
        final_storage_reserve_valid
    ),
}

failed_final_checks = [
    check_name
    for check_name, passed in final_readiness_checks.items()
    if passed is not True
]

if failed_final_checks:
    raise RuntimeError(
        "Final Notebook 8 readiness validation failed: "
        f"{failed_final_checks}"
    )

ui_integration_readiness = {
    "readiness_version": (
        "streamlit-interface-integration-readiness-v1"
    ),
    "evaluated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": "ready_for_packaging_and_deployment",
    "ui_version": persisted_ui_configuration[
        "ui_version"
    ],
    "backend_boundary": "http_only",
    "educational_use_only": True,
    "api_lineage": {
        "readiness_version": READINESS_VERSION,
        "readiness_status": READINESS_STATUS,
        "openapi_paths": OPENAPI_PATH_COUNT,
        "openapi_operations": OPENAPI_OPERATION_COUNT,
    },
    "testing": {
        "api_client_tests_passed": passed_client_tests,
        "ui_component_tests_passed": passed_ui_tests,
        "screenshot_checks_passed": len(
            screenshot_checks
        ),
        "live_workflow_verified": True,
    },
    "artifact_registry": {
        "path": str(UI_ARTIFACT_REGISTRY_PATH),
        "registry_version": UI_ARTIFACT_REGISTRY_VERSION,
        "registered_artifact_files": len(
            persisted_registry["artifacts"]
        ),
    },
    "mlflow": {
        "experiment_id": (
            persisted_mlflow_registration[
                "experiment_id"
            ]
        ),
        "run_id": persisted_mlflow_registration[
            "run_id"
        ],
        "run_status": persisted_mlflow_registration[
            "run_status"
        ],
    },
    "service_health": {
        "fastapi": True,
        "streamlit": True,
        "streamlit_proxy_path": STREAMLIT_PROXY_PATH,
    },
    "storage": {
        "free_storage_gib": round(
            final_free_storage_gib,
            2,
        ),
        "protected_reserve_gib": (
            MINIMUM_FREE_STORAGE_GIB
        ),
        "reserve_available": True,
    },
    "checks": final_readiness_checks,
}

UI_INTEGRATION_READINESS_PATH.write_text(
    json.dumps(
        ui_integration_readiness,
        indent=2,
    ),
    encoding="utf-8",
)

print("FINAL NOTEBOOK 8 READINESS GATE")
print("-" * 110)

for check_name, passed in final_readiness_checks.items():
    print(f"{check_name:<62}: {'PASS' if passed else 'FAIL'}")

print()
print(f"Validated checks            : {len(final_readiness_checks)}")
print(f"Failed checks               : 0")
print(f"UI artifact files           : {len(persisted_registry['artifacts'])}")
print(f"MLflow run status           : FINISHED")
print(f"FastAPI health              : PASS")
print(f"Streamlit health            : PASS")
print(f"Streamlit proxy path        : {STREAMLIT_PROXY_PATH}")
print(f"Free storage                : {final_free_storage_gib:.2f} GiB")
print(f"Protected storage reserve   : PASS")
print(f"Final readiness artifact    : {UI_INTEGRATION_READINESS_PATH}")
print()
print("NOTEBOOK 8 COMPLETE — STREAMLIT INTERFACE READY FOR PACKAGING AND DEPLOYMENT")


FINAL NOTEBOOK 8 READINESS GATE
--------------------------------------------------------------------------------------------------------------
Notebook 7 API readiness is preserved                         : PASS
Twelve OpenAPI paths are preserved                            : PASS
Twelve OpenAPI operations are preserved                       : PASS
UI HTTP client tests passed                                   : PASS
UI component and state tests passed                           : PASS
Live interface workflow passed                                : PASS
Screenshot-ready checks passed                                : PASS
UI artifact checksums are complete                            : PASS
MLflow UI registration is finished                            : PASS
UI preserves the HTTP-only backend boundary                   : PASS
FastAPI and Streamlit are healthy                             : PASS
Educational-use boundary remains enabled                      : PASS
Protected storage reserve rem